# Brain tissue: aggregate DVP of post-mortem human brain

Formalin-fixed post-mortem brain tissue from donors with Parkinson's disease or Lewy body disease and from neurologically healthy controls, from two brain banks: Neurobiobank München ("Munich") and the Banner Sun Health Research Institute ("BH", shown as "Banner"). Sections were stained for pS129 α-synuclein with DAB, and seven fractions were cut by laser microdissection for Deep Visual Proteomics (DVP): neurons and glia with and without a DAB-positive inclusion, neuropil, and excised round and elongated inclusions. The protein groups come from DIA-NN. Section 13 uses a second DIA-NN search that quantifies a replication cohort together with these samples. This notebook makes the panels of Figure 4 and Figure S6.

| Short | Meaning |
|---|---|
| DAB+, DAB− | cell body with, without a pS129 α-synuclein (DAB-stained) inclusion |
| EA | excised aggregate (inclusion); obs values "Round Aggregate" and "Elongated Aggregate" |
| MTG | middle temporal gyrus |
| log2FC | log2 fold change |
| FDR, BH | false discovery rate, Benjamini-Hochberg |
| eBayes | limma linear model with empirical Bayes moderation of the variances |
| CV | coefficient of variation |
| PCA | principal component analysis |
| GO CC | Gene Ontology cellular component |
| GSEA, NES | gene set enrichment analysis, normalised enrichment score |

**Inputs:** `input_dir` (DIA-NN `report.pg_matrix.tsv` and `report.pr_matrix.tsv`, `metadata.csv`) of the discovery search; `JOINT_DIR` (section 13.1) with the same files of the joint search; the UniProt-to-gene table `src/Uniprot_to_GeneSymbol.csv` (from the alphapepttools documentation); the MSigDB GO CC gene sets cached in `src/gsea_genesets_gocc_cache.parquet` (retrieved 2026-09-08, identical to MSigDB on 2026-10-29); the GSEA decision file `review.csv` in `gsea_dir/<contrast>/`, read by section 12.1. Its decisions were made by the authors with the help of an AI review (Claude, Anthropic) that followed the rules in `GSEA_REVIEW_PROMPT.md`. All paths are set in the first cell of section 2.

**Outputs:** result tables in `output_dir`; GSEA results in `gsea_dir`; figure panels in `figures/Figure4_Brain` and `figures/SIs/SI6_Brain`, written by `save_panel` and assembled in section 14. The notebook runs from top to bottom and needs no other notebook.

# 1. Imports

In [ ]:
# --- standard library ---
import glob
import itertools
import logging
import re
import sys
import textwrap
import warnings
from collections import Counter
from datetime import date
from math import factorial
from pathlib import Path

# --- scientific stack ---
import anndata as ad
import decoupler as dc
import numpy as np
import pandas as pd
import scanpy as sc
from inmoose.limma import lmFit, eBayes, topTable
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

# --- plotting ---
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Polygon
from matplotlib.transforms import ScaledTranslation, blended_transform_factory
from adjustText import adjust_text

# --- alphapepttools ---
import alphapepttools as at
from alphapepttools.pl.figure import create_figure, label_axes

# --- project modules in src/ ---
# The path is relative to this notebook's folder, which VS Code uses as the
# working directory of a notebook. Importing figure_style sets the matplotlib and
# alphapepttools style of every panel.
sys.path.insert(0, "src")
from figure_style import (ANNOT_FS, AXES_LW, BG_COLOR, DIRECTION_CMAP, DOWN_COLOR,
                          DRAW_SCALE, HIGHLIGHT_COLOR, LABEL_FS, LEGEND_FS, LINE_LW,
                          MARKER_LW, MM_PER_IN, REF_COLOR, SCALE, THIN_LW, TICK_FS,
                          TITLE_FS, UP_COLOR, compose_figure, figsize, native_pt,
                          save_panel, shade, style_summary)
from gsea_pipeline import (GSEAConfig, load_gene_sets, run_contrast_from_rank,
                           build_review, assert_pass2_selection, report_undecided)
from boxplot import fraction_boxplot
from heatmap import group_heatmap
from preprocessing import flag_contaminants
from scatter import replication_scatter

logging.getLogger("fontTools.subset").setLevel(logging.WARNING)
today = date.today().strftime("%Y%m%d")      # date stamp of the output file names

# 2. Data loading

In [ ]:
# Every path of the notebook, relative to the repository root.
input_dir  = 'data/BrainTissue/ADVP04_011_Combined_withBannerLibrary'     # DIA-NN output of the discovery search, metadata.csv
JOINT_DIR  = 'data/BrainTissue/ADVP04_011_17_CombinedwithBannerLibrary'   # DIA-NN output of the joint search (section 13), metadata.csv
output_dir = 'results/BrainTissue'              # result tables
# GSEA folder: <contrast>/review.csv in it holds the review decisions, an input;
# the GSEA results are written next to it. It is set apart from output_dir, so
# that a new output_dir keeps the decisions.
gsea_dir   = 'gsea/BrainTissue'
var_metadata_path = 'src/Uniprot_to_GeneSymbol.csv'             # alphapepttools documentation dataset
GOCC_CACHE = Path('src/gsea_genesets_gocc_cache.parquet')       # GO CC gene sets; section 10 builds it when absent
# Panel folders: save_panel writes into them, section 14 assembles them.
FIG4_DIR = "figures/Figure4_Brain"
SI6_DIR  = "figures/SIs/SI6_Brain"
Path(output_dir).mkdir(parents=True, exist_ok=True)

pg_table_path = f'{input_dir}/report.pg_matrix.tsv'
obs_metadata_path = f'{input_dir}/metadata.csv'

adata = at.io.read_pg_table(pg_table_path, search_engine="diann")
obs_metadata = pd.read_csv(obs_metadata_path, sep=",", index_col=0)
var_metadata = pd.read_csv(var_metadata_path, sep=",", index_col=0)

# Sample metadata onto obs; samples without a metadata row are dropped.
adata = at.pp.add_metadata(
    adata=adata,
    incoming_metadata=obs_metadata,
    axis=0,
    keep_data_shape=False,
)
# Protein.Group, Protein.Names and Genes of the UniProt-to-gene table onto var,
# matched on the protein group ID; protein groups without a row get NaN.
adata.var = adata.var.merge(var_metadata, left_index=True, right_index=True, how="left")

display(adata.to_df().head())
display(adata.obs.head())
display(adata.var.head())

## 2.1 Samples excluded before analysis

In [ ]:
# Samples excluded after inspection of the cutting: one empty well and three cuts
# that were not clean. Each entry is a substring of exactly one raw file name.
DROP_SUBSTRINGS = [
    # empty sample
    "Plate3C2_D05_260212_CTRL2_Glia_WhiteMatter",
    # cuts that were not clean
    "Plate4C3_D06_260212_LBD2_RoundAggs1_Cortex",
    "Plate3C2_A10_260212_LBD2_ElonAggs1_Cortex",
    "Plate3C2_G06_260212_LBD2_RoundAggs2_Cortex"
]

def _mask_keep(obs_index):
    bad = pd.Series(False, index=obs_index)
    for s in DROP_SUBSTRINGS:
        bad |= obs_index.str.contains(s, regex=False)
    return ~bad

keep = _mask_keep(adata.obs.index)
print(f"Dropping {(~keep).sum()} problematic samples (of {keep.size}): "
      f"{list(adata.obs.index[~keep])}")
adata = adata[keep, :].copy()

# The random controls (RandomControl_Random) have no brain region in the metadata
# and are not assigned to any fraction.
random_mask = adata.obs["Brain Region"].isna()
print(f"Dropping {random_mask.sum()} RandomControl_Random samples (missing Brain_Region)")
adata = adata[~random_mask, :].copy()

print(f"Final: adata = {adata.n_obs} samples x {adata.n_vars} proteins")

# The preprocessing cell starts from this copy, so running it a second time
# filters the same matrix again.
adata_raw = adata.copy()

## 2.2 Style and colours

In [ ]:
# Page grid, fonts, line widths, colour roles, save_panel and compose_figure come
# from src/figure_style.py (imported in section 1), shared by all notebooks of the
# project. This cell holds the colours of the brain fractions.
print(style_summary())

# ---------------------------------------------------------------- COLOURS
# Aggregate-status colours shared with the 1_HEK and 2_iDA notebooks; Figure
# S6i colours its two highlight groups with them.
CAT_BASE = {
    "without": "#7fb3d8",   # aggregate-free cell, the 1_HEK CWO blue
    "dab_pos": "#e8a07a",   # aggregate-bearing cell, the 1_HEK CW orange
    "excised": "#8e6ba8",   # excised aggregate, the 1_HEK EA violet
}

# hue       = cell type or compartment
# lightness = DAB status: base = DAB+ (carries an inclusion), lighter = DAB−
# Hues of Paul Tol's bright scheme with the chroma lowered to that of the EA
# violet (CIELAB C* 28-40; the Tol entries reach 70), so the panels have the
# intensity of the iPSC ones. Orange is the lightest (L* 68): darker at this
# chroma it turns brown. DAB− is shade 0.55: on these lighter bases it gives
# the lightness steps of shade 0.50 on the Tol colours.
CT_BASE = {
    "neuron":   "#5e89ba",   # blue
    "glia":     "#df946d",   # orange; blue and orange are the most separable
                             # pair, for the two groups compared most often
    "neuropil": "#54a395",   # teal, a compartment and not a cell type
    "excised":  "#8e6ba8",   # violet, EA, the same in every notebook
}

DAB_NEG_SHADE  = 0.55   # how much lighter DAB− is than DAB+
EA_ELONG_SHADE = 0.50   # how much lighter elongated EA is than round EA

# Colour per obs["CellType_Aggregate"] (disease status collapsed).
CELLTYPE_AGG_COLORS = {
    "Neuron Without Aggregate":      shade(CT_BASE["neuron"],   DAB_NEG_SHADE),
    "Neuron DAB Positive":           shade(CT_BASE["neuron"],   0.00),
    "Glia Without Aggregate":        shade(CT_BASE["glia"],     DAB_NEG_SHADE),
    "Glia DAB Positive":             shade(CT_BASE["glia"],     0.00),
    "Neuropil Without Aggregate":    shade(CT_BASE["neuropil"], 0.00),
    "Round Aggregate Aggregate":     shade(CT_BASE["excised"],  0.00),
    "Elongated Aggregate Aggregate": shade(CT_BASE["excised"],  EA_ELONG_SHADE),
}

# Colour per obs["Cell Type"], for the panels grouped by cell type (Figures S6a, S6b).
CELLTYPE_COLORS = {
    "Neuron":              CT_BASE["neuron"],
    "Glia":                CT_BASE["glia"],
    "Neuropil":            CT_BASE["neuropil"],
    "Round Aggregate":     CT_BASE["excised"],
    "Elongated Aggregate": shade(CT_BASE["excised"], EA_ELONG_SHADE),
}

## 2.3 Preprocessing

In [ ]:
# Preprocessing. The panels read these objects:
#   adata_sf          samples filtered, contaminants removed, all proteins (linear)
#   adata_strict      proteins quantified in every sample (linear)
#   adata             proteins in >= 80 % of the samples of at least one
#                     CellType_Aggregate group (linear)
#   adata_log         log2(x + 1) of adata
#   adata_pca         log2(x + 1) of adata_strict, with the PCA
# The cell starts from adata_raw (section 2.1), so a second run filters the same
# matrix again.

# --- Parameters ---
biological   = "Disease_CellType_Aggregate"   # condition column of the sample filters and the group counts
forfiltering = "CellType_Aggregate"           # groups of the 80 % completeness filter, disease status collapsed
MIN_PROTEINS = 2000          # samples with fewer protein groups are excluded
MAX_MISSING_COND = 0.20      # a protein is kept if it is missing in at most 20 % of one group

adata = adata_raw.copy()

# ------------------------------------------------------------
# 1. Identifications per sample: protein groups, and precursors from the DIA-NN
#    precursor matrix (Figure S6a)
# ------------------------------------------------------------
adata.obs["num_proteins"] = np.sum(~np.isnan(adata.X), axis=1)

pr_table_path = f"{input_dir}/report.pr_matrix.tsv"
_pr_matrix = pd.read_csv(pr_table_path, sep="\t", index_col=0).T
_pr_matrix = _pr_matrix.reindex(adata.obs.index)
_pr_numeric = _pr_matrix.apply(pd.to_numeric, errors="coerce")
adata.obs["num_precursors"] = np.sum(~np.isnan(_pr_numeric.values), axis=1)
del _pr_matrix, _pr_numeric

n_start_samples, n_start_proteins = adata.shape

# ------------------------------------------------------------
# 2. Sample filters: blanks, fewer than MIN_PROTEINS protein groups, missing group label
# ------------------------------------------------------------
adata = adata[~adata.obs[biological].isin(["Blank", "EvoTip Blank"]), :].copy()
n_after_blanks = adata.n_obs

adata.obs["is_outlier"] = adata.obs["num_proteins"] < MIN_PROTEINS
adata_unfiltered = adata.copy()      # all samples before the protein-count filter
adata = adata_unfiltered[~adata_unfiltered.obs["is_outlier"], :].copy()
n_after_outliers = adata.n_obs

nan_mask = adata.obs[biological].isna() | adata.obs[forfiltering].isna()
if nan_mask.any():
    print(f"WARNING: dropping {nan_mask.sum()} samples with NaN in {biological} or {forfiltering}")
    adata = adata[~nan_mask, :].copy()
n_after_nan = adata.n_obs

# Samples per group: the completeness filter works on the forfiltering groups,
# the panels on the biological groups.
filter_counts = adata.obs[forfiltering].value_counts()
print(f"\nGroups feeding completeness filter (column={forfiltering!r}, n={len(filter_counts)}):")
for g, n in filter_counts.items():
    print(f"  {g!r:30s} n={n}")
bio_counts = adata.obs[biological].value_counts()
print(f"\nDownstream biological groups (column={biological!r}, n={len(bio_counts)}):")
for g, n in bio_counts.items():
    print(f"  {g!r:30s} n={n}")
print()

# ------------------------------------------------------------
# 3. Contaminants
# ------------------------------------------------------------
# flag_contaminants (src/preprocessing.py) applies the keratin, cRAP and
# immunoglobulin rules and this list, which is specific to this sample type.
# UniProt entry names.
adata.var["single_gene_name"] = adata.var["genes"].str.split(";").str[0].copy()

BRAIN_EXTRA_CONTAMINANTS = [
    # Keratins
    "K1C13_HUMAN","K1C18_HUMAN","K1C27_HUMAN",
    "K2C1B_HUMAN","K2C3_HUMAN","K2C4_HUMAN",
    "K2C72_HUMAN","K2C78_HUMAN","K2C79_HUMAN","K2C8_HUMAN",
    # Skin barrier / epidermal
    "HORN_HUMAN","FILA_HUMAN","FILA2_HUMAN",
    "DCD_HUMAN","CDSN_HUMAN","DMKN_HUMAN","SBSN_HUMAN","PROL4_HUMAN",
    "DSG1_HUMAN","DSC1_HUMAN","DSC3_HUMAN","DESP_HUMAN","PLAK_HUMAN",
    "TGM1_HUMAN","TGM3_HUMAN",
    # Skin/secretory + S100 family
    "SPB12_HUMAN","SG1D2_HUMAN","LCN1_HUMAN","PIP_HUMAN",
    "ZA2G_HUMAN","CYTA_HUMAN","CYTM_HUMAN","BPIA1_HUMAN",
    "S10A7_HUMAN","S10A8_HUMAN","S10A9_HUMAN","S10AB_HUMAN",
    # Serum / blood
    "ALBU_HUMAN","HBA_HUMAN","HBB_HUMAN","HBD_HUMAN",
    "APOA1_HUMAN","APOB_HUMAN",
    "TRFE_HUMAN","TRFL_HUMAN","LYSC_HUMAN","ANT3_HUMAN",
    # Immunoglobulins (the gene-symbol rule catches the IGK, IGL and IGH families)
    "IGHA1_HUMAN","IGHG1_HUMAN",
    # Lab / non-human
    "cRAP-TRYP_PIG","cRAP-K2M3_SHEEP","INS_HUMAN","TRY3_HUMAN",
    # Matrigel coating
    "CILP2_HUMAN",
]
adata.var = flag_contaminants(adata.var, extra=BRAIN_EXTRA_CONTAMINANTS)

# A protein group is a cRAP contaminant only if every one of its IDs is
# cRAP-prefixed. The cRAP database lists tau, which this tissue contains.
assert not adata.var.loc[adata.var["single_gene_name"].isin(["MAPT", "TAU"]),
                         "is_contaminant"].any(), "MAPT (tau) is flagged as a contaminant"

n_contaminants = int(adata.var["is_contaminant"].sum())
adata = adata[:, ~adata.var["is_contaminant"]].copy()
n_after_contaminants = adata.n_vars

# Median intensity and its rank, for the rank plot (Figure 4a) and for choosing one
# protein group per gene symbol (section 12.1).
adata_sf = adata.copy()
adata_sf.var["median_intensity"] = np.nanmedian(adata_sf.X, axis=0)
adata_sf.var["protein_rank"]     = np.argsort(np.argsort(-adata_sf.var["median_intensity"]))

# ------------------------------------------------------------
# 4. Completeness filters
# ------------------------------------------------------------
adata_strict = at.pp.filter_data_completeness(adata.copy(), max_missing=0, action="drop")


def complete_in_any_group(adata, group_column, max_missing):
    """Keep the proteins missing in at most max_missing of the samples of at least
    one group, so a protein specific to one fraction stays. With group_column,
    filter_data_completeness drops a protein that fails in any group, so each group
    is flagged on its own and the flags are combined with OR. Section 13.1 applies
    the same rule to the joint search."""
    passes = np.zeros(adata.n_vars, dtype=bool)
    for group in adata.obs[group_column].dropna().unique():
        flagged = at.pp.filter_data_completeness(
            adata.copy(), max_missing=max_missing,
            group_column=group_column, groups=[group], action="flag",
        )
        passes |= flagged.var["passed_threshold_missing_values"].to_numpy()
    return adata[:, passes].copy()


adata = complete_in_any_group(adata, forfiltering, MAX_MISSING_COND)

# ------------------------------------------------------------
# 5. log2 copies and the PCA (no imputation)
# ------------------------------------------------------------
adata_log = adata.copy()
sc.pp.log1p(adata_log, base=2)
adata_strict_log = adata_strict.copy()
sc.pp.log1p(adata_strict_log, base=2)

adata_pca = adata_strict_log.copy()
at.tl.pca(adata_pca)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("=" * 64)
print("Preprocessing summary")
print("=" * 64)
print(f"Starting:                    {n_start_samples} samples × {n_start_proteins} proteins")
print(f"After blank filter:          {n_after_blanks} samples  (dropped {n_start_samples - n_after_blanks})")
print(f"After <{MIN_PROTEINS}-protein filter: {n_after_outliers} samples  (dropped {n_after_blanks - n_after_outliers})")
print(f"After NaN-{biological} filter: {n_after_nan} samples  (dropped {n_after_outliers - n_after_nan})")
print(f"After contaminant filter:    {n_after_contaminants} proteins  (dropped {n_contaminants})")
print("-" * 64)
print(f"adata_sf       (all non-contaminant):    {adata_sf.n_obs} × {adata_sf.n_vars}")
print(f"adata_strict   (100% complete):          {adata_strict.n_obs} × {adata_strict.n_vars}")
print(f"adata          (80% in >= 1 group):      {adata.n_obs} × {adata.n_vars}")
print(f"adata_log      (80% → log2):             {adata_log.n_obs} × {adata_log.n_vars}")
print(f"adata_pca      (strict log2 + PCA):      {adata_pca.n_obs} × {adata_pca.n_vars}")
print("=" * 64)

## 2.4 Groups by brain bank and cell type (Figures S6a, S6b)

In [ ]:
# Groups of Figures S6a and S6b: brain bank x cell type, banks in alphabetical
# order, cell types in CELLTYPE_ORDER within each bank. The names are upper case
# because both panels read them and no other cell assigns them.
BANK_CT_COL = "bank_celltype"
adata.obs[BANK_CT_COL] = (
    adata.obs["BrainBank"].astype(str) + " | " + adata.obs["Cell Type"].astype(str)
)
CELLTYPE_ORDER = ["Neuron", "Glia", "Neuropil", "Round Aggregate", "Elongated Aggregate"]

# A cell type missing from CELLTYPE_ORDER would drop out of both panels without a message.
_unordered = set(adata.obs["Cell Type"].dropna().astype(str)) - set(CELLTYPE_ORDER)
assert not _unordered, (
    f"cell types present in the data but missing from CELLTYPE_ORDER: {sorted(_unordered)}"
)
assert set(CELLTYPE_ORDER) <= set(CELLTYPE_COLORS), (
    f"no colour for: {sorted(set(CELLTYPE_ORDER) - set(CELLTYPE_COLORS))}"
)

BANKS = sorted(adata.obs["BrainBank"].dropna().unique().tolist())

# (bank, cell type, group key) of every group with at least one sample, in panel order
BANK_CT_GROUPS = []
for bank in BANKS:
    for ct in CELLTYPE_ORDER:
        key = f"{bank} | {ct}"
        if (adata.obs[BANK_CT_COL] == key).any():
            BANK_CT_GROUPS.append((bank, ct, key))

# 3. Protein groups and precursors per sample (Figure S6a)

In [ ]:
# Figure S6a. Protein groups (left axis) and precursors (right axis, hatched) per
# sample, one box per brain bank x cell type (groups of section 2.4), with the
# group means above the axes.

# x-tick labels: the two aggregate names are the long ones, so they wrap onto a
# second line.
celltype_display = {
    "Round Aggregate":     "Round\nAggregate",
    "Elongated Aggregate": "Elongated\nAggregate",
}

conditions = [key for _, _, key in BANK_CT_GROUPS]
n_cond = len(conditions)

# Box positions, with a gap between the banks
BANK_GAP = 2.0
positions_base, bank_of_box = [], []
pos, prev_bank = 0.0, None
for bank, ct, key in BANK_CT_GROUPS:
    if prev_bank is not None and bank != prev_bank:
        pos += BANK_GAP
    positions_base.append(pos)
    bank_of_box.append(bank)
    pos += 2.0
    prev_bank = bank
positions_base = np.array(positions_base)
positions_left  = positions_base
positions_right = positions_base + 0.8

bank_separators = [
    (positions_base[i-1] + positions_base[i]) / 2 + 0.4
    for i in range(1, len(bank_of_box)) if bank_of_box[i] != bank_of_box[i-1]
]
bank_centers = {
    bank: positions_base[[i for i, b in enumerate(bank_of_box) if b == bank]].mean() + 0.4
    for bank in BANKS
}

S_DOT  = native_pt(1.5) ** 2     # sample dots, 1.5 pt across in print
JITTER = 0.15                    # largest horizontal offset of a dot, in x units

fig, axm = create_figure(1, 1, figsize=figsize(2, 1))
ax_left = axm.next()
ax_right = ax_left.twinx()

# twinx() follows rcParams and hides the right spine, which carries the
# precursor scale here; its left spine would be drawn over ax_left's.
ax_right.spines["right"].set_visible(True)
ax_right.spines["left"].set_visible(False)
ax_right.spines["top"].set_visible(False)
ax_left.grid(axis="x", visible=False)        # no vertical grid lines between the boxes

pg_data = [adata.obs.loc[adata.obs[BANK_CT_COL] == c, "num_proteins"].values for c in conditions]
pr_data = [adata.obs.loc[adata.obs[BANK_CT_COL] == c, "num_precursors"].values for c in conditions]

# Protein groups on the left axis, precursors on the right axis (hatched).
for ax, data, pos, hatch, marker, seed in (
        (ax_left,  pg_data, positions_left,  None,  "o", 42),
        (ax_right, pr_data, positions_right, "///", "D", 43)):
    bp = ax.boxplot(data, positions=pos, widths=0.6, patch_artist=True,
                    showfliers=False, zorder=2)
    for i, (box, (bank, ct, key)) in enumerate(zip(bp["boxes"], BANK_CT_GROUPS)):
        color = CELLTYPE_COLORS[ct]
        box.set_facecolor(color)
        box.set_edgecolor("black")
        if hatch:
            box.set_hatch(hatch)
        jitter = np.random.default_rng(seed).uniform(-JITTER, JITTER, len(data[i]))
        ax.scatter(pos[i] + jitter, data[i], s=S_DOT, color=color, marker=marker,
                   edgecolors="black", linewidths=MARKER_LW, zorder=3)

# The two y axes are aligned by the ratio of the medians, with headroom above the
# largest value.
pg_max = max(np.max(d) for d in pg_data)
pr_max = max(np.max(d) for d in pr_data)
scale  = np.median(np.concatenate(pr_data)) / np.median(np.concatenate(pg_data))
HEADROOM = 1.12
ax_left.set_ylim(bottom=0, top=pg_max * HEADROOM)
ax_right.set_ylim(bottom=0, top=max(pg_max * HEADROOM * scale, pr_max * HEADROOM))

# --- Means and bank names above the axes ---
# Protein-group means in the lower row (bold), precursor means in the row above
# (bold italic), the bank names above both. The offsets are in points, so the
# rows keep their spacing whatever height the axes get.
# The means print at 5 pt: a five-digit precursor mean is 5.9 mm wide at 6 pt,
# and the boxes of one row are 6.4 mm apart.
MEAN_FS = native_pt(5.0)
GAP = native_pt(1.5)           # axes top to the lower row
ROW = 1.25 * MEAN_FS           # height of one row of numbers
top_left  = blended_transform_factory(ax_left.transData,  ax_left.transAxes)
top_right = blended_transform_factory(ax_right.transData, ax_right.transAxes)
for i in range(n_cond):
    ax_left.annotate(f"{np.mean(pg_data[i]):.0f}", (positions_left[i], 1), xycoords=top_left,
                     xytext=(0, GAP), textcoords="offset points",
                     ha="center", va="bottom", fontsize=MEAN_FS, fontweight="bold")
    ax_right.annotate(f"{np.mean(pr_data[i]):.0f}", (positions_right[i], 1), xycoords=top_right,
                      xytext=(0, GAP + ROW), textcoords="offset points",
                      ha="center", va="bottom", fontsize=MEAN_FS, fontweight="bold",
                      style="italic")
for bank, center in bank_centers.items():
    ax_left.annotate(bank, (center, 1), xycoords=top_left,
                     xytext=(0, GAP + 2 * ROW), textcoords="offset points",
                     ha="center", va="bottom", fontsize=LABEL_FS, fontweight="bold")

for sep in bank_separators:
    ax_left.axvline(sep, color=REF_COLOR, linewidth=THIN_LW, linestyle="--", zorder=0)

ax_left.set_xticks(positions_base + 0.4)
ax_left.set_xticklabels([celltype_display.get(ct, ct) for _, ct, _ in BANK_CT_GROUPS])
ax_left.tick_params(axis="x", labelrotation=90)
ax_left.set_ylabel("Protein groups")
ax_right.set_ylabel("Precursors")

X_PAD = 0.5
ax_left.set_xlim(positions_left[0] - X_PAD, positions_right[-1] + X_PAD)

# --- Legend: lower right of the first bank's block ---
# On ax_right, which matplotlib draws after ax_left, so the hatched precursor
# boxes cannot cover it. The anchor is the right edge of that block's last box in
# data coordinates, with the height as an axes fraction.
_first_bank_idx = [i for i, b in enumerate(bank_of_box) if b == BANKS[0]]
LEGEND_X = positions_right[_first_bank_idx[-1]] + 0.3   # right edge of the last box
LEGEND_Y = 0.02                                         # axes fraction above the floor
legend_handles = [
    Patch(facecolor=BG_COLOR, edgecolor="black", linewidth=MARKER_LW, label="Protein groups"),
    Patch(facecolor=BG_COLOR, edgecolor="black", linewidth=MARKER_LW, hatch="///", label="Precursors"),
]
ax_right.legend(handles=legend_handles, loc="lower right",
                bbox_to_anchor=(LEGEND_X, LEGEND_Y), bbox_transform=top_right)

fig.canvas.draw()
assert ax_right.spines["right"].get_visible(), "right spine was removed after plotting"
assert ax_right.get_yticklabels(), "right axis has no tick labels"

save_panel(fig, "a", "boxplot_pg_precursors", SI6_DIR)

# 4. Abundance rank plot (Figure 4a)

In [ ]:
# Figure 4a. Every protein ranked by its median intensity across all samples
# (adata_sf; median_intensity and protein_rank are set in the preprocessing
# cell), with landmark proteins labelled.
S_BG = native_pt(0.8) ** 2      # background dots, 0.8 pt across in print
S_HL = native_pt(2.4) ** 2      # labelled proteins, 2.4 pt across

highlight_genes = [
    "MBP",      # myelin basic protein
    "GFAP",     # glial fibrillary acidic protein
    "SNCA",     # alpha-synuclein
    "SYN1",     # synapsin-1
    "SYP",      # synaptophysin
    "SQSTM1",   # p62
]

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

at.pl.scatter(
    ax=ax, data=adata_sf.var, x_column="protein_rank", y_column="median_intensity",
    color=BG_COLOR, scatter_kwargs={"s": S_BG, "linewidths": 0},
)
ax.set_yscale("log")
# The exponent in a 10^n tick label is set at 70 % size and prints below 5 pt,
# so the ticks show the exponent itself and the axis label says log10.
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{np.log10(v):.0f}"))
# Ticks at the powers of ten only: the log scale adds minor ticks at 2-9 x each power.
ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())

# --- Match against every symbol in the protein group ---
# var["genes"] is semicolon-separated, so .isin() misses a gene that shares a
# group with another (e.g. "SNCA;SNCB").
_want = set(highlight_genes)
_hit = adata_sf.var["genes"].fillna("").apply(
    lambda s: next((g for g in s.split(";") if g in _want), None)
)
highlight = adata_sf.var[_hit.notna()].copy()
highlight["label"] = _hit[_hit.notna()]

# One protein group per symbol: the most abundant, since abundance is the axis.
_n_before = len(highlight)
highlight = (highlight.sort_values("median_intensity", ascending=False)
                      .drop_duplicates("label"))
if len(highlight) < _n_before:
    print(f"collapsed {_n_before - len(highlight)} duplicate protein groups "
          f"(kept the highest median per symbol)")

_missing = _want - set(highlight["label"])
if _missing:
    print(f"WARNING: not detected, no label drawn: {sorted(_missing)}")

ax.scatter(highlight["protein_rank"], highlight["median_intensity"], s=S_HL,
           color=HIGHLIGHT_COLOR, edgecolors="black", linewidths=MARKER_LW, zorder=5)

# --- Labels placed in empty space by adjust_text ---
texts = [
    ax.text(row["protein_rank"], row["median_intensity"], row["label"],
            ha="left", va="center", fontsize=ANNOT_FS)
    for _, row in highlight.iterrows()
]

# Every 5th point of the curve is passed as an obstacle, so adjust_text moves
# the labels off the curve.
adjust_text(
    texts,
    x=adata_sf.var["protein_rank"].values[::5],
    y=adata_sf.var["median_intensity"].values[::5],
    ax=ax,
    arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW),
    expand=(1.5, 1.0),         # least distance from the points and between labels
    force_static=(1.0, 1.5),   # repulsion from the points
    force_text=(0.5, 1.0),
    force_explode=(0.5, 0.8),  # initial spread, which separates overlapping labels
)

label_axes(ax, xlabel="Protein rank", ylabel="log10 median intensity")

save_panel(fig, "a", "rank_abundance", FIG4_DIR)

# 5. Coefficient of variation per brain bank and cell type (Figure S6b)

In [ ]:
# Figure S6b. CV of every protein per brain bank x cell type (groups of section
# 2.4): standard deviation over mean of the linear intensities (ddof = 1), for
# proteins with >= 3 values in the group. The dashed line marks a CV of 20 %.

# Labels on one line, in the short names of Figure 4e: rotated two-line labels
# touch their neighbours at the 3.4 mm spacing of a 1 x 1 panel.
celltype_display = {
    "Round Aggregate":     "EA round",
    "Elongated Aggregate": "EA elong.",
}
group_info = {key: (bank, ct) for bank, ct, key in BANK_CT_GROUPS}

# One long table in panel order: violinplot draws the groups in the order they
# first appear in it.
cv_parts = []
for bank, ct, key in BANK_CT_GROUPS:
    subset = adata[adata.obs[BANK_CT_COL] == key, :]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=RuntimeWarning)
        mean_vals = np.nanmean(subset.X, axis=0)
        std_vals  = np.nanstd(subset.X, axis=0, ddof=1)
        cvs = std_vals / mean_vals
        cvs[np.sum(~np.isnan(subset.X), axis=0) < 3] = np.nan
    cv_parts.append(pd.DataFrame({"Group": key, "CV": cvs[np.isfinite(cvs)]}))
cv_df = pd.concat(cv_parts, ignore_index=True)

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.grid(False)        # the 20 % reference line would be drawn on top of a grid line

at.pl.violinplot(
    ax=ax, data=cv_df, grouping_column="Group", value_column="CV",
    color_dict={key: CELLTYPE_COLORS[ct] for key, (bank, ct) in group_info.items()},
)

# The order is read back from the axes: violinplot sets the tick labels from the
# groups it drew, and the separators and bank names below need each bank in one
# block.
groups = [t.get_text() for t in ax.get_xticklabels()]
xs = ax.get_xticks()
_present = set(cv_df["Group"])
_expected = [key for key in group_info if key in _present]
assert groups == _expected, f"violinplot drew {groups}, expected {_expected}"
bank_of = [group_info[g][0] for g in groups]

# violinplot fills at 50 % opacity; the cell-type colours are shown at full
# strength, as in Figure S6a.
bodies = [c for c in ax.collections if isinstance(c, mpl.collections.PolyCollection)]
assert len(bodies) == len(groups), f"{len(bodies)} violin bodies for {len(groups)} groups"
for body, g in zip(bodies, groups):
    body.set_facecolor(CELLTYPE_COLORS[group_info[g][1]])

ax.set_ylim(0, cv_df["CV"].quantile(0.99) * 1.15)
ax.axhline(0.2, color=REF_COLOR, linestyle="--", linewidth=AXES_LW, zorder=0)    # CV 20 %

# Bank separators, halfway between the last group of one bank and the first of the next
for i in range(1, len(groups)):
    if bank_of[i] != bank_of[i - 1]:
        ax.axvline((xs[i - 1] + xs[i]) / 2, color=REF_COLOR, linewidth=THIN_LW,
                   linestyle="--", zorder=0)

ax.set_xticklabels([celltype_display.get(group_info[g][1], group_info[g][1]) for g in groups],
                   rotation=90)

# Mean CV per group above the violins, in two staggered rows: in one row the
# values (4.3 mm wide in bold) overlap at the 3.4 mm spacing of a 1 x 1 panel.
# The bank names sit above both rows. Offsets in points.
GAP = native_pt(1.5)           # axes top to the lower row of means
ROW = 1.25 * ANNOT_FS          # height of one row of means
top = blended_transform_factory(ax.transData, ax.transAxes)
for i, (x, g) in enumerate(zip(xs, groups)):
    ax.annotate(f"{cv_df.loc[cv_df['Group'] == g, 'CV'].mean():.2f}", (x, 1),
                xycoords=top, xytext=(0, GAP + (i % 2) * ROW), textcoords="offset points",
                ha="center", va="bottom", fontsize=ANNOT_FS, fontweight="bold")
for bank in BANKS:
    _x = [x for x, b in zip(xs, bank_of) if b == bank]
    if _x:
        ax.annotate(bank, (np.mean(_x), 1), xycoords=top,
                    xytext=(0, GAP + 2 * ROW), textcoords="offset points",
                    ha="center", va="bottom", fontsize=LABEL_FS, fontweight="bold")

label_axes(ax, ylabel="CV per protein")

save_panel(fig, "b", "cv_per_bank_celltype", SI6_DIR)

# 6. PCA

## 6.1 Fractions (Figure 4b)

In [ ]:
# Figure 4b. PCA of all samples on the proteins quantified in every sample
# (adata_pca, log2, no imputation), coloured by CellType_Aggregate (disease
# status collapsed). Colours from CELLTYPE_AGG_COLORS (section 2.2).
celltype_agg_col = "CellType_Aggregate"

_missing = sorted(set(adata_pca.obs[celltype_agg_col].dropna()) - set(CELLTYPE_AGG_COLORS))
if _missing:
    print(f"WARNING: no colour defined for: {_missing}")

DOT         = native_pt(2.0)   # sample dots, 2.0 pt across in print
DOT_ALPHA   = 0.85
# Legend in two rows: one entry per cell type, its lighter and darker shade side
# by side. Seven single entries in two rows are 63 mm wide at 6 pt; these four
# are 40.5 mm, the panel 42 mm. Listed column by column (legends fill
# column-major), so the rows read Neuron, Glia / Neuropil, EA.
LEGEND_PAIRS = [
    ("Neuron DAB−/+",   ["Neuron Without Aggregate", "Neuron DAB Positive"]),
    ("Neuropil",        ["Neuropil Without Aggregate"]),
    ("Glia DAB−/+",     ["Glia Without Aggregate", "Glia DAB Positive"]),
    ("EA elong./round", ["Elongated Aggregate Aggregate", "Round Aggregate Aggregate"]),
]
LEGEND_NCOL = 2

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

at.pl.plot_pca(
    ax=ax, data=adata_pca, x_column=1, y_column=2, label=False,
    color_map_column=celltype_agg_col, color_dict=CELLTYPE_AGG_COLORS,
    legend="auto",
    scatter_kwargs={"s": DOT ** 2, "alpha": DOT_ALPHA,
                    "linewidth": MARKER_LW, "edgecolor": "black"},
)

# plot_pca draws the dots at zorder 1, below the grid (1.5); the grid goes below
# the data.
ax.set_axisbelow(True)

# The legend is rebuilt below the plot, in the order and with the short names
# above. fig.legend with an "outside" location: constrained_layout reserves the
# space for it. An axes legend anchored outside the axes is counted as part of
# the axes by the layout engine, which then shrinks the plot.
leg = ax.get_legend()
handle_map = {t.get_text(): h for h, t in zip(leg.legend_handles, leg.texts)}
leg.remove()
_paired = {k for _, keys in LEGEND_PAIRS for k in keys}
if _paired != set(handle_map):
    print(f"WARNING: legend pairs and plotted levels differ: {sorted(_paired ^ set(handle_map))}")
pair_handles, pair_labels = [], []
for label, keys in LEGEND_PAIRS:
    hs = tuple(handle_map[k] for k in keys if k in handle_map)
    if hs:
        pair_handles.append(hs)
        pair_labels.append(label)
# ndivide=2: every square has the same size; the single Neuropil square takes
# the left place.
fig.legend(pair_handles, pair_labels,
           loc="outside lower center", ncol=LEGEND_NCOL, frameon=False, borderaxespad=0,
           handlelength=1.4, handletextpad=0.3, columnspacing=0.6,
           handler_map={tuple: mpl.legend_handler.HandlerTuple(ndivide=2, pad=0.2)})

save_panel(fig, "b", "pca_celltype_aggregate", FIG4_DIR)

## 6.2 Plate, sex, donor and brain bank (Figures S6c–f)

In [ ]:
# Figures S6c-f. The PCA of Figure 4b, coloured by one sample annotation per panel.
confounder_cols = ["Plate", "Sex", "Brain", "BrainBank"]
CONFOUNDER_LETTER = {"Plate": "c", "Sex": "d", "Brain": "e", "BrainBank": "f"}
# The annotation is the panel title; the legend below the axes has none.
CONFOUNDER_TITLE = {"Plate": "Plate", "Sex": "Sex", "Brain": "Brain",
                    "BrainBank": "Brain bank"}
# Short legend labels; the obs values are unchanged, and a level missing here
# keeps its name. Brains are numbered within each group in the order of their IDs.
LEVEL_SHORT = {
    "Plate": {"Plate1": "1", "Plate4": "4", "Plate1B2": "1B2", "Plate2B3": "2B3",
              "Plate3C2": "3C2", "Plate4C3": "4C3"},
    "Brain": {"Control 1": "Ctrl 1", "Control 2": "Ctrl 2", "Control 3": "Ctrl 3",
              "LBD 1": "Muc 1", "LBD 2": "Muc 2", "LBD 3": "Muc 3",
              "MTG 20 03": "BH 1", "MTG 22 30": "BH 2", "MTG 22 61": "BH 3",
              "MTG 24 15": "BH 4"},
}

# The legends go below the axes: inside, the "Brain" legend is wider than the
# axes of a 1 x 1 panel. Columns per annotation: two rows where they fit in the
# 42 mm panel; "Brain" needs three rows (in two rows its five columns are 48 mm
# wide). Labels longer than LABEL_WRAP characters wrap onto a second line.
NCOLS = {"Plate": 3, "Sex": 2, "Brain": 4, "BrainBank": 2}
LABEL_WRAP = 15
DOT = native_pt(2.0)   # sample dots, 2.0 pt across in print, as in Figure 4b
# The default qualitative palette has 9 colours and repeats from the 10th level;
# "Brain" has 10 donors, so a dark grey is appended as a 10th colour.
PALETTE = list(at.pl.BasePalettes.get("qualitative")) + ["#4d4d4d"]

for col in confounder_cols:
    if col not in adata_pca.obs.columns:
        print(f"Skipping {col!r} — not in obs")
        continue

    levels = adata_pca.obs[col].dropna().astype(str).unique()
    n_levels = len(levels)
    ncols = NCOLS.get(col, 1)
    if n_levels > len(PALETTE):
        print(f"!! {col}: {n_levels} levels for {len(PALETTE)} colours; colours repeat")

    fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
    ax = axm.next()

    at.pl.plot_pca(
        ax=ax, data=adata_pca, x_column=1, y_column=2, label=False,
        color_map_column=col, palette=PALETTE,
        legend="auto",
        scatter_kwargs={"s": DOT ** 2, "alpha": 0.7, "linewidth": MARKER_LW, "edgecolor": "black"},
    )
    # plot_pca draws the dots at zorder 1, below the grid (1.5); the grid goes
    # below the data.
    ax.set_axisbelow(True)
    ax.set_title(CONFOUNDER_TITLE.get(col, col), fontsize=TITLE_FS)

    # plot_pca builds the legend from its own config; it is rebuilt below the
    # axes with the short labels and without a title.
    leg = ax.get_legend()
    if leg is not None:
        handles = leg.legend_handles
        short   = LEVEL_SHORT.get(col, {})
        labels  = [textwrap.fill(short.get(t.get_text(), t.get_text()), LABEL_WRAP)
                   for t in leg.texts]
        leg.remove()
        fig.legend(handles=handles, labels=labels,
                   loc="outside lower center", ncols=ncols, frameon=False, borderaxespad=0,
                   handlelength=1.0, handletextpad=0.4, columnspacing=0.6)

    print(f"CHECK — {col}: {n_levels} levels, legend below in {ncols} column(s)")

    safe_name = col.replace(" ", "_")
    save_panel(fig, CONFOUNDER_LETTER[col], f"pca_confounder_{safe_name}", SI6_DIR)

# 7. Differential abundance, donor level

In [ ]:
def _mask_for(adata, criteria):
    """Combine equality filters across obs columns into one boolean mask.
    A list/tuple/set value matches if the column is ANY of the listed values."""
    m = np.ones(adata.n_obs, dtype=bool)
    for col, val in criteria.items():
        if col not in adata.obs.columns:
            raise KeyError(f"Column {col!r} not in adata.obs")
        if isinstance(val, (list, tuple, set)):
            m &= adata.obs[col].astype(str).isin([str(v) for v in val]).values
        else:
            m &= (adata.obs[col].astype(str) == str(val)).values
    return m

In [ ]:
# Fraction label of every sample, from obs["CellType_Aggregate"]; neuron and glia
# keep their DAB status. Sections 7 and 10 select their groups by it.
fraction_map = {
    "Neuron Without Aggregate":      "Neuron | Without Aggregate",
    "Neuron DAB Positive":           "Neuron | DAB Positive",
    "Glia Without Aggregate":        "Glia | Without Aggregate",
    "Glia DAB Positive":             "Glia | DAB Positive",
    "Neuropil Without Aggregate":    "Neuropil",
    "Round Aggregate Aggregate":     "Round Aggregate",
    "Elongated Aggregate Aggregate": "Elongated Aggregate",
}
adata_log.obs["fraction"] = adata_log.obs["CellType_Aggregate"].map(fraction_map)
unmapped = adata_log.obs["fraction"].isna().sum()
if unmapped:
    print(f"WARNING: {unmapped} samples have no fraction mapping. "
          f"Unique CellType_Aggregate not in map: "
          f"{set(adata_log.obs.loc[adata_log.obs['fraction'].isna(),'CellType_Aggregate']) }")

## 7.1 Round inclusion against DAB+ host neuron, region-paired

In [ ]:
# ============================================================================
# Round inclusion against DAB+ host neuron: region-paired, donor-level, limma
# with empirical Bayes moderation. Figures 4d and 4f, and the y axis of S6i.
#
#   1. within donor x region: mean log2 of the round-inclusion pools minus the
#      mean log2 of the DAB+ neuron wells
#   2. these region deltas averaged per donor, unweighted, so every donor
#      contributes one number per protein
#   3. limma intercept-only fit on the donor deltas, with empirical Bayes
#      moderation. An intercept-only model tests mean(delta) = 0, the paired
#      one-sample test, with each protein's variance shrunk toward a prior fitted
#      across all proteins. With 3 to 6 donors an unmoderated t-test has 2 to 5
#      degrees of freedom, and a small p-value then results more often from an
#      underestimated variance than from a large effect.
#
# .X carries no normalisation, only log2. A loading difference between the two
# well types appears as a global shift in every delta; the printed median delta
# shows it.
# ============================================================================
CONTRAST_NAME = "Round_vs_NeuronDABpos_Disease_regionpaired"
GROUP_A = {"fraction": "Round Aggregate",       "disease_binary": "Disease"}
GROUP_B = {"fraction": "Neuron | DAB Positive", "disease_binary": "Disease"}
DONOR_COL, REGION_COL = "Brain", "Brain Region"
MIN_DONORS_PAIRED = 3

_ad = adata_log.copy()
_ad.obs["fraction"] = _ad.obs["CellType_Aggregate"].map(fraction_map)

# The two banks name the cortex differently (middle temporal gyrus, cortex); one
# name, so the cortex wells of a donor form one region.
REGION_HARMONISE = {"Middle Temporal Gyrus": "Cortex"}
_ad.obs["region_h"] = _ad.obs[REGION_COL].replace(REGION_HARMONISE)

mask_a = _mask_for(_ad, GROUP_A)
mask_b = _mask_for(_ad, GROUP_B)
_var = _ad.var_names.tolist()
_gene_map = _ad.var["single_gene_name"].to_dict()

def _cell_means(mask):
    """(donor, region) × proteins matrix of mean log2 intensity."""
    sub = _ad[mask, :]
    df = sub.to_df()
    df["__donor"]  = sub.obs[DONOR_COL].astype(str).values
    df["__region"] = sub.obs["region_h"].astype(str).values
    return df.groupby(["__donor", "__region"]).mean().reindex(columns=_var)

A = _cell_means(mask_a)
B = _cell_means(mask_b)

# --- Step 1: region deltas, only where both sides exist in donor x region
shared = A.index.intersection(B.index)
if len(shared) == 0:
    raise ValueError("No donor × region has both round aggregates and DAB+ neurons.")
region_delta = A.loc[shared] - B.loc[shared]

print("Donor × region pairs contributing (both sides present):")
_counts = pd.DataFrame({
    "n_wells_round":  _ad[mask_a].obs.groupby([_ad[mask_a].obs[DONOR_COL].astype(str),
                                               _ad[mask_a].obs["region_h"].astype(str)],
                                              observed=True).size(),
    "n_wells_dabpos": _ad[mask_b].obs.groupby([_ad[mask_b].obs[DONOR_COL].astype(str),
                                               _ad[mask_b].obs["region_h"].astype(str)],
                                              observed=True).size(),
}).reindex(shared)
print(_counts.to_string())

# --- Step 2: unweighted mean across the regions of a donor, one row per donor
donor_delta = region_delta.groupby(level="__donor").mean()
print(f"\nRegions averaged per donor:\n{region_delta.groupby(level='__donor').size().to_string()}")
print(f"\nDonors contributing: {len(donor_delta)}  ({list(donor_delta.index)})")

# Loading check: without normalisation, a global offset is a difference in
# loading. A median far from zero shifts the whole volcano to one side.
_global = np.nanmedian(donor_delta.values)
print(f"\nMedian delta across all proteins and donors: {_global:+.3f} log2 "
      f"({'near zero — no obvious loading offset' if abs(_global) < 0.1 else 'NOT near zero — global offset present'})")

if len(donor_delta) < MIN_DONORS_PAIRED:
    raise ValueError(f"Only {len(donor_delta)} donors; need ≥{MIN_DONORS_PAIRED}.")

# --- Step 3: limma with empirical Bayes moderation on the complete proteins ---
# limma needs complete rows: a protein must have a delta in every contributing
# donor, that is, be detected on both sides in every donor x region pair used.
complete = donor_delta.columns[donor_delta.notna().all(axis=0)]
print(f"\nProteins complete across all {len(donor_delta)} donors: "
      f"{len(complete)} / {donor_delta.shape[1]}")
if len(complete) == 0:
    raise ValueError("No protein is complete across all donors — use adata_strict_log "
                     "as the input, or drop the sparsest donor.")

E = donor_delta[complete].T                      # proteins × donors, as limma expects
design = pd.DataFrame({"delta": np.ones(E.shape[1])}, index=E.columns)

fit = eBayes(lmFit(E, design))
_coef = fit.coefficients.columns[0]              # lmFit renames it to "column0"
# sort_by="none" is documented but rejected by inmoose's validation list, so the
# table is sorted by p and reindexed to the protein order.
tt = topTable(fit, coef=_coef, number=E.shape[0], sort_by="P").reindex(E.index)

D_c = donor_delta[complete].values
res_eb = pd.DataFrame({
    "protein": list(E.index),
    "single_gene_name": [_gene_map.get(v, v) for v in E.index],
    "log2fc":  tt["log2FoldChange"].values,      # = mean donor delta
    "p_value": tt["pvalue"].values,
    "fdr":     tt["adj_pvalue"].values,
    "n_donors": D_c.shape[0],
    "n_sign_consistent": (np.sign(D_c) == np.sign(tt["log2FoldChange"].values)).sum(axis=0),
    "test": "region-paired eBayes",
})
res_eb["-log10(fdr)"] = -np.log10(np.clip(res_eb["fdr"], 1e-300, None))

print(f"\neBayes: tested={len(res_eb)}, FDR<0.05={int((res_eb['fdr'] < 0.05).sum())}, "
      f"df.prior={np.ravel(fit.df_prior)[0]:.1f}, df.residual={np.ravel(fit.df_residual)[0]:.0f}")

# Figures 4d and S6i read the result from the kernel. Section 12.1 reads the two
# CSV files, so it does not need this cell in the same kernel.
ttest_results_regionpaired = {CONTRAST_NAME: res_eb}

res_eb.to_csv(f"{output_dir}/{today}_ebayes_regionpaired_{CONTRAST_NAME}.csv", index=False)
donor_delta.to_csv(f"{output_dir}/{today}_donordelta_regionpaired_{CONTRAST_NAME}.csv")

## 7.2 Neuron DAB− from disease donors against neurons from control donors

In [ ]:
# Neuron DAB− from disease donors against neuron DAB− from control donors,
# donor-level limma with empirical Bayes moderation. Figure S6h and the x axis of
# S6i.
# Disease and control donors are different people, so the test is unpaired: the
# wells of each donor are averaged to one value per protein (all regions), and
# at.tl.diff_exp_ebayes compares the disease donor means with the control donor
# means. It drops every protein with a missing donor mean.
DVC_NAME = "Neuron_DABneg_Disease_vs_Control"
DVC_GROUP_A = {"fraction": "Neuron | Without Aggregate", "disease_binary": "Disease"}
DVC_GROUP_B = {"fraction": "Neuron | Without Aggregate", "disease_binary": "Control"}
DVC_MIN_DONORS = 3

_ad = adata_log.copy()
_ad.obs["fraction"] = _ad.obs["CellType_Aggregate"].map(fraction_map)

# One row per donor and group: the mean log2 intensity over that donor's wells.
_rows, _index, _side = [], [], []
for _label, _group in (("A", DVC_GROUP_A), ("B", DVC_GROUP_B)):
    _mask = _mask_for(_ad, _group)
    _donors = sorted(_ad.obs.loc[_mask, "Brain"].astype(str).unique())
    if len(_donors) < DVC_MIN_DONORS:
        raise ValueError(f"{_label}: {len(_donors)} donors; need {DVC_MIN_DONORS}.")
    for _d in _donors:
        _sel = _mask & (_ad.obs["Brain"].astype(str).values == _d)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)   # proteins absent in every well
            _rows.append(np.nanmean(_ad.X[_sel], axis=0))
        _index.append(f"{_d}__{_label}")
        _side.append(_label)
_pb = ad.AnnData(X=np.vstack(_rows), obs=pd.DataFrame({"__contrast": _side}, index=_index),
                 var=_ad.var.copy())

_, res_dvc = at.tl.diff_exp_ebayes(_pb, "__contrast", ("A", "B"))
res_dvc["single_gene_name"] = res_dvc["protein"].map(_ad.var["single_gene_name"])

print(f"{DVC_NAME}: donors disease {_side.count('A')}, control {_side.count('B')}; "
      f"tested {len(res_dvc)}, FDR<0.05 {int((res_dvc['fdr'] < 0.05).sum())}, "
      f"smallest FDR {res_dvc['fdr'].min():.3f}")
res_dvc.to_csv(f"{output_dir}/{today}_ebayes_donor_{DVC_NAME}.csv", index=False)

# 8. Volcano plot, round inclusion against host neuron (Figure 4d)

In [ ]:
# Genes labelled in Figure 4d.
volcano_highlights = [
    # UP — synaptic vesicle, SNARE, active-zone scaffold, V-ATPase
    "SYP", "SNAP25", "BSN", "ATP6V0A1",
    # DOWN — chromatin (nuclear), RNA processing
    "H2AX", "HNRNPC",
]

In [ ]:
# ============================================================================
# Figure 4d. Volcano plot of the region-paired donor-level fit (section 7.1):
# round inclusion against its DAB+ host neuron. Significant: FDR < 0.05 and
# |log2FC| > 0.5.
# ============================================================================
name   = CONTRAST_NAME                     # "Round_vs_NeuronDABpos_Disease_regionpaired"
result = ttest_results_regionpaired[name].copy()

FDR_CUT, LFC_CUT = 0.05, 0.5
DOT_BG, DOT_HL   = native_pt(1.0), native_pt(2.4)   # dot diameters in print
ALPHA_BG         = 0.35       # significant background points
ALPHA_NS         = 0.30       # non-significant points

# Label budget per side: more labels than this crowd the 1 x 1 panel.
MAX_LABELS_PER_SIDE = 5

result["status"] = "unchanged"
result.loc[(result["fdr"] < FDR_CUT) & (result["log2fc"] >  LFC_CUT), "status"] = "upregulated"
result.loc[(result["fdr"] < FDR_CUT) & (result["log2fc"] < -LFC_CUT), "status"] = "downregulated"

n_up   = int((result["status"] == "upregulated").sum())
n_down = int((result["status"] == "downregulated").sum())

# --- Which genes to label ---
# The fit tests only proteins with a delta in every donor, so a listed gene can be
# absent; the print names it.
requested = set(volcano_highlights)
_present = set(result["single_gene_name"]) & requested
_absent  = requested - _present
print(f"Labelled genes: {len(requested)} requested, {len(_present)} found in this result.")
if _absent:
    print(f"  NOT in this result — either not detected or dropped as "
          f"incomplete across donors: {sorted(_absent)}")
highlight_genes = _present

# The Results state that the labelled genes change in the same direction in all
# five donors.
print("CHECK — labelled genes, FDR and donors with the sign of the mean:")
print(result.loc[result["single_gene_name"].isin(requested),
                 ["single_gene_name", "log2fc", "fdr", "n_sign_consistent", "n_donors"]]
      .to_string(index=False))

# Trim to the label budget, keeping the strongest effect on each side.
_cand = result[result["single_gene_name"].isin(highlight_genes)]
_keep = pd.concat([
    _cand[_cand["log2fc"] > 0].nlargest(MAX_LABELS_PER_SIDE, "log2fc"),
    _cand[_cand["log2fc"] < 0].nsmallest(MAX_LABELS_PER_SIDE, "log2fc"),
])
_dropped = sorted(set(_cand["single_gene_name"]) - set(_keep["single_gene_name"]))
if _dropped:
    print(f"  Not labelled (over the {MAX_LABELS_PER_SIDE}-per-side budget, "
          f"weakest effects): {_dropped}")
is_highlight = result["protein"].isin(_keep["protein"])
print(f"  Drawing {int(is_highlight.sum())} labels.")

# --- Plot ---
STATUS_COLOR = {
    "upregulated":   UP_COLOR,
    "downregulated": DOWN_COLOR,
    "unchanged":     BG_COLOR,
}

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.spines[["top", "right"]].set_visible(True)     # boxed, as the volcano plots of Figures 2 and 3

at.pl.volcano(
    data=result, x_column="log2fc", y_column="-log10(fdr)", ax=ax,
    x_thresholds=(-LFC_CUT, LFC_CUT),
    layers=[
        ("status", "upregulated",   "up",   {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": ALPHA_BG, "linewidth": 0}}),
        ("status", "downregulated", "down", {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": ALPHA_BG, "linewidth": 0}}),
        ("status", "unchanged",     "ns",   {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": ALPHA_NS, "linewidth": 0}}),
    ],
    color_dict={"up": UP_COLOR, "down": DOWN_COLOR, "ns": BG_COLOR},
    label_layers=None, display_id_column="single_gene_name",
    scatter_kwargs={"s": DOT_BG ** 2}, legend=None,
)
# at.pl.volcano sets the title "Volcano Plot" and has no parameter for it.
ax.set_title("")

# The only lines on the axes so far are the threshold lines, which the volcano
# function draws 1 pt wide on the canvas (0.22 pt in print, below the minimum).
for _line in ax.lines:
    _line.set(linewidth=THIN_LW, color=REF_COLOR)

# Axis limits, set with matplotlib so they do not depend on the alphapepttools
# version. x: 1.1 x the largest |log2FC| on both sides, plus 0.4 on the right for
# the ATP6V0A1 label. y: up to 1.2 x the largest value (the volcano function uses
# 1.1), so the label above the top row of dots stays below the top spine.
X_MAX = result["log2fc"].abs().max() * 1.1
ax.set_xlim(-X_MAX, X_MAX + 0.4)
ax.set_ylim(0, result["-log10(fdr)"].max() * 1.2)

# Labelled genes are redrawn opaque and larger, in the colour of their direction.
labeled = result[is_highlight]
ax.scatter(labeled["log2fc"], labeled["-log10(fdr)"],
           c=[STATUS_COLOR[s] for s in labeled["status"]],
           s=DOT_HL ** 2, edgecolors="black", linewidths=MARKER_LW, zorder=5)

# Legend inside the axes, lower left, one column. In two columns the four entries
# are 36.3 mm wide, wider than the 34.2 mm axes, and the lower right holds the
# cloud of non-significant points. The direction is in the x label, the entries
# carry the counts, and the thresholds are stated in the figure legend text.
_entry = dict(marker="o", linestyle="none", markersize=DOT_HL, markeredgewidth=MARKER_LW)
legend_handles = [
    Line2D([0], [0], markerfacecolor=UP_COLOR, markeredgecolor="none",
           label=f"Enriched ({n_up})", **_entry),
    Line2D([0], [0], markerfacecolor=DOWN_COLOR, markeredgecolor="none",
           label=f"Depleted ({n_down})", **_entry),
    Line2D([0], [0], markerfacecolor=BG_COLOR, markeredgecolor="none",
           label="Not significant", **_entry),
    Line2D([0], [0], markerfacecolor="none", markeredgecolor="black",
           label="Labelled", **_entry),
]
ax.legend(handles=legend_handles, loc="lower left", frameon=False,
          borderaxespad=0.3, borderpad=0.2, labelspacing=0.3,
          handlelength=1.0, handletextpad=0.4)

label_axes(ax, xlabel="log2FC (EA round/ Neuron DAB+)", ylabel="−log10 FDR")

# Labels in white boxes, each at a fixed offset from its dot in printed points
# (dx, dy): negative dx = left, negative dy = down; the alignment follows from the
# signs. A label more than 3 pt from its dot gets a grey leader line.
LABEL_OFFSET = {
    "H2AX":     (-2.4,   0),
    "HNRNPC":   (   0, -2.4),
    "BSN":      (-2.4,   0),
    "ATP6V0A1": (   2,   3),   # above the top row of dots, right of the cluster
    "SYP":      (   3,   0),
    "SNAP25":   (   5, -15),   # below SYP; the leader passes 0.3 mm below the SYP dot
}
_box = dict(boxstyle="round,pad=0.1", facecolor="white", edgecolor="none", alpha=0.85)
for _, row in labeled.iterrows():
    g = row["single_gene_name"]
    if g not in LABEL_OFFSET:
        print(f"  {g}: no entry in LABEL_OFFSET, placed next to its dot")
    dx, dy = LABEL_OFFSET.get(g, (-2.4 if row["log2fc"] < 0 else 2.4, 0))
    ax.annotate(g, (row["log2fc"], row["-log10(fdr)"]),
                xytext=(native_pt(dx), native_pt(dy)), textcoords="offset points",
                ha="right" if dx < 0 else ("left" if dx > 0 else "center"),
                va="center" if dy == 0 else ("top" if dy < 0 else "bottom"),
                fontsize=ANNOT_FS, zorder=10, bbox=_box,
                arrowprops=(dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW,
                                 shrinkA=0, shrinkB=DOT_HL / 2)
                            if np.hypot(dx, dy) > 3 else None))

save_panel(fig, "d", f"volcano_{name}", FIG4_DIR)

# 9. Marker composition

## 9.1 Markers per fraction (Figure 4c)

In [ ]:
# Marker genes of Figure 4c, grouped by cell class and compartment, in row order.
marker_groups = {
    "Nuclear/\ncell body":  ["H2AX", "LMNA", "HNRNPC", "NCL", "PARP1", "RANBP2"],
    "Neuron soma":          ["RBFOX3", "RBFOX1", "ELAVL2", "ELAVL3", "NOVA2", "CELF2", "CELF4", "RALYL", "MAP2"],
    "Synaptic":  ["SYP", "SLC17A7", "VAMP2", "SNAP25", "SYN1", "SYT1", "BSN"],
    "Axonal":    ["NEFM", "NEFL", "NEFH"],
    "Glia":                 ["GFAP", "S100B", "AQP4", "ALDH1L1", "SLC1A2", "MBP", "PLP1", "CNP", "MAG"],
    "Aggregate":            ["SNCA", "SNCB", "SQSTM1", "SCG2"],
}

all_groups = list(marker_groups.items())   # [(label, [genes]), ...]
row_order = [g for _, genes in all_groups for g in genes]

In [ ]:
# ============================================================
# Figure 4c. Mean log2 intensity of each marker per cell type, z-scored per row
# across the five cell types. Input: adata_sf (all non-contaminant proteins), so
# no marker is lost to the completeness filter. Drawn by group_heatmap
# (src/heatmap.py).
# ============================================================
GROUP_COL = "Cell Type"
group_order = ["Neuron", "Glia", "Neuropil", "Round Aggregate", "Elongated Aggregate"]

# Short column labels, in the words of the Figure 4b legend
CELLTYPE_DISPLAY = {
    "Neuron":              "Neuron",
    "Glia":                "Glia",
    "Neuropil":            "Neuropil",
    "Round Aggregate":     "EA round",
    "Elongated Aggregate": "EA elongated",
}

adata_use = adata_sf.copy()
sc.pp.log1p(adata_use, base=2)

# --- marker -> protein group ---
# A marker is matched against every symbol of a protein group (var["genes"] is
# separated by ";"). When several groups carry it, the group with the highest
# mean intensity is kept, since mean intensity is what the panel shows.
_pid_mean = pd.Series(np.asarray(np.nanmean(adata_use.X, axis=0)).flatten(),
                      index=adata_use.var_names)
_want = set(row_order)
_hits = []
for pid, raw in adata_use.var["genes"].fillna("").items():
    hit = next((s.strip() for s in raw.split(";") if s.strip() in _want), None)
    if hit is not None:
        _hits.append((hit, pid, _pid_mean[pid]))

_gene_tbl = (pd.DataFrame(_hits, columns=["gene", "pid", "mean"])
               .sort_values("mean", ascending=False)
               .drop_duplicates("gene", keep="first"))
gene_to_pid = dict(zip(_gene_tbl["gene"], _gene_tbl["pid"]))

# Mean log2 intensity per marker and cell type
present_groups = [g for g in group_order if (adata_use.obs[GROUP_COL] == g).any()]
data = {}
for grp in present_groups:
    sub = adata_use[adata_use.obs[GROUP_COL] == grp]
    sub_means = np.asarray(np.nanmean(sub.X, axis=0)).flatten()
    means_by_pid = pd.Series(sub_means, index=adata_use.var_names)
    data[grp] = {g: means_by_pid.get(gene_to_pid[g], np.nan)
                 for g in row_order if g in gene_to_pid}

intensity_matrix = pd.DataFrame(data)
row_order_present = [g for g in row_order if g in intensity_matrix.index]
intensity_matrix = intensity_matrix.loc[row_order_present, present_groups]

# Row z-score (per protein across cell types)
z_matrix = intensity_matrix.sub(intensity_matrix.mean(axis=1), axis=0) \
                           .div(intensity_matrix.std(axis=1), axis=0)
z_matrix = z_matrix.replace([np.inf, -np.inf], np.nan)

group_heatmap(z_matrix.rename(columns=CELLTYPE_DISPLAY), all_groups,
              "c", "heatmap_marker_zscore", FIG4_DIR, size=(1, 2),
              cbar_label="Row\nz-score", col_rotation=90)

## 9.2 Regional markers of neurons and round inclusions (Figure S6g)

In [ ]:
# ============================================================
# Figure S6g, matrix. Markers of cortical excitatory neurons and of striatal
# medium spiny neurons: log2 intensity per sample, z-scored per row, for the
# neurons and round inclusions of the cortex (cortex and middle temporal gyrus)
# and of the striatum (nucleus accumbens, caudate, putamen). White matter is not
# mapped and drops out. Columns: region, then cell type, then donor.
# ============================================================
REGION_COL = "Brain Region"
ALLOWED_CELLTYPES = ["Neuron", "Round Aggregate"]

REGION_MAP = {
    # Cortex
    "Cortex":                "Cortex",
    "Middle Temporal Gyrus": "Cortex",
    # Striatum: ventral (nucleus accumbens) and dorsal (caudate, putamen)
    "Nucleus Accumben":      "Striatum",
    "Nucleus Caudatus":      "Striatum",
    "Putamen":               "Striatum",
}
region_order = ["Cortex", "Striatum"]

marker_groups_region = {
    "Cortical excitatory":    ["SLC17A7", "CAMK2A", "SATB2",
                               "TBR1", "CUX2", "NRGN", "FEZF2"],
    "Striatal MSN":           ["PCP4", "GNAL", "ADCY5", "PDE1B",
                               "ANKRD63", "CALB1", "RGS14"],
}
all_groups_region = list(marker_groups_region.items())
row_order_region = []
for _label, genes in all_groups_region:
    row_order_region.extend(genes)

adata_use = adata_sf.copy()
sc.pp.log1p(adata_use, base=2)
adata_use.obs["_region_pooled"] = adata_use.obs[REGION_COL].map(REGION_MAP)

# Samples with a mapped region and one of the two cell types
keep_mask = (
    adata_use.obs["_region_pooled"].isin(region_order) &
    adata_use.obs["Cell Type"].isin(ALLOWED_CELLTYPES)
)
adata_use = adata_use[keep_mask].copy()

# Column order: region, cell type, donor
sample_order, block_info = [], []   # block_info: (region, cell_type, n)
for reg in region_order:
    for ct in ALLOWED_CELLTYPES:
        sub = (adata_use.obs[
                  (adata_use.obs["_region_pooled"]    == reg) &
                  (adata_use.obs["Cell Type"] == ct)]
               .sort_values("Brain"))
        if len(sub):
            sample_order.extend(sub.index.tolist())
            block_info.append((reg, ct, len(sub)))
adata_use = adata_use[sample_order, :].copy()

print(f"\nSamples used: {adata_use.n_obs}")
for reg, ct, n in block_info:
    print(f"  {reg:>10} | {ct:>16} : {n} samples")

# One protein group per marker: the first group whose first symbol is the marker.
# No marker of this panel has a second protein group.
gene_to_pid = {}
for pid, gname in adata_use.var["single_gene_name"].items():
    gene_to_pid.setdefault(gname, pid)

X = adata_use.to_df()   # samples × proteins (log2)
data = {}
for g in row_order_region:
    if g in gene_to_pid and gene_to_pid[g] in X.columns:
        data[g] = X[gene_to_pid[g]].values

intensity_matrix_r = pd.DataFrame(data, index=adata_use.obs_names).T
row_present = [g for g in row_order_region if g in intensity_matrix_r.index]
intensity_matrix_r = intensity_matrix_r.loc[row_present, :]

# Row z-score across samples
z_matrix_r = (intensity_matrix_r.sub(intensity_matrix_r.mean(axis=1), axis=0)
                                 .div(intensity_matrix_r.std(axis=1), axis=0)
                                 .replace([np.inf, -np.inf], np.nan))

vmax = float(np.nanmax(np.abs(z_matrix_r.values)))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

In [ ]:
# ============================================================
# Figure S6g, plot. One column per sample, in blocks of region (outer, bold
# label) and cell type (inner).
#
# Colour: DIRECTION_CMAP, the diverging ramp of every signed panel. Missing values
# are grey (BG_COLOR): on a diverging map an unpainted cell would read as z = 0.
# seaborn does not draw NaN cells, so the grey is the axes background.
#
# The column labels are placed in printed points above the axes, so their
# distance from the axes does not depend on the number of rows.
# ============================================================
# Column labels in the short names of Figure 4c; "Round Aggregate" is wider than
# its block.
CELLTYPE_DISPLAY = {"Neuron": "Neuron", "Round Aggregate": "EA round"}

# Column-label stack above the axes, in points.
CT_LABEL_PT        = native_pt(1.5)                   # cell-type labels
REGION_LABEL_PT    = CT_LABEL_PT + 1.25 * ANNOT_FS    # region labels, one line higher
GROUP_LABEL_PAD_PT = native_pt(1.5)                   # gap between gene names and group labels

fig, axm = create_figure(1, 1, figsize=figsize(2, 1))
ax = axm.next()
ax.grid(False)

sns.heatmap(
    z_matrix_r, ax=ax, cmap=DIRECTION_CMAP, norm=norm, linewidths=0,
    cbar_kws={"shrink": 0.9, "pad": 0.02},
    annot=False,
)

ax.set_xticks([])
ax.set_xlabel("")
# Every row labelled; matplotlib thins the y tick labels otherwise.
ax.set_yticks(np.arange(len(z_matrix_r.index)) + 0.5)
ax.set_yticklabels(z_matrix_r.index, rotation=0)
ax.tick_params(axis="y", length=0)          # every row is labelled

# Two lines: along the bar of a one-unit panel the label fits only on two lines.
cbar = ax.collections[0].colorbar
cbar.set_label("Row z-score\n(log2 intensity)")

# Column separators and labels, in one pass over the blocks:
#   - thin grey dashed line between cell types within a region
#   - black line between regions
#   - cell-type label above each block
#   - region label, bold and larger, one line above the cell-type labels
# x in data coordinates, y = the axes top plus an offset in points.
_trans = blended_transform_factory(ax.transData, ax.transAxes)

running = 0
region_span = {}        # reg → [start, end]
for i, (reg, ct, n) in enumerate(block_info):
    block_start, block_end = running, running + n
    block_mid = (block_start + block_end) / 2

    ax.text(block_mid, 1.0, CELLTYPE_DISPLAY.get(ct, ct), ha="center", va="bottom",
            fontsize=ANNOT_FS, clip_on=False,
            transform=_trans + ScaledTranslation(0, CT_LABEL_PT / 72, fig.dpi_scale_trans))

    if i > 0:
        prev_reg = block_info[i - 1][0]
        if prev_reg == reg:
            ax.axvline(block_start, color=REF_COLOR, linewidth=THIN_LW, linestyle="--")
        else:
            ax.axvline(block_start, color="black", linewidth=AXES_LW)

    span = region_span.setdefault(reg, [block_start, block_end])
    span[1] = block_end
    running = block_end

for reg, (start, end) in region_span.items():
    ax.text((start + end) / 2, 1.0, reg, ha="center", va="bottom",
            fontsize=LABEL_FS, fontweight="bold", clip_on=False,
            transform=_trans + ScaledTranslation(0, REGION_LABEL_PT / 72, fig.dpi_scale_trans))

# Row separators and marker-group labels on the left
group_mids_actual, group_labels_actual, running = [], [], 0
for label, genes in all_groups_region:
    n = sum(1 for g in genes if g in z_matrix_r.index)
    if n > 0:
        if running > 0:
            ax.axhline(running, color="black", linewidth=THIN_LW)
        group_mids_actual.append(running + n / 2)
        group_labels_actual.append(label)
    running += n

# The group labels must clear the gene-name column, so the pad is measured from
# the rendered labels: from the axis line to the left edge of the widest name,
# which includes the tick pad in front of it.
fig.canvas.draw()
_names_pt = (ax.get_window_extent().x0
             - min(t.get_window_extent().x0 for t in ax.get_yticklabels())) / fig.dpi * 72

ax2 = ax.secondary_yaxis("left")
ax2.set_yticks(group_mids_actual)
ax2.set_yticklabels(group_labels_actual, fontweight="bold",
                    rotation=90, va="center", ha="right", ma="center")
ax2.tick_params(length=0, pad=_names_pt + GROUP_LABEL_PAD_PT)
for spine in ax2.spines.values():
    spine.set_visible(False)
ax.set_facecolor(BG_COLOR)   # NaN cells are not drawn; this grey shows through
ax.set_ylabel("")

# ----------------------------------------------------------------- CHECKS
_missing = [g for g in row_order_region if g not in z_matrix_r.index]
_nan = int(np.isnan(z_matrix_r.values).sum())
print(f"\nCHECK — matrix {z_matrix_r.shape[0]} markers x {z_matrix_r.shape[1]} samples")
print(f"CHECK — markers requested but not detected: {len(_missing)}"
      + (f" -> {_missing}" if _missing else ""))
print(f"CHECK — missing cells drawn in BG_COLOR, not white: {_nan} "
      f"({100 * _nan / z_matrix_r.size:.1f}%)")
print(f"CHECK — colour range symmetric at +/-{vmax:.2f} z")
# Capitals and digits are 0.72 of the font size tall, so rows closer than that overlap.
fig.canvas.draw()
_pitch = ax.get_window_extent().height / fig.dpi * 72 / len(z_matrix_r)
print(f"CHECK — row pitch {_pitch / DRAW_SCALE:.1f} pt for {TICK_FS / DRAW_SCALE:.1f} pt "
      f"gene names, {len(z_matrix_r)} rows (printed)"
      + ("   !! the names overlap" if _pitch < 0.75 * TICK_FS else ""))

save_panel(fig, "g", "heatmap_regional_NeuronVsRound_zscore", SI6_DIR)

# 10. Compartment composition

In [ ]:
# GO CC gene sets for sections 10, 12.1, 13.2 and 13.4: MSigDB GO cellular
# component, through decoupler, cached in GOCC_CACHE (section 2) so that every run
# uses the same release. The cache is built from MSigDB when the file is absent.
# Terms are selected by the GOCC_ prefix, because the collection name differs
# between MSigDB releases.
if not GOCC_CACHE.exists():
    print("Fetching MSigDB (live call, slow) ...")
    _all = dc.op.resource("MSigDB", organism="human")
    _sel = _all[_all["geneset"].astype(str).str.startswith("GOCC_")]
    if len(_sel) == 0:
        raise ValueError("No GOCC_ sets in this MSigDB release; inspect "
                         "_all['collection'].unique() and select by collection.")
    (_sel.rename(columns={"geneset": "source", "genesymbol": "target"})
         [["source", "target"]].drop_duplicates().reset_index(drop=True)
         .to_parquet(GOCC_CACHE, index=False))
    print(f"Cached -> {GOCC_CACHE}")
gocc = pd.read_parquet(GOCC_CACHE)[["source", "target"]]
print(f"GO CC: {gocc['source'].nunique()} terms in {GOCC_CACHE}")


def genes_for(terms):
    """Genes annotated to any of `terms`, and the terms absent from the gene sets."""
    found, missing = set(), []
    for t in terms:
        g = gocc.loc[gocc["source"] == t, "target"]
        if len(g):
            found.update(g.values)
        else:
            missing.append(t)
    return found, missing


# Terms excluded from the GSEA (section 12.1) and from the set-level replication
# (section 13.2).
# Container terms, which make no statement about composition:
GENERIC_COMPARTMENTS = {
    "GOCC_CYTOPLASM", "GOCC_CYTOSOL", "GOCC_MEMBRANE", "GOCC_ORGANELLE",
    "GOCC_INTRACELLULAR_MEMBRANE_BOUNDED_ORGANELLE", "GOCC_INTRACELLULAR_ORGANELLE",
    "GOCC_PROTEIN_CONTAINING_COMPLEX", "GOCC_CYTOPLASMIC_VESICLE",
    "GOCC_INTRACELLULAR_ANATOMICAL_STRUCTURE", "GOCC_ORGANELLE_MEMBRANE",
    "GOCC_CELL_BODY", "GOCC_NEURON_PROJECTION", "GOCC_SIDE_OF_MEMBRANE",
    "GOCC_MEMBRANE_PROTEIN_COMPLEX", "GOCC_CATALYTIC_COMPLEX",
}
# Structures absent from post-mortem cortex and nucleus accumbens, and terms whose
# proteins the contaminant filter removes. Synaptic, myelin, glial, vascular and
# blood-microparticle terms describe structures present in brain tissue and stay.
BRAIN_CC_ARTIFACTS = {
    # Germ cells
    "GOCC_SPERM_FIBROUS_SHEATH", "GOCC_SPERM_PRINCIPAL_PIECE",
    "GOCC_SPERM_FLAGELLUM", "GOCC_ACROSOMAL_VESICLE",
    # Retina
    "GOCC_PHOTORECEPTOR_OUTER_SEGMENT", "GOCC_PHOTORECEPTOR_DISC_MEMBRANE",
    "GOCC_PHOTORECEPTOR_INNER_SEGMENT",
    # Striated muscle; vessel walls carry smooth muscle without sarcomeres
    "GOCC_MYOFIBRIL", "GOCC_SARCOMERE", "GOCC_Z_DISC", "GOCC_I_BAND",
    "GOCC_A_BAND", "GOCC_MUSCLE_MYOSIN_COMPLEX", "GOCC_CONTRACTILE_FIBER",
    # Epithelial and exocrine structures
    "GOCC_BRUSH_BORDER", "GOCC_BRUSH_BORDER_MEMBRANE", "GOCC_ZYMOGEN_GRANULE",
    "GOCC_CORNIFIED_ENVELOPE", "GOCC_KERATIN_FILAMENT",
    # Proteins removed by the contaminant filter
    "GOCC_IMMUNOGLOBULIN_COMPLEX", "GOCC_IMMUNOGLOBULIN_COMPLEX_CIRCULATING",
    "GOCC_HEMOGLOBIN_COMPLEX",
}

In [ ]:
# Compartments of section 10.2 (Figure S6j) and their GO CC terms. "Other" holds
# the terms of no named compartment.
compartment_terms = {
    "Nucleus": [
        "GOCC_NUCLEAR_BODY", "GOCC_NUCLEAR_CHROMOSOME", "GOCC_NUCLEAR_ENVELOPE",
        "GOCC_CHROMATIN", "GOCC_NUCLEOLUS", "GOCC_NUCLEAR_SPECK",
        "GOCC_NUCLEAR_PROTEIN_CONTAINING_COMPLEX", "GOCC_SPLICEOSOMAL_COMPLEX",
    ],
    "Translation / ribosome": [
        "GOCC_RIBONUCLEOPROTEIN_COMPLEX", "GOCC_CYTOSOLIC_RIBOSOME",
        "GOCC_LARGE_RIBOSOMAL_SUBUNIT", "GOCC_SMALL_RIBOSOMAL_SUBUNIT",
        "GOCC_RIBOSOME", "GOCC_PRERIBOSOME",
    ],
    "Mitochondrion": [
        "GOCC_MITOCHONDRION", "GOCC_MITOCHONDRIAL_MATRIX",
    ],
    "ER / Golgi": [
        "GOCC_ENDOPLASMIC_RETICULUM_LUMEN",
        "GOCC_ENDOPLASMIC_RETICULUM_PROTEIN_CONTAINING_COMPLEX",
        "GOCC_NUCLEAR_OUTER_MEMBRANE_ENDOPLASMIC_RETICULUM_MEMBRANE_NETWORK",
        "GOCC_GOLGI_APPARATUS",
    ],
    "Cytoskeleton": [
        "GOCC_MICROTUBULE_CYTOSKELETON", "GOCC_POLYMERIC_CYTOSKELETAL_FIBER",
        "GOCC_ACTIN_CYTOSKELETON", "GOCC_INTERMEDIATE_FILAMENT_CYTOSKELETON",
    ],
    "Lysosome / autophagy": [
        "GOCC_LYSOSOMAL_LUMEN",
        "GOCC_AUTOPHAGOSOME", "GOCC_PHAGOPHORE_ASSEMBLY_SITE",
    ],
    "Vesicular / endosomal": [
        "GOCC_ENDOSOME", "GOCC_LATE_ENDOSOME", "GOCC_EARLY_ENDOSOME",
        "GOCC_RECYCLING_ENDOSOME", "GOCC_TRANSPORT_VESICLE",
        "GOCC_COATED_VESICLE", "GOCC_SECRETORY_VESICLE", "GOCC_SECRETORY_GRANULE",
        "GOCC_VESICLE_MEMBRANE",
    ],
    "Synaptic": [
        "GOCC_SYNAPSE", "GOCC_PRESYNAPSE", "GOCC_POSTSYNAPSE",
        "GOCC_SYNAPTIC_MEMBRANE",
        "GOCC_PRESYNAPTIC_ACTIVE_ZONE", "GOCC_GLUTAMATERGIC_SYNAPSE",
        "GOCC_NEURON_PROJECTION", "GOCC_AXON",
    ],
    "Plasma membrane / ion channel": [
        "GOCC_PLASMA_MEMBRANE_REGION", "GOCC_PLASMA_MEMBRANE_PROTEIN_COMPLEX",
        "GOCC_CELL_SURFACE", "GOCC_ANCHORING_JUNCTION",
    ],
    "Other": [
        "GOCC_CATALYTIC_COMPLEX", "GOCC_INTRACELLULAR_PROTEIN_CONTAINING_COMPLEX",
        "GOCC_TRANSFERASE_COMPLEX", "GOCC_ORGANELLE_SUBCOMPARTMENT",
    ],
}
compartment_genes, compartment_sizes = {}, {}
for label, terms in compartment_terms.items():
    genes, missing = genes_for(terms)
    if missing:
        print(f"!! {label}: terms not in the gene sets: {missing}")
    compartment_genes[label] = genes
    compartment_sizes[label] = len(genes)
print("Compartment gene-set sizes:")
for label, genes in compartment_genes.items():
    print(f"  {label:25s}: {len(genes)} genes")

In [ ]:
# ============================================================================
# One compartment per protein group, the drawing order and the colour per
# compartment; Figure S6j reads all three.
#
# A gene in several compartments is assigned to the smallest of them, because
# the smaller set is the more specific statement. A gene in none is "Other".
# ============================================================================
compartment_order = list(compartment_terms.keys())

compartment_colors = {
    "Nucleus":                       "#3b5b8c",   # deep blue
    "Translation / ribosome":        "#7a9bb5",   # blue-grey
    "Mitochondrion":                 "#c0725a",   # rust
    "ER / Golgi":                    "#8cc5b2",   # mint
    "Cytoskeleton":                  "#8cb556",   # olive green
    "Lysosome / autophagy":          "#d4a574",   # warm tan
    "Vesicular / endosomal":         "#b5a0c0",   # mauve
    "Synaptic":                      "#9678b8",   # purple
    "Plasma membrane / ion channel": "#e8a07a",   # peach
    "Other":                         "#d0d0d0",   # grey
}
_missing_colors = [c for c in compartment_order if c not in compartment_colors]
assert not _missing_colors, f"no colour for {_missing_colors}"

# Keyed by protein-group ID (the var index), because the panels index their
# matrices by it; single_gene_name is not unique across groups.
name_map = adata_log.var["single_gene_name"].to_dict()
protein_compartment = {}
for pid, gene in name_map.items():
    matches = [(label, compartment_sizes[label])
               for label, genes in compartment_genes.items() if gene in genes]
    if matches:
        matches.sort(key=lambda x: x[1])      # smallest set = most specific
        protein_compartment[pid] = matches[0][0]
    else:
        protein_compartment[pid] = "Other"

_counts = Counter(protein_compartment.values())
print(f"CHECK — {len(protein_compartment)} protein groups assigned")
for comp in compartment_order:
    print(f"  {comp:32s} {_counts.get(comp, 0):5d}")
print(f"CHECK — unassigned ('Other'): {_counts.get('Other', 0)} "
      f"({100 * _counts.get('Other', 0) / len(protein_compartment):.1f}%)")

## 10.1 Membrane-to-nuclear ratio (Figure 4e)

In [ ]:
# ============================================================================
# Membrane and vesicle proteins over soluble nuclear and translation proteins,
# per sample: log2 of the summed linear intensities of the two protein groups;
# genes annotated to both groups are dropped from both. Figure 4e (per donor,
# next cell) and Figure S6n (section 13.4) use these two term lists. The ratio
# measures the membrane and organelle crowding that Shahmoradian et al. (2019)
# describe in Lewy bodies.
# ============================================================================

MEMBRANE_TERMS = [
    # Vesicles
    "GOCC_SYNAPTIC_VESICLE_MEMBRANE",
    "GOCC_VESICLE_MEMBRANE", "GOCC_TRANSPORT_VESICLE", "GOCC_COATED_VESICLE",
    "GOCC_SECRETORY_VESICLE", "GOCC_SECRETORY_GRANULE",
    # Endosomes
    "GOCC_ENDOSOME", "GOCC_EARLY_ENDOSOME", "GOCC_LATE_ENDOSOME",
    "GOCC_RECYCLING_ENDOSOME",
    # Autophagosome and phagophore
    "GOCC_AUTOPHAGOSOME",
    "GOCC_PHAGOPHORE_ASSEMBLY_SITE",
    # Synaptic membranes
    "GOCC_PRESYNAPTIC_MEMBRANE", "GOCC_POSTSYNAPTIC_MEMBRANE",
    "GOCC_SYNAPTIC_MEMBRANE",
    # Plasma membrane
    "GOCC_PLASMA_MEMBRANE_REGION", "GOCC_PLASMA_MEMBRANE_PROTEIN_COMPLEX",
    # Golgi membrane
    "GOCC_GOLGI_MEMBRANE",
    # Mitochondria, the whole organelle (Shahmoradian et al. find them in Lewy bodies)
    "GOCC_MITOCHONDRION",
    "GOCC_INNER_MITOCHONDRIAL_MEMBRANE_PROTEIN_COMPLEX",
    # Trafficking machinery
    "GOCC_SNARE_COMPLEX",
    "GOCC_ESCRT_I_COMPLEX",
]

NUCLEAR_TRANSLATION_TERMS = [
    # Soluble nuclear terms; the nuclear envelope is left out
    "GOCC_NUCLEAR_BODY", "GOCC_NUCLEAR_SPECK", "GOCC_NUCLEOLUS",
    "GOCC_CHROMATIN", "GOCC_NUCLEAR_CHROMOSOME",
    "GOCC_NUCLEAR_PROTEIN_CONTAINING_COMPLEX",
    # Splicing / RNP
    "GOCC_SPLICEOSOMAL_COMPLEX", "GOCC_RIBONUCLEOPROTEIN_COMPLEX",
    # Translation machinery
    "GOCC_RIBOSOME", "GOCC_CYTOSOLIC_RIBOSOME",
    "GOCC_LARGE_RIBOSOMAL_SUBUNIT", "GOCC_SMALL_RIBOSOMAL_SUBUNIT",
    "GOCC_PRERIBOSOME",
]

# --- GO terms to genes (genes_for, section 10) ---
mem_genes, mem_missing = genes_for(MEMBRANE_TERMS)
nuc_genes, nuc_missing = genes_for(NUCLEAR_TRANSLATION_TERMS)
overlap  = mem_genes & nuc_genes
mem_only = mem_genes - overlap
nuc_only = nuc_genes - overlap

print(f"MEMBRANE/VESICULAR:    {len(mem_genes)} genes ({len(mem_only)} after dropping overlap)")
print(f"NUCLEAR/TRANSLATION:   {len(nuc_genes)} genes ({len(nuc_only)} after dropping overlap)")
print(f"Ambiguous (in both):   {len(overlap)} genes — dropped from BOTH sets")
if mem_missing: print(f"  MEMBRANE terms not in the gene sets: {mem_missing}")
if nuc_missing: print(f"  NUCLEAR  terms not in the gene sets: {nuc_missing}")

# --- Genes to protein groups of adata_log ---
# One protein group per symbol: the first in var order.
adata_to_use = adata_log     # NaN-preserving, log2
gene_to_protein = {}
for pid, gene in adata_to_use.var["single_gene_name"].to_dict().items():
    if pd.notna(gene):
        gene_to_protein.setdefault(str(gene), pid)

mem_proteins = [gene_to_protein[g] for g in mem_only if g in gene_to_protein]
nuc_proteins = [gene_to_protein[g] for g in nuc_only if g in gene_to_protein]
print(f"\nDetected in adata: MEMBRANE  = {len(mem_proteins):>4} proteins (of {len(mem_only)} genes)")
print(f"Detected in adata: NUCLEAR   = {len(nuc_proteins):>4} proteins (of {len(nuc_only)} genes)")

# --- Per-sample sums of the linear intensities ---
X = adata_to_use.to_df()                    # samples × proteins, log2 with NaN
mem_sum = np.power(2, X[mem_proteins]).sum(axis=1)   # pandas .sum skips NaN
nuc_sum = np.power(2, X[nuc_proteins]).sum(axis=1)

eps = 1e-12
ratio = np.log2((mem_sum + eps) / (nuc_sum + eps))

# Per-sample values on adata_log.obs and in mem_nuc_df, which the next cell reads.
adata_to_use.obs["mem_sum"]       = mem_sum.values
adata_to_use.obs["nuc_sum"]       = nuc_sum.values
adata_to_use.obs["mem_nuc_log2r"] = ratio.values

if "fraction" not in adata_to_use.obs:
    adata_to_use.obs["fraction"] = adata_to_use.obs["CellType_Aggregate"].map(fraction_map)

mem_nuc_df = (adata_to_use.obs
              [["fraction", "disease_binary", "Brain", "BrainBank",
                "mem_sum", "nuc_sum", "mem_nuc_log2r"]]
              .copy())
mem_nuc_df.index.name = "sample"

In [ ]:
# ============================================================================
# Figure 4e. Membrane-to-nuclear ratio per donor and fraction: the mean over the
# samples of that donor, so every donor counts once. Disease donors as coloured
# dots over a box, control donors as crosses. Drawn by fraction_boxplot
# (src/boxplot.py).
# ============================================================================
fraction_order = [
    "Neuron | Without Aggregate", "Neuron | DAB Positive",
    "Glia | Without Aggregate",   "Glia | DAB Positive",
    "Neuropil", "Round Aggregate", "Elongated Aggregate",
]
# Fraction names to the CellType_Aggregate keys of CELLTYPE_AGG_COLORS, so the
# colours are those of Figure 4b.
FRACTION_TO_CAT = {
    "Neuron | Without Aggregate": "Neuron Without Aggregate",
    "Neuron | DAB Positive":      "Neuron DAB Positive",
    "Glia | Without Aggregate":   "Glia Without Aggregate",
    "Glia | DAB Positive":        "Glia DAB Positive",
    "Neuropil":                   "Neuropil Without Aggregate",
    "Round Aggregate":            "Round Aggregate Aggregate",
    "Elongated Aggregate":        "Elongated Aggregate Aggregate",
}
fraction_colors = {f: CELLTYPE_AGG_COLORS[FRACTION_TO_CAT[f]] for f in fraction_order}
# Two-level x labels: the DAB state or EA shape under each box, the cell type in
# a row below, centred under its boxes. Neuropil, a single box, is named in the
# row of rotated labels.
SUB_LABEL = {
    "Neuron | Without Aggregate": "DAB−", "Neuron | DAB Positive": "DAB+",
    "Glia | Without Aggregate":   "DAB−", "Glia | DAB Positive":   "DAB+",
    "Neuropil": "Neuropil", "Round Aggregate": "round", "Elongated Aggregate": "elong.",
}
GROUP_OF = [
    ("Neuron",   ["Neuron | Without Aggregate", "Neuron | DAB Positive"]),
    ("Glia",     ["Glia | Without Aggregate", "Glia | DAB Positive"]),
    ("EA",       ["Round Aggregate", "Elongated Aggregate"]),
]

donor_df = (mem_nuc_df
            .groupby(["Brain","BrainBank","disease_binary","fraction"], observed=True)["mem_nuc_log2r"]
            .mean()
            .reset_index())
donor_df = donor_df[donor_df["fraction"].isin(fraction_order)]

print(f"Donor-level rows: {len(donor_df)}  (from {len(mem_nuc_df)} samples)\n")
print("Disease — donor mean per fraction:")
print(donor_df[donor_df["disease_binary"]=="Disease"]
      .groupby("fraction")["mem_nuc_log2r"]
      .agg(["count","mean","median","std"])
      .reindex([f for f in fraction_order
                if f in donor_df[donor_df["disease_binary"]=="Disease"]["fraction"].values])
      .round(2).to_string())
print("\nControl — donor mean per fraction:")
print(donor_df[donor_df["disease_binary"]=="Control"]
      .groupby("fraction")["mem_nuc_log2r"]
      .agg(["count","mean","median","std"])
      .reindex([f for f in fraction_order
                if f in donor_df[donor_df["disease_binary"]=="Control"]["fraction"].values])
      .round(2).to_string())
donor_df.to_csv(f"{output_dir}/{today}_membrane_vs_nuclear_perdonor.csv", index=False)

# The y label is one line; the figure legend text names the two protein groups.
fraction_boxplot(donor_df, "fraction", "mem_nuc_log2r", fraction_order, fraction_colors,
                 SUB_LABEL, GROUP_OF, "e", "membrane_vs_nuclear_boxplot_perdonor", FIG4_DIR,
                 unit="donor", dot_pt=2.4)

## 10.2 Compartment shift from host neuron to round inclusion (Figure S6j)

In [ ]:
# ============================================================================
# Figure S6j. Change in the share of the summed linear intensity per compartment
# (section 10) from the DAB+ host neuron to the round inclusion, paired within
# donor: cortical samples of the disease donors that have both fractions. Stick
# and large dot: the mean over donors; small dots: the donors.
# ============================================================================
HOST_FRACTION   = "Neuron | DAB Positive"        # the host cell of round inclusions
TARGET_FRACTION = "Round Aggregate"
REGIONS         = {"Cortex", "Middle Temporal Gyrus"}

adata_to_use = adata_log
if "fraction" not in adata_to_use.obs:
    adata_to_use.obs["fraction"] = adata_to_use.obs["CellType_Aggregate"].map(fraction_map)

cortex_mask  = adata_to_use.obs["Brain Region"].isin(REGIONS).values
disease_mask = (adata_to_use.obs["disease_binary"].astype(str) == "Disease").values
frac_mask    = adata_to_use.obs["fraction"].isin([HOST_FRACTION, TARGET_FRACTION]).values
sub = adata_to_use[cortex_mask & disease_mask & frac_mask, :].copy()
print(f"Cortex+Disease (host/target only): {sub.n_obs} samples")
print(sub.obs.groupby(["Brain","fraction"], observed=True).size().to_string())

# --- Per-sample share of the summed linear intensity in each named compartment ---
X       = sub.to_df()
lin     = np.power(2, X)
totals  = lin.sum(axis=1, skipna=True)

named_compartments = [c for c in compartment_order if c != "Other"]
mass_pct = pd.DataFrame(index=sub.obs_names, columns=named_compartments, dtype=float)
for comp in named_compartments:
    prots = [p for p, c in protein_compartment.items() if c == comp and p in X.columns]
    mass_pct[comp] = lin[prots].sum(axis=1, skipna=True) / totals * 100
mass_pct["Brain"]    = sub.obs["Brain"].astype(str).values
mass_pct["fraction"] = sub.obs["fraction"].astype(str).values

# --- Donor mean per (donor, fraction); donors with both fractions ---
donor_pct = (mass_pct.groupby(["Brain","fraction"], observed=True)[named_compartments]
             .mean().reset_index())
paired = donor_pct.groupby("Brain")["fraction"].nunique()
paired_donors = paired[paired == 2].index.tolist()
print(f"\nPaired donors (have both {HOST_FRACTION!r} + {TARGET_FRACTION!r}): "
      f"{len(paired_donors)}  → {paired_donors}")

# --- Per-donor shift (inclusion minus host) ---
shifts = []
for d in paired_donors:
    sd = donor_pct[donor_pct["Brain"] == d]
    host = sd[sd["fraction"] == HOST_FRACTION  ][named_compartments].iloc[0]
    targ = sd[sd["fraction"] == TARGET_FRACTION][named_compartments].iloc[0]
    shifts.append((targ - host).rename(d))
shift_df = pd.concat(shifts, axis=1).T
shift_df.index.name = "Brain"

# Mean shift, sorted from the largest loss (bottom) to the largest gain (top)
mean_shift = shift_df.mean(axis=0).sort_values(ascending=True)
ordered = mean_shift.index.tolist()
print(f"\nMean shift (percentage points; positive = gained in {TARGET_FRACTION}):")
print(mean_shift.round(2).to_string())
shift_df.to_csv(f"{output_dir}/{today}_compartment_shift_perdonor_"
                f"{TARGET_FRACTION.replace(' ','')}_vs_{HOST_FRACTION.split(' | ')[-1].replace(' ','')}.csv")

# --- Lollipop ---
DOT_MEAN, DOT_DONOR = native_pt(3.0), native_pt(1.6)   # dot diameters in print
# Compartment names longer than this wrap onto a second line. Only "Plasma membrane
# / ion channel" does: with 9 rows in one unit of height a two-line name reaches
# into the rows above and below, so the others stay on one line.
LABEL_WRAP = 22

# The contrast, the donor count and the direction, for the figure legend text; at
# 1 x 1 the panel has no room for them.
print(f"CHECK — mass-fraction shift ({TARGET_FRACTION} − {HOST_FRACTION}), paired within "
      f"donor, N = {len(paired_donors)} cortical Disease donors; positive = gained in the aggregate")

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.set_axisbelow(True)     # grid below the sticks and dots
y = np.arange(len(ordered))

# Stick and mean dot
for i, comp in enumerate(ordered):
    m     = mean_shift[comp]
    color = compartment_colors.get(comp, "#777777")
    ax.hlines(y=i, xmin=0, xmax=m, color=color, linewidth=LINE_LW, alpha=0.85, zorder=2)
    ax.scatter(m, i, s=DOT_MEAN ** 2, color=color, edgecolor="black", linewidth=MARKER_LW, zorder=4)

# Per-donor dots
rng = np.random.default_rng(42)
for i, comp in enumerate(ordered):
    vals = shift_df[comp].dropna().values
    jit  = rng.uniform(-0.10, 0.10, size=len(vals))
    ax.scatter(vals, np.full(len(vals), i) + jit, s=DOT_DONOR ** 2,
               facecolor="white", edgecolor=REF_COLOR, linewidth=MARKER_LW,
               alpha=0.9, zorder=3)

ax.axvline(0, color=REF_COLOR, linewidth=AXES_LW, zorder=1)
ax.set_yticks(y)
ax.set_yticklabels([textwrap.fill(c, LABEL_WRAP) for c in ordered])
label_axes(ax, xlabel="Δ mass fraction\n(% points)", ylabel="")

save_panel(fig, "j", "compartment_shift_lollipop_RoundVsNeuronDABpos", SI6_DIR)

# 11. Aggregate specificity (Figures S6h, S6i)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure S6h. Volcano plot of neuron DAB− from disease donors against control
# donors (section 7.2), with genes of the Figure 4d signature labelled.
# Colours are the style roles: UP_COLOR and DOWN_COLOR for the direction, BG_COLOR
# for the non-significant points, HIGHLIGHT_COLOR for the labelled genes.
# ═══════════════════════════════════════════════════════════════════════
# The contrast is named in the x label (panels carry no titles).
XLABEL     = "log2 FC, Neuron DAB−\nDisease / Control"
FNAME_STUB = f"ebayes_allregions_{DVC_NAME}"

FDR_CUT, LFC_CUT = 0.05, 0.5
DOT_BG, DOT_HL = native_pt(1.0), native_pt(2.4)   # dot diameters in print
ALPHA_BG     = 0.35   # significant background points
ALPHA_NS     = 0.30   # non-significant points

VOLCANO_HIGHLIGHTS = [
    # UP — synaptic-vesicle / SNARE / V-ATPase
    "SYP", "VAMP2", "SV2A", "SNAP25", "SYT1", "RAB3A", "BSN", "ATP6V0A1",
    # DOWN — nuclear / chromatin / RNA processing
    "RANBP2", "H2AX", "TOP2B", "HNRNPC", "FUS",
    # SNCA and LMNA; their FDR is stated in the figure legend text
    "SNCA", "LMNA",
]

result = res_dvc.copy()

result["status"] = "unchanged"
result.loc[(result["fdr"] < FDR_CUT) & (result["log2fc"] >  LFC_CUT), "status"] = "upregulated"
result.loc[(result["fdr"] < FDR_CUT) & (result["log2fc"] < -LFC_CUT), "status"] = "downregulated"
n_up   = int((result["status"] == "upregulated").sum())
n_down = int((result["status"] == "downregulated").sum())

result["plot_layer"] = result["status"]
is_highlight = result["single_gene_name"].isin(VOLCANO_HIGHLIGHTS)
result.loc[is_highlight, "plot_layer"] = "highlight"

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()
ax.spines[["top", "right"]].set_visible(True)     # boxed, as the other volcano plots
at.pl.volcano(
    data=result, x_column="log2fc", y_column="-log10(fdr)", ax=ax,
    x_thresholds=(-LFC_CUT, LFC_CUT),
    layers=[
        ("plot_layer", "highlight",     "highlight", {"scatter_kwargs": {"s": DOT_HL ** 2, "alpha": 1.0, "zorder": 5,
                                                                         "edgecolor": "black", "linewidth": MARKER_LW}}),
        ("plot_layer", "upregulated",   "up",        {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": ALPHA_BG, "linewidth": 0}}),
        ("plot_layer", "downregulated", "down",      {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": ALPHA_BG, "linewidth": 0}}),
        ("plot_layer", "unchanged",     "ns",        {"scatter_kwargs": {"s": DOT_BG ** 2, "alpha": ALPHA_NS, "linewidth": 0}}),
    ],
    color_dict={"highlight": HIGHLIGHT_COLOR, "up": UP_COLOR,
                "down": DOWN_COLOR, "ns": BG_COLOR},
    label_layers=None, display_id_column="single_gene_name",
    scatter_kwargs={"s": DOT_BG ** 2}, legend=None,
)
ax.set_title("")   # at.pl.volcano sets the title "Volcano Plot" and has no parameter for it

# The only lines on the axes so far are the threshold lines, which the volcano
# function draws 1 pt wide on the canvas (0.22 pt in print, below the minimum).
for _line in ax.lines:
    _line.set(linewidth=THIN_LW, color=REF_COLOR)

labeled = result[is_highlight].dropna(subset=["log2fc", "-log10(fdr)"])
# Legend under the axes in two columns; the thresholds are stated in the figure
# legend text.
_entry = dict(marker="o", linestyle="none", markersize=DOT_HL, markeredgewidth=MARKER_LW)
legend_handles = [
    Line2D([0], [0], markerfacecolor=UP_COLOR, markeredgecolor="none",
           label=f"Up ({n_up})", **_entry),
    Line2D([0], [0], markerfacecolor=BG_COLOR, markeredgecolor="none",
           label="Not significant", **_entry),
    Line2D([0], [0], markerfacecolor=DOWN_COLOR, markeredgecolor="none",
           label=f"Down ({n_down})", **_entry),
    Line2D([0], [0], markerfacecolor=HIGHLIGHT_COLOR, markeredgecolor="black",
           label="Highlighted", **_entry),
]
fig.legend(handles=legend_handles, loc="outside lower center", ncol=2,
           frameon=False, borderaxespad=0, borderpad=0.2,
           labelspacing=0.3, handlelength=1.0, handletextpad=0.4, columnspacing=1.0)
fig.get_layout_engine().set(h_pad=0.01)

label_axes(ax, xlabel=XLABEL, ylabel="−log10 FDR")

# The labels are placed after the legend and the axis labels: constrained layout
# shrinks the axes for them, and the label spacing is measured on the final axes.
# Labels in two columns along the axes edges, in the empty upper part of the
# panel, joined to their dots by leader lines; label_plot places its columns at
# fixed x anchors, which here run over the y tick labels or meet in the middle.
# The labels are split by the x of their dots into two columns of equal length
# (one column per direction would hold 10 of the 15 and run into the x axis).
# Within a column the widest label is on top and each leader leaves its label at
# the end facing the data, so a leader passes the narrower labels below it
# without crossing them.
LABEL_COL_X  = {"left": 0.02, "right": 0.98}   # left edge / right edge, axes fraction
LABEL_TOP_Y  = 0.92                            # first label of each column, axes fraction
LABEL_STEP_MM = 2.3                            # between labels, printed
fig.canvas.draw()
_step = LABEL_STEP_MM / (ax.get_window_extent().height / fig.dpi * 25.4 / DRAW_SCALE)
_by_x = labeled.sort_values("log2fc")
_n_left = (len(_by_x) + 1) // 2
for side, rows in (("left", _by_x.iloc[:_n_left]), ("right", _by_x.iloc[_n_left:])):
    texts = [ax.text(LABEL_COL_X[side], 0, row["single_gene_name"], transform=ax.transAxes,
                     ha=side, va="center", fontsize=ANNOT_FS, zorder=10,
                     bbox=dict(boxstyle="round,pad=0.1", facecolor="white", edgecolor="none"))
             for _, row in rows.iterrows()]
    fig.canvas.draw()
    order = np.argsort([-t.get_window_extent().width for t in texts], kind="stable")
    for k, i in enumerate(order):
        texts[i].set_y(LABEL_TOP_Y - k * _step)
    fig.canvas.draw()
    for t, (_, row) in zip(texts, rows.iterrows()):
        bb = t.get_window_extent().transformed(ax.transAxes.inverted())
        ax.annotate("", (row["log2fc"], row["-log10(fdr)"]),
                    xytext=(bb.x1 if side == "left" else bb.x0, t.get_position()[1]),
                    textcoords="axes fraction",
                    arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW,
                                    shrinkA=0, shrinkB=DOT_HL / 2),
                    zorder=9)
if LABEL_TOP_Y - (_n_left - 1) * _step < 0.03:
    print("!! a label column reaches the x axis; label fewer genes")

# ----------------------------------------------------------------- CHECKS
_absent = sorted(set(VOLCANO_HIGHLIGHTS) - set(result["single_gene_name"].dropna()))
print(f"CHECK — contrast {DVC_NAME} (section 7.2)")
print(f"CHECK — {len(result)} proteins tested, {n_up} up, {n_down} down "
      f"at FDR < {FDR_CUT} and |LFC| > {LFC_CUT}")
print(f"CHECK — smallest FDR {result['fdr'].min():.3f} "
      f"(max -log10(fdr) = {result['-log10(fdr)'].max():.2f})")
print(f"CHECK — highlights not in this contrast: {len(_absent)}"
      + (f" -> {_absent}" if _absent else ""))

save_panel(fig, "h", f"volcano_{FNAME_STUB}_highlights", SI6_DIR)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure S6i. Disease shift against aggregate shift, one dot per protein:
#   x = disease shift:   neuron DAB− from disease against control donors
#                        (section 7.2)
#   y = aggregate shift: round inclusion against DAB+ host neuron, the
#                        region-paired fit of Figure 4d (section 7.1)
# Both fits come from adata_log of this notebook, so `protein` (the var index)
# names the same protein in both.
#
# The axes are not equal-aspect: a square axes box leaves the axis labels no
# room in the slot. The x and y ranges are identical, and the legend states that
# the dotted line has slope 1.
#
# Colours: each highlight group takes the CAT_BASE hue of the compartment whose
# axis it moves along: Disease-tracking the aggregate-free diseased neuron (the x
# axis), Aggregate-specific the excised aggregate (the y axis). UP_COLOR and
# DOWN_COLOR mean "higher" and "lower", and the CT_BASE teal and orange mean
# neuropil and glia, so neither is used here.
# ═══════════════════════════════════════════════════════════════════════
X_CONTRAST = DVC_NAME          # x = disease shift
Y_CONTRAST = CONTRAST_NAME     # y = aggregate shift

AXIS_MARGIN = 0.3     # data units added beyond the extreme points
DOT_BG, DOT_HL = native_pt(1.0), native_pt(2.4)   # dot diameters in print

disease_tracking   = ["SYP", "SLC17A7", "SYT1", "FUS"]
aggregate_specific = ["SQSTM1", "ATP6V0A1", "LMNA", "H2AX", "HP1BP3", "TOP2B"]
group_colors = {
    "Disease-tracking":   CAT_BASE["without"],   # aggregate-free diseased neuron
    "Aggregate-specific": CAT_BASE["excised"],   # excised aggregate
}

x_df = (res_dvc[["protein", "single_gene_name", "log2fc"]]
        .rename(columns={"log2fc": "disease_shift"}))
y_df = (ttest_results_regionpaired[Y_CONTRAST][["protein", "log2fc"]]
        .rename(columns={"log2fc": "aggregate_shift"}))
merged = (x_df.merge(y_df, on="protein", how="inner")
              .dropna(subset=["disease_shift", "aggregate_shift"]))

gene_group = {g: "Disease-tracking" for g in disease_tracking}
gene_group.update({g: "Aggregate-specific" for g in aggregate_specific})
merged["hl_group"] = merged["single_gene_name"].map(gene_group)

# The panel concept, printed for the figure legend text (panels carry no titles).
print("CHECK — panel: disease amplification vs aggregate-specific recruitment")

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

bg = merged[merged["hl_group"].isna()]
ax.scatter(bg["disease_shift"], bg["aggregate_shift"], s=DOT_BG ** 2, alpha=0.25,
           color=BG_COLOR, linewidths=0, zorder=1)

lim_lo = min(merged["disease_shift"].min(), merged["aggregate_shift"].min()) - AXIS_MARGIN
lim_hi = max(merged["disease_shift"].max(), merged["aggregate_shift"].max()) + AXIS_MARGIN
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], ls=":", color=REF_COLOR, lw=AXES_LW,
        zorder=2, label="Pure amplification (slope 1)")
ax.axhline(0, color=BG_COLOR, lw=THIN_LW, zorder=0)
ax.axvline(0, color=BG_COLOR, lw=THIN_LW, zorder=0)

# Label position per gene: offset from the dot in printed points, and the side the
# text extends to. Genes not listed go 6 pt right.
LABEL_POS = {
    "SLC17A7":  (2, 8, "center"),   "SYP":    (6, -2, "left"),   "SYT1":  (6, -9, "left"),
    "ATP6V0A1": (-6, 5, "right"),   "SQSTM1": (-6, 0, "right"),  "LMNA":  (6, 0, "left"),
    "HP1BP3":   (-6, 5, "right"),   "H2AX":   (-5, -7, "right"), "TOP2B": (5, -7, "left"),
    "FUS":      (-6, 0, "right"),
}
for grp, color in group_colors.items():
    sub = merged[merged["hl_group"] == grp]
    ax.scatter(sub["disease_shift"], sub["aggregate_shift"], s=DOT_HL ** 2, color=color,
               edgecolor="black", linewidth=MARKER_LW, zorder=5, label=grp)
    for _, row in sub.iterrows():
        dx, dy, ha = LABEL_POS.get(row["single_gene_name"], (6, 0, "left"))
        ax.annotate(row["single_gene_name"], (row["disease_shift"], row["aggregate_shift"]),
                    xytext=(native_pt(dx), native_pt(dy)), textcoords="offset points",
                    ha=ha, va="center", fontsize=ANNOT_FS, zorder=10,
                    bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                              edgecolor="none", alpha=0.85),
                    arrowprops=dict(arrowstyle="-", color=REF_COLOR, lw=THIN_LW,
                                    shrinkA=0, shrinkB=DOT_HL / 2))

ax.set_xlim(lim_lo, lim_hi)
ax.set_ylim(lim_lo, lim_hi)

# Correlation across all proteins
r_p = pearsonr(merged["disease_shift"], merged["aggregate_shift"])[0]
r_s = spearmanr(merged["disease_shift"], merged["aggregate_shift"])[0]

# Legend under the axes: inside, its 29 mm first entry spans the axes and covers
# the ATP6V0A1 dot. The correlation is one line in the empty top of the axes, with
# the symbols r (Pearson) and ρ (Spearman); written out it is wider than the panel.
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center", ncol=1,
           frameon=False, borderaxespad=0, borderpad=0.2, labelspacing=0.3,
           handlelength=1.0, handletextpad=0.4)
fig.get_layout_engine().set(h_pad=0.01)
ax.text(0.02, 0.98, f"r = {r_p:.2f}, ρ = {r_s:.2f}, n = {len(merged)}",
        transform=ax.transAxes, ha="left", va="top", fontsize=ANNOT_FS)

# y label on two lines: on one line it is taller than the axes.
label_axes(ax, xlabel="Disease shift (log2FC)", ylabel="Aggregate shift\n(log2FC)")

# ----------------------------------------------------------------- CHECKS
_absent = sorted((set(disease_tracking) | set(aggregate_specific))
                 - set(merged["single_gene_name"].dropna()))
print(f"CHECK — x: {X_CONTRAST}  (section 7.2)")
print(f"CHECK — y: {Y_CONTRAST}  (section 7.1)")
print(f"CHECK — {len(merged)} proteins in both contrasts, "
      f"{int(merged['hl_group'].notna().sum())} highlighted")
print(f"CHECK — highlights absent from the merge: {len(_absent)}"
      + (f" -> {_absent}" if _absent else ""))
print(f"CHECK — Pearson r {r_p:.3f}, Spearman {r_s:.3f}")
print(f"CHECK — range {lim_lo:.2f} to {lim_hi:.2f} on both axes")

save_panel(fig, "i", "scatter_disease_vs_aggregate_shift_ebayes_DabnegvCtr_RoundvDabpos", SI6_DIR)

# 12. Variation between donors

## 12.1 GSEA per donor (Figure 4f)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 4f, GSEA [1/3]: GO CC gene sets, ranked by the region-paired
# donor-level fit of section 7.1.
#
# Ranking: signed z of the eBayes p-value. Null: gene permutation (prerank). The
# phenotype null permutes sample labels, and the unit of this design is the donor
# delta, so it is not run. run_contrast is not used: it ranks by a Welch t-test
# over wells, which treats every pool as an independent sample.
# The cell reads the two CSV files of section 7.1 from disk; it needs sections 1
# and 2 and the GO CC cell of section 10.
# ═══════════════════════════════════════════════════════════════════════
cc_contrast   = "RoundAgg_vs_NeuronDABpos_regionpaired"
AGG_CLASS     = "Round Aggregate Aggregate"
HOST_CLASS    = "Neuron DAB Positive"

for _c in (AGG_CLASS, HOST_CLASS):
    if _c not in set(adata_log.obs["CellType_Aggregate"].dropna().unique()):
        raise KeyError(f"{_c!r} is not a value of obs['CellType_Aggregate'].")

# Gene sets: the GO CC cache of section 10. The generic containers and the
# wrong-lineage terms (artifact_pathways) are tested with the other terms and
# removed before the clustering of GSEA [2/3]. GO CC has its own cache file,
# so the gene universe of the Reactome and GO biological process cache of
# src/gsea_pipeline.py stays the same.
cc_cfg = GSEAConfig(
    output_root=gsea_dir,
    artifact_pathways=frozenset(BRAIN_CC_ARTIFACTS | GENERIC_COMPARTMENTS),
    collections=("go_cellular_component",),
    genesets_cache=GOCC_CACHE,
    run_phenotype=False,          # required by run_contrast_from_rank
    min_size=20, max_size=500,
    n_review_clusters=20,
)
cc_gene_sets = load_gene_sets(cc_cfg)
print(f"GO CC: {cc_gene_sets['source'].nunique()} compartments\n")

# ------------------------------------- eBayes output of section 7.1, read from disk
# The newest file wins, so the date prefix does not have to be today's.
def _newest(pattern):
    hits = glob.glob(f"{output_dir}/{pattern}")
    if not hits:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {output_dir}. Run the cell of "
            f"section 7.1 once; it writes both files this section needs.")
    return max(hits, key=lambda f: Path(f).stat().st_mtime)

_eb_path = _newest("*_ebayes_regionpaired_*.csv")
_dd_path = _newest("*_donordelta_regionpaired_*.csv")
res_rp      = pd.read_csv(_eb_path)
donor_delta = pd.read_csv(_dd_path, index_col=0)
# The protein identifiers are integer-like, and the CSV round trip splits their
# type: read_csv infers the eBayes 'protein' column as int64, while a CSV header
# always comes back as strings. int against str matches nothing and gives an
# empty ranking, so both sides are cast to str.
res_rp["protein"]   = res_rp["protein"].astype(str)
donor_delta.columns = donor_delta.columns.astype(str)
print(f"eBayes result : {Path(_eb_path).name}  ({len(res_rp)} proteins)")
print(f"Donor deltas  : {Path(_dd_path).name}  {donor_delta.shape}")

# ------------------------------------- complete-case donor delta matrix
_pids = [p for p in res_rp["protein"] if p in donor_delta.columns]
if not _pids:
    raise ValueError(
        f"None of the {len(res_rp)} eBayes proteins appear in the donor-delta "
        f"columns. Check the types: res_rp['protein'] is {res_rp['protein'].dtype}, "
        f"donor_delta.columns is {donor_delta.columns.dtype}. Examples: "
        f"{res_rp['protein'].head(3).tolist()} against {list(donor_delta.columns[:3])}.")
D     = donor_delta[_pids]
D     = D.drop(columns=D.columns[D.isna().any(axis=0)])
donors = list(D.index)
print(f"\nDonors: {len(donors)} ({donors})")
print(f"Proteins complete across all donors: {D.shape[1]}")

# ------------------------------------- donor support and bank
_o = adata_log.obs.copy()
_o["region_h"] = _o["Brain Region"].replace({"Middle Temporal Gyrus": "Cortex"})
_dis = _o[_o["disease_binary"].astype(str) == "Disease"]
_cnt = (_dis.groupby(["Brain", "region_h", "CellType_Aggregate"], observed=True)
            .size().unstack(fill_value=0))
for _c in (AGG_CLASS, HOST_CLASS):
    if _c not in _cnt:
        _cnt[_c] = 0
_both = _cnt[(_cnt[AGG_CLASS] > 0) & (_cnt[HOST_CLASS] > 0)]
cc_support = (_both.groupby(level="Brain", observed=True).sum()[[AGG_CLASS, HOST_CLASS]]
                   .rename(columns={AGG_CLASS: "n_pools", HOST_CLASS: "n_host_wells"})
                   .reindex(donors))
_bank_of = _dis.groupby("Brain", observed=True)["BrainBank"].agg(
    lambda s: sorted(set(s.astype(str))))
for d in donors:
    if len(_bank_of[d]) != 1:
        raise ValueError(f"Donor {d!r} maps to several banks: {_bank_of[d]}")
BANK_DISPLAY = {"BH": "Banner", "Munich": "Munich"}
cc_donor_bank = {d: BANK_DISPLAY.get(_bank_of[d][0], _bank_of[d][0]) for d in donors}
_sup = cc_support.copy(); _sup["bank"] = [cc_donor_bank[d] for d in donors]
print(f"\nCHECK — donor support and bank:\n{_sup.to_string()}")
_thin = list(cc_support.index[cc_support["n_pools"] <= 2])
if _thin:
    print(f"  NOTE: {_thin} rest on 2 or fewer pools and carry the largest standard "
          f"error of any donor on every row.")

_off = D.median(axis=1)
print("\nCHECK — per-donor global offset (median delta over all proteins):")
for d in donors:
    print(f"  {d:<12s} {_off[d]:+.3f} log2   [{cc_donor_bank[d]}]")
print(f"  spread {_off.max() - _off.min():.3f} log2")

# ------------------------- one protein group per gene symbol
# The most abundant group (median intensity in adata_sf) represents the symbol.
_sym   = adata_log.var["single_gene_name"].reindex(D.columns).astype(str)
_abund = adata_sf.var["median_intensity"].reindex(D.columns)
_rep = (pd.DataFrame({"pid": D.columns, "gene": _sym.values, "abund": _abund.values})
          .query("gene not in ['nan', '', 'None']")
          .sort_values("abund", ascending=False)
          .drop_duplicates("gene"))
cc_G = D[_rep["pid"].tolist()]
cc_G.columns = _rep["gene"].tolist()          # donors x genes, read by GSEA [3/3]
print(f"\nGene universe: {cc_G.shape[1]} symbols from {D.shape[1]} protein groups")

# --------------------------------------------------------------- ranking
# Signed z of the p-value. The intercept-only eBayes fit gives every protein the
# same degrees of freedom, so the order equals that of the moderated t. prerank
# weights genes by |metric|, and signed -log10(p) has tails heavy enough that a
# few proteins would dominate every set; the z scale does not.
_r = res_rp.set_index("protein").reindex(_rep["pid"].tolist())
cc_rank = pd.Series(
    np.sign(_r["log2fc"].values) * stats.norm.isf(np.clip(_r["p_value"].values, 1e-300, None) / 2),
    index=cc_G.columns).sort_values(ascending=False)
print(f"Ranking: signed z, {len(cc_rank)} genes "
      f"[{cc_rank.iloc[0]:+.2f} … {cc_rank.iloc[-1]:+.2f}]")

cc_res = run_contrast_from_rank(
    cc_rank, cc_contrast, cc_gene_sets, cc_cfg,
    ranking_description=("signed z from the region-paired donor-level eBayes fit: "
                         "within donor x region, mean log2 round aggregate minus mean "
                         "log2 DAB+ neuron; regions averaged unweighted per donor; "
                         "limma intercept-only fit with empirical Bayes"),
    extra_params={
        "design": "region-paired, donor-level, one delta per donor",
        "n_donors": len(donors), "donors": list(donors),
        "donor_banks": cc_donor_bank,
        "pools_per_donor": cc_support["n_pools"].to_dict(),
        "n_proteins_complete": int(D.shape[1]),
        "upstream_ebayes_file": Path(_eb_path).name,
        "upstream_donordelta_file": Path(_dd_path).name,
    })

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 4f, GSEA [2/3]: writes the decision file review.csv. The terms at
# FDR < 0.05 are clustered within each direction by the overlap coefficient of
# their leading edges, with complete linkage, so every pair inside a cluster
# clears the threshold. Decisions of an earlier review are merged in.
# The decisions in review.csv (representative term, exclusion and display label
# of each HIGH-priority cluster) were made by the authors with the help of an AI
# review (Claude, Anthropic) that followed the rules in GSEA_REVIEW_PROMPT.md; the
# reviewed file is an input of this notebook.
# ═══════════════════════════════════════════════════════════════════════
build_review(cc_res, cc_cfg)

CC_REVIEW_PATH = cc_cfg.contrast_dir(cc_contrast) / "review.csv"

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 4f, GSEA [3/3]: rows and per-donor values from the reviewed decision
# file, one row per cluster, labelled by its representative term.
# assert_pass2_selection stops on a decision file whose selection does not
# replay; report_undecided prints the HIGH-priority clusters without a decision.
# The value of a row for a donor is the mean donor delta over the detected
# members of the representative term: the row is labelled with that term, and
# the full membership is used because the leading edge was selected on the
# group effect.
# ═══════════════════════════════════════════════════════════════════════
TOP_N_PLOT = 6              # per direction

ACRONYMS = ["RNA", "DNA", "mRNA", "rRNA", "snRNA", "ATP", "NADH",
            "II", "I", "ER", "COPI", "COPII", "SNARE", "MHC"]

def fix_acronyms(s):
    for a in ACRONYMS:
        s = re.sub(rf'\b{a}\b', a, s, flags=re.IGNORECASE)
    return s

rev = pd.read_csv(CC_REVIEW_PATH, keep_default_na=False)
assert_pass2_selection(rev)
for c in ("NES", "fdr", "cluster_plot_rank", "detected_size", "cluster_size"):
    rev[c] = pd.to_numeric(rev[c], errors="coerce")

unrev = rev[(rev["review_status"].isin(["NEW", "MOVED"])) &
            (rev["review_priority"] == "HIGH")]
if len(unrev):
    print(f"!! {len(unrev)} HIGH-priority term(s) NEW or MOVED — using auto-suggestion:")
    for _, r in unrev.iterrows():
        print(f"   [{r['review_status']:<5}] {r['term'][:70]}")

report_undecided(rev)

_ch  = rev["display_choice"].astype(str).str.strip().str.lower()
excluded = set(rev.loc[_ch == "exclude", "cluster_id"])
if excluded:
    print(f"\nExcluded {len(excluded)} cluster(s):")
    for n in rev[rev["cluster_id"].isin(excluded)
                 & rev["is_suggested_rep"].astype(str).str.lower().isin(["true", "1"])]["term"]:
        print(f"   {n[:74]}")
rev = rev[~rev["cluster_id"].isin(excluded)]

reps = rev[rev["display_choice"].astype(str).str.strip().str.lower() == "rep"].copy()
_dupes = reps.groupby("cluster_id").size()
for cid, n in _dupes[_dupes != 1].items():
    print(f"!! cluster {cid} has {n} rows marked 'rep' — keeping the strongest |NES|")
reps["_abs"] = reps["NES"].abs()
reps = reps.sort_values("_abs", ascending=False).drop_duplicates("cluster_id", keep="first")
_missing = set(rev["cluster_id"]) - set(reps["cluster_id"])
if _missing:
    print(f"!! {len(_missing)} cluster(s) have NO 'rep' and are omitted")

def _clean(s):
    for p in ("GOCC_", "GOBP_", "GOMF_", "REACTOME_"):
        s = s.replace(p, "")
    return fix_acronyms(s.replace("_", " ").capitalize())

def _base_label(r):
    """The reviewed display label, or the cleaned term name when the label is empty."""
    return str(r["display_label"]).strip() or _clean(r["term"])

reps["label_base"] = reps.apply(_base_label, axis=1)
reps["label"] = [f"{b} ({int(n)} collapsed)" if n > 1 else b
                 for b, n in zip(reps["label_base"], reps["cluster_size"])]

sel = pd.concat([reps[reps["NES"] > 0].nlargest(TOP_N_PLOT, "NES"),
                 reps[reps["NES"] < 0].nsmallest(TOP_N_PLOT, "NES")])
n_up, n_dn = int((sel["NES"] > 0).sum()), int((sel["NES"] < 0).sum())
print(f"\nPlotting {len(sel)} clusters ({n_up} up, {n_dn} down)")
if n_up < TOP_N_PLOT or n_dn < TOP_N_PLOT:
    print(f"   NOTE: fewer than {TOP_N_PLOT} available in a direction after exclusions — "
          f"raise n_review_clusters in cc_cfg for more headroom.")

# --------------------------------- per-donor value for each plotted row
_members  = cc_gene_sets.groupby("source")["target"].apply(lambda s: set(s.dropna()))
_universe = set(cc_G.columns)
_banks    = sorted(set(cc_donor_bank.values()))
donors    = list(cc_G.index)

rows, long_rows = [], []
for _, srow in sel.iterrows():
    term = srow["term"]
    mem  = sorted(_members[term] & _universe)
    if len(mem) < cc_cfg.min_size:
        raise ValueError(f"{term} has {len(mem)} detected members, below cc_cfg.min_size "
                         f"({cc_cfg.min_size}). The size filter and the gene universe "
                         f"have diverged.")
    others = [g for g in cc_G.columns if g not in _members[term]]
    vals   = cc_G[mem].mean(axis=1)
    comp   = vals - cc_G[others].mean(axis=1)
    mean_v = float(vals.mean())

    # Gap between the banks: the lowest value of the higher bank minus the highest
    # value of the lower bank; positive when the two banks do not overlap.
    if len(_banks) == 2:
        gA = vals[[d for d in donors if cc_donor_bank[d] == _banks[0]]]
        gB = vals[[d for d in donors if cc_donor_bank[d] == _banks[1]]]
        hi_, lo_ = (gA, gB) if gA.mean() > gB.mean() else (gB, gA)
        bank_gap = float(hi_.min() - lo_.max())
    else:
        bank_gap = np.nan

    rows.append({"term": term, "label": srow["label"], "cluster_id": srow["cluster_id"],
                 "cluster_size": int(srow["cluster_size"]),
                 "n_detected": len(mem), "n_in_set": len(_members[term]),
                 "NES": srow["NES"], "gsea_fdr": srow["fdr"],
                 "mean_effect": mean_v, "min_effect": float(vals.min()),
                 "max_effect": float(vals.max()),
                 "range": float(vals.max() - vals.min()),
                 "competitive_mean": float(comp.mean()), "bank_gap": bank_gap,
                 "banks_separated": bool(bank_gap > 0) if np.isfinite(bank_gap) else False,
                 "n_donors_same_sign_as_mean": int((np.sign(vals) == np.sign(mean_v)).sum())})
    for d in donors:
        long_rows.append({"term": term, "label": srow["label"], "donor": d,
                          "bank": cc_donor_bank[d],
                          "n_pools": int(cc_support.loc[d, "n_pools"]),
                          "set_effect": float(vals[d]), "competitive": float(comp[d])})

panelI_terms = (pd.DataFrame(rows).sort_values("mean_effect", ascending=False)
                .reset_index(drop=True))
panelI_long = pd.DataFrame(long_rows)
panelI_long["donor"] = pd.Categorical(panelI_long["donor"], categories=donors, ordered=True)

print(f"\n{'compartment':<44s} {'det/set':>9s} {'NES':>6s} {'FDR':>8s} "
      f"{'mean':>7s} {'range':>7s} {'comp':>7s} {'gap':>7s} {'sign':>5s}")
for _, r in panelI_terms.iterrows():
    print(f"{r['label'][:42]:<44s} {r['n_detected']:>4d}/{r['n_in_set']:<4d} "
          f"{r['NES']:>+6.2f} {r['gsea_fdr']:>8.1e} {r['mean_effect']:>+7.2f} "
          f"{r['range']:>7.2f} {r['competitive_mean']:>+7.2f} {r['bank_gap']:>+7.2f} "
          f"{r['n_donors_same_sign_as_mean']:>3d}/{len(donors)}")

_nsep = int(panelI_terms["banks_separated"].sum())
print(f"\nCHECK — rows where the two banks do not overlap at all: {_nsep} / "
      f"{len(panelI_terms)}")
for _, r in panelI_terms[panelI_terms["banks_separated"]].iterrows():
    print(f"    {r['label'][:48]:<50s} gap {r['bank_gap']:+.2f} log2, "
          f"row range {r['range']:.2f}")
# Over all kept representatives, plotted or not: a kept row that contains a
# plotted row makes the same statement twice, and a larger TOP_N_PLOT would put
# both on the figure.
_kept_pairs = [(a, b) for a, b in itertools.combinations(
    reps.loc[reps["direction"].notna(), "term"], 2)]
_worst = max((len(_members[a] & _members[b] & _universe)
              / max(min(len(_members[a] & _universe), len(_members[b] & _universe)), 1),
              a, b) for a, b in _kept_pairs)
print(f"CHECK — largest detected-membership overlap among all "
      f"{len(reps)} kept rows: {_worst[0]:.2f}")
if _worst[0] > cc_cfg.overlap_thresh:
    print(f"  Two rows share {_worst[0]:.0%} of their detected members:\n"
          f"    {_worst[1]}\n    {_worst[2]}\n"
          f"  The clustering is on the LEADING EDGE, so full-membership overlap above "
          f"the threshold is possible and is a judgement call for the reviewer, not "
          f"a bug.")

# The Results state how many significant terms change in the same direction in
# every donor. Over every term at FDR < fdr_cut that is not on the artifact list,
# plotted or not, two per-donor values: the set value of the rows above (mean
# donor delta over the detected members, the dots of Figure 4f), and the
# competitive value (set value minus the mean of the other detected genes). The
# competitive value is the comparison the gene-permutation null of the GSEA
# makes; it removes each donor's global offset printed in GSEA [1/3].
_sig = cc_res[(cc_res["fdr"] < cc_cfg.fdr_cut) & ~cc_res["term"].isin(cc_cfg.artifact_pathways)]
_n_same = {}
for t, nes in zip(_sig["term"], _sig["NES"]):
    _set   = cc_G[sorted(_members[t] & _universe)].mean(axis=1)
    _other = cc_G[[g for g in cc_G.columns if g not in _members[t]]].mean(axis=1)
    _n_same[t] = (int((np.sign(_set) == np.sign(nes)).sum()),
                  int((np.sign(_set - _other) == np.sign(nes)).sum()))
_nd = len(donors)
print(f"\nCHECK — significant terms (FDR < {cc_cfg.fdr_cut}, artifact list removed) with "
      f"the sign of the NES in all {_nd} donors, of {len(_n_same)}:\n"
      f"    set value          {sum(a == _nd for a, _ in _n_same.values())}\n"
      f"    competitive value  {sum(c == _nd for _, c in _n_same.values())}")
print("  Terms whose competitive value has the NES sign in fewer donors:")
for t, (_, c) in _n_same.items():
    if c < _nd:
        print(f"    {t[:60]:<62s} {c}/{_nd}")

_stub = f"{today}_panelI_donorsets_gocc_reviewed".replace(" ", "")
panelI_terms.to_csv(f"{output_dir}/{_stub}_terms.csv", index=False)
panelI_long.to_csv(f"{output_dir}/{_stub}_perdonor.csv", index=False)
print(f"\nSaved {_stub}_terms.csv and {_stub}_perdonor.csv")

In [ ]:
# ============================================================================
# Figure 4f, plot: one row per compartment, one dot per donor. Marker shape
# encodes the brain bank, because the depleted rows separate by bank. Reads
# panelI_terms and panelI_long from GSEA [3/3].
#
# Text sizes and line widths are the style roles; sizes below are printed sizes.
# The grey minimum-to-maximum range is a bar (a filled rectangle): a 2.5 pt line
# is above Nature's 1 pt line limit, and a filled bar is not a line.
# ============================================================================
RANGE_H     = 0.18         # height of the grey min-to-max bar, in rows
DOT         = native_pt(3.5)   # donor dot diameter, 3.5 pt in print
MEAN_HALFH  = 0.30         # half height of the mean tick, in rows
XLIM_PAD    = 1.12
WRAP        = 40           # characters per line in a row label

DONOR_PALETTE = ["#4477aa", "#ee6677", "#228833", "#ccbb44",
                 "#66ccee", "#aa3377", "#999933", "#bbbbbb"]
BANK_MARKER   = {"Munich": "o", "Banner": "s"}
# Donor names as in the "Brain" panel (Figure S6e, LEVEL_SHORT in section 6.2):
# the Munich brains are Muc 1-3; the Banner brains are BH 1-4, numbered by their
# IDs over all four, so the two PD brains here are BH 1 and BH 3.
DONOR_SHORT   = {"LBD 1": "Muc 1", "LBD 2": "Muc 2", "LBD 3": "Muc 3",
                 "MTG 20 03": "BH 1", "MTG 22 61": "BH 3"}


_donors = list(panelI_long["donor"].cat.categories)
if len(_donors) > len(DONOR_PALETTE):
    raise ValueError(f"{len(_donors)} donors but {len(DONOR_PALETTE)} palette entries.")
DONOR_COLORS = dict(zip(_donors, DONOR_PALETTE))

_meta  = panelI_long.drop_duplicates("donor").set_index("donor")[["bank", "n_pools"]]
_banks = sorted(_meta["bank"].unique())
_unknown = [b for b in _banks if b not in BANK_MARKER]
if _unknown:
    raise KeyError(f"No marker assigned for bank(s) {_unknown}. Extend BANK_MARKER.")

_terms = panelI_terms["term"].tolist()
n_rows = len(_terms)
_vals  = panelI_long.set_index(["term", "donor"])["set_effect"]
lim    = float(panelI_long["set_effect"].abs().max()) * XLIM_PAD

# The row label carries two counts: the proteins measured in the compartment, and
# the number of GO CC terms the cluster collapsed. Wrapped as one string: a line
# break before either count makes three lines, which do not fit the row.
_labels = []
for lab, nset, ndet in zip(panelI_terms["label"], panelI_terms["cluster_size"],
                           panelI_terms["n_detected"]):
    base = re.sub(r"\s*\(\d+ collapsed\)$", "", lab)
    suffix = (f"({int(ndet)} proteins, {int(nset)} terms collapsed)" if nset > 1
              else f"({int(ndet)} proteins)")
    _labels.append(textwrap.fill(f"{base} {suffix}", WRAP))

fig, axm = create_figure(1, 1, figsize=figsize(2, 2))
ax = axm.next()

# 1. grey range bar, minimum to maximum donor value
for i, term in enumerate(_terms):
    v = _vals.loc[term].reindex(_donors)
    ax.barh(i, v.max() - v.min(), left=v.min(), height=RANGE_H, color=BG_COLOR,
            linewidth=0, zorder=2)

# 2. zero line, above the range bars and below the dots
ax.axvline(0, color=REF_COLOR, lw=AXES_LW, zorder=3)

# 3. one dot per donor, shape by bank; 4. mean tick
for i, term in enumerate(_terms):
    v = _vals.loc[term].reindex(_donors)
    for d in _donors:
        ax.scatter([v[d]], [i], s=DOT ** 2, marker=BANK_MARKER[_meta.loc[d, "bank"]],
                   color=DONOR_COLORS[d], edgecolors="white",
                   linewidths=MARKER_LW, zorder=4)
    m = float(v.mean())
    ax.plot([m, m], [i - MEAN_HALFH, i + MEAN_HALFH], color="black",
            lw=LINE_LW, solid_capstyle="butt", zorder=5)

ax.set_yticks(range(n_rows))
ax.set_yticklabels(_labels)
for tick, eff in zip(ax.get_yticklabels(), panelI_terms["mean_effect"]):
    tick.set_color(UP_COLOR if eff > 0 else DOWN_COLOR)

ax.set_ylim(n_rows - 0.5, -0.5)
ax.set_xlim(-lim, lim)
ax.grid(False)
# Two lines: on one line the x label is wider than the panel.
label_axes(ax, xlabel="Compartment effect\n(log2 round aggregate / DAB+ neuron)")

# Legend inside the lower right: every depleted row has its dots left of zero, so
# that corner is empty. The legend title names the pool count ("Donor (pools)"),
# so each entry is short enough for the empty corner. An axes legend inside the
# axes is not counted twice by constrained_layout, as one anchored outside is.
_handles = [Line2D([0], [0], marker=BANK_MARKER[_meta.loc[d, "bank"]], linestyle="none",
                   markersize=DOT, markerfacecolor=DONOR_COLORS[d],
                   markeredgecolor="white", markeredgewidth=MARKER_LW,
                   label=f"{DONOR_SHORT.get(d, d)} ({int(_meta.loc[d, 'n_pools'])})")
            for d in _donors]
_handles.append(Line2D([0], [0], marker="|", linestyle="none", color="black",
                       markersize=DOT * 1.3, markeredgewidth=LINE_LW, label="Mean"))
_leg = ax.legend(handles=_handles, loc="lower right", title="Donor (pools)",
                 frameon=True, framealpha=0.9,
                 edgecolor="none", handletextpad=0.6, borderpad=0.7,
                 labelspacing=0.4)
_leg.set_zorder(6)

# ----------------------------------------------------------------- CHECKS
_sizes = {"y ticks": {t.get_fontsize() for t in ax.get_yticklabels()},
          "x ticks": {t.get_fontsize() for t in ax.get_xticklabels()},
          "x label": {ax.xaxis.label.get_fontsize()},
          "legend":  {t.get_fontsize() for t in _leg.get_texts()}}
print(f"CHECK — text sizes against the style (TICK_FS={TICK_FS}, "
      f"LABEL_FS={LABEL_FS}, LEGEND_FS={LEGEND_FS}):")
for what, s in _sizes.items():
    print(f"    {what:<8} {sorted(s)}  "
          f"{'ok' if s <= {TICK_FS, LABEL_FS, LEGEND_FS} else '<-- NOT a style size'}")

_max_lines = max(l.count("\n") + 1 for l in _labels)
fig.canvas.draw()
_row_pt    = ax.get_window_extent().height / fig.dpi * 72 / n_rows
print(f"CHECK — labels at {TICK_FS} pt: tallest is {_max_lines} lines, needs "
      f"{_max_lines * TICK_FS * 1.2:.0f} pt of a {_row_pt:.0f} pt row -> "
      f"{'fits' if _max_lines * TICK_FS * 1.2 < _row_pt * 0.85 else 'TOO TIGHT'}")

fig.canvas.draw()
_bb = _leg.get_window_extent().transformed(ax.transAxes.inverted())
_hits = sum(1 for i, term in enumerate(_terms)
            for val in _vals.loc[term].reindex(_donors).values
            if _bb.x0 <= (val + lim) / (2 * lim) <= _bb.x1
            and _bb.y0 <= 1 - (i + 0.5) / n_rows <= _bb.y1)
print(f"CHECK — plotted points falling inside the legend box: {_hits}")

_nsep = int(panelI_terms["banks_separated"].sum())
print(f"CHECK — {_nsep} of {n_rows} rows have no overlap between the two banks.")
print(f"CHECK — x limits {-lim:.2f} to {lim:.2f}; largest plotted value "
      f"{panelI_long['set_effect'].abs().max():.2f}")

save_panel(fig, "f", "panelI_donorsets_gocc_reviewed", FIG4_DIR)

## 12.2 Variance per patient (Figure S6k)

In [ ]:
# ============================================================================
# Figure S6k, analysis: how the variance of the aggregate-over-host ratio divides
# between the shared signature, the patients and the pools.
#
# One region, one bank, disease donors, two dissected classes. Each round-
# inclusion pool is referenced to its own donor's DAB+ neuron mean in the same
# region, so a pool-level value is a log2 aggregate-over-host ratio.
#
# Decomposition, in sum-of-squares units, so the two bars nest exactly:
#   bar 1   total across pools = common signature (protein main effect) + remainder
#   bar 2   remainder          = between-patient + sampling error in the donor
#                                means + within-patient
#
# The sampling-error segment is the part removed from the raw between-patient
# sum of squares. Two errors enter a donor's mean that are not patient variation:
# the pool-to-pool error, averaged over that donor's n_d pools, and the error of
# that donor's host reference, averaged over its m_d host wells. The second is
# constant within a donor and so absent from the within-donor term; it is
# subtracted explicitly.
#
# The estimator is method-of-moments, per protein, and negative per-protein
# between-patient estimates are summed as they are; truncating them at zero first
# would bias the share upward. No resampling: with this many pools a bootstrap
# distribution of a variance share cannot be interpreted.
# ============================================================================
FRACTION_AGG, FRACTION_HOST = "Round Aggregate Aggregate", "Neuron DAB Positive"
DONOR_COL, REGION_COL, BANK_COL, BATCH_COL = "Brain", "Brain Region", "BrainBank", "Plate"
MIN_POOLS_PER_DONOR = 2      # a within-patient term needs within-donor replication
MIN_HOSTS_PER_DONOR = 1      # the host error is pooled over donors with 2 or more
MIN_DONORS          = 3

for _needed in ("adata_log",):
    if _needed not in globals():
        raise NameError(f"{_needed!r} is not defined. Run the preprocessing "
                        f"cell (section 2.3).")

_ad = adata_log.copy()
_ad.obs["fraction"] = _ad.obs["CellType_Aggregate"].astype(str)
for _c in (FRACTION_AGG, FRACTION_HOST):
    if _c not in set(_ad.obs["fraction"].unique()):
        raise KeyError(f"{_c!r} is not a value of obs['CellType_Aggregate']. "
                       f"Observed: {sorted(_ad.obs['fraction'].unique())}")

_gene_map = _ad.var["single_gene_name"].to_dict()
if "num_proteins" not in _ad.obs.columns:
    raise KeyError("obs['num_proteins'] is missing; it is set in step 1 of the preprocessing cell.")

# ------------------------------------------------------------- 1. stratum
# The bank x region stratum with the most usable donors; ties go to a single plate,
# then to more pools.
obs  = _ad.obs
elig = obs[(obs["disease_binary"].astype(str) == "Disease")
           & (obs["fraction"].isin([FRACTION_AGG, FRACTION_HOST]))]

_rows = []
for (bank, region), g in elig.groupby([BANK_COL, REGION_COL], observed=True):
    per_donor = g.groupby([DONOR_COL, "fraction"], observed=True).size().unstack(fill_value=0)
    for col in (FRACTION_AGG, FRACTION_HOST):
        if col not in per_donor:
            per_donor[col] = 0
    ok = per_donor[(per_donor[FRACTION_AGG] >= MIN_POOLS_PER_DONOR)
                   & (per_donor[FRACTION_HOST] >= MIN_HOSTS_PER_DONOR)]
    _rows.append({"bank": bank, "region": region, "n_donors_usable": len(ok),
                  "n_pools": int(ok[FRACTION_AGG].sum()), "n_hosts": int(ok[FRACTION_HOST].sum()),
                  "n_plates": g[BATCH_COL].nunique(), "single_plate": g[BATCH_COL].nunique() == 1,
                  "donors": ", ".join(map(str, ok.index))})
strata = pd.DataFrame(_rows).sort_values(
    ["n_donors_usable", "single_plate", "n_pools"], ascending=False)

print(f"Candidate strata (donors needing >={MIN_POOLS_PER_DONOR} round pools "
      f"and >={MIN_HOSTS_PER_DONOR} DAB+ neuron wells):")
print(strata.to_string(index=False))

BANK, REGION = strata.iloc[0]["bank"], strata.iloc[0]["region"]
if strata.set_index(["bank", "region"]).loc[(BANK, REGION), "n_donors_usable"] < MIN_DONORS:
    raise ValueError(f"({BANK}, {REGION}) has fewer than {MIN_DONORS} usable donors.")
print(f"\nSelected stratum: bank={BANK!r}, region={REGION!r}")

sel = elig[(elig[BANK_COL].astype(str) == str(BANK))
           & (elig[REGION_COL].astype(str) == str(REGION))]
_counts = sel.groupby([DONOR_COL, "fraction"], observed=True).size().unstack(fill_value=0)
for col in (FRACTION_AGG, FRACTION_HOST):
    if col not in _counts:
        _counts[col] = 0
donors = [d for d in _counts.index
          if _counts.loc[d, FRACTION_AGG] >= MIN_POOLS_PER_DONOR
          and _counts.loc[d, FRACTION_HOST] >= MIN_HOSTS_PER_DONOR]
_excl = [d for d in _counts.index if d not in donors]
sel = sel[sel[DONOR_COL].isin(donors)]

# ---------------------------------------------------- 2. CHECK: composition
print(f"\nCHECK — pools and host wells per donor:")
print(_counts.loc[donors, [FRACTION_AGG, FRACTION_HOST]]
      .rename(columns={FRACTION_AGG: "round_pools", FRACTION_HOST: "host_wells"}).to_string())
if _excl:
    print(f"Donors dropped (too few on one side): {_excl}")
    print(_counts.loc[_excl, [FRACTION_AGG, FRACTION_HOST]].to_string())

print(f"CHECK — one region:  {sorted(sel[REGION_COL].astype(str).unique())}")
print(f"CHECK — one bank:    {sorted(sel[BANK_COL].astype(str).unique())}")
_plates = sorted(sel[BATCH_COL].astype(str).unique())
print(f"CHECK — plate(s):    {_plates}")
if len(_plates) == 1:
    print("  Single plate — donor effects are not confounded with batch.")
else:
    _ct = pd.crosstab(sel[DONOR_COL].astype(str), sel[BATCH_COL].astype(str))
    if ((_ct > 0).sum(axis=1) == 1).all():
        print("  WARNING: each donor sits on exactly one plate. Donor and plate are "
              "CONFOUNDED and the between-patient component includes batch.")
    else:
        print("  Donors are CROSSED with plate, not confounded. Plate imbalance adds "
              "noise to the within-patient term but does not create a donor effect.")
    print(_ct.to_string())
print("NOTE — staining round is not recorded in the metadata, so its constancy "
      "cannot be verified from the data. Plate is the only batch field available.")

# --------------------------------------------- 3. complete-case protein set
pool_idx = sel.index[sel["fraction"] == FRACTION_AGG]
host_idx = sel.index[sel["fraction"] == FRACTION_HOST]
X_all = _ad[sel.index, :].to_df()
complete = X_all.columns[X_all.notna().all(axis=0)]
print(f"\nComplete-case proteins across the {len(pool_idx)} pools + "
      f"{len(host_idx)} host wells: {len(complete)} / {X_all.shape[1]}")
if len(complete) < 200:
    print("  WARNING: very few complete cases. Consider adata_strict_log as the input.")
X = X_all[complete]

# ------------------------------------- 4. pool-level aggregate-over-host ratio
_host_by_donor = {d: sel.index[(sel[DONOR_COL] == d) & (sel["fraction"] == FRACTION_HOST)]
                  for d in donors}
host_mean = pd.DataFrame({d: X.loc[_host_by_donor[d]].mean(axis=0) for d in donors}).T
donor_of_pool = sel.loc[pool_idx, DONOR_COL].astype(str)
Y = X.loc[pool_idx].sub(host_mean.reindex(donor_of_pool.values).values)

Yv   = Y.values
N, P = Yv.shape
k    = len(donors)
n_d  = np.array([int((donor_of_pool.values == d).sum()) for d in donors], dtype=float)
m_d  = np.array([len(_host_by_donor[d]) for d in donors], dtype=float)
if N - k < 2:
    raise ValueError(f"{N} pools in {k} donors leaves {N - k} within-donor degrees of "
                     f"freedom. The within-patient term is not estimable.")

# ------------------------------------------------ 5. bar 1: signature vs rest
grand  = Yv.mean()
prot   = Yv.mean(axis=0)                    # the signature common to all pools
SS_tot = ((Yv - grand) ** 2).sum()
SS_sig = N * ((prot - grand) ** 2).sum()
SS_rem = SS_tot - SS_sig

# --------------------------- 6. bar 2: between, sampling error, within
R = Yv - prot                               # remainder, per protein mean removed
Rbar = np.stack([R[donor_of_pool.values == d].mean(axis=0) for d in donors])   # k x P
SS_B_p = (n_d[:, None] * Rbar ** 2).sum(axis=0)
SS_W_p = np.stack([((R[donor_of_pool.values == d] - Rbar[j]) ** 2).sum(axis=0)
                   for j, d in enumerate(donors)]).sum(axis=0)
SS_B, SS_W = SS_B_p.sum(), SS_W_p.sum()
assert abs(SS_B + SS_W - SS_rem) < 1e-6 * max(SS_rem, 1.0), "bar 2 does not close on bar 1"

MS_W_p = SS_W_p / (N - k)                   # pool-to-pool sampling variance
_hss, _hdf = np.zeros(P), 0                 # host-well sampling variance, pooled
for d in donors:
    h = X.loc[_host_by_donor[d]].values
    if len(h) >= 2:
        _hss += ((h - h.mean(axis=0)) ** 2).sum(axis=0)
        _hdf += len(h) - 1
if _hdf > 0:
    host_var_p = _hss / _hdf
else:
    host_var_p = np.zeros(P)
    print("\nWARNING: no donor has 2 or more host wells, so the host reference error "
          "is not estimable and is treated as zero. The between-patient component "
          "below is therefore an UPPER BOUND.")

# c_d weights SS_B under this design: E[SS_B] = sigma2_patient * sum(c_d)
#                                              + sum(c_d * (MS_W/n_d + host_var/m_d))
c_d = n_d * (1.0 - n_d / N)
adj_p = (c_d[:, None] * (MS_W_p[None, :] / n_d[:, None]
                         + host_var_p[None, :] / m_d[:, None])).sum(axis=0)
sigma2_patient_p = (SS_B_p - adj_p) / c_d.sum()      # kept signed, never truncated
SS_patient  = float((SS_B_p - adj_p).sum())
SS_sampling = float(adj_p.sum())

vp_shares = {
    "SS_total": float(SS_tot), "SS_signature": float(SS_sig), "SS_remainder": float(SS_rem),
    "SS_between_raw": float(SS_B), "SS_between_corrected": SS_patient,
    "SS_sampling_removed": SS_sampling, "SS_within": float(SS_W),
    "pct_signature": 100 * SS_sig / SS_tot, "pct_remainder": 100 * SS_rem / SS_tot,
    "pct_between_of_remainder": 100 * SS_patient / SS_rem,
    "pct_sampling_of_remainder": 100 * SS_sampling / SS_rem,
    "pct_within_of_remainder": 100 * SS_W / SS_rem,
    "pct_between_of_total": 100 * SS_patient / SS_tot,
    "n_pools": N, "n_donors": k, "n_proteins": P,
    # The between-patient component has k - 1 degrees of freedom; with three
    # donors that is 2, so the share is a point estimate on 2 df. The df is stated
    # in place of an interval, which a bootstrap at this n cannot supply.
    "df_between": k - 1,
    "df_within": N - k, "df_host": _hdf,
    "n_proteins_negative_between": int((sigma2_patient_p < 0).sum()),
}

# ------------------------------------------------------------- 7. CHECKS
print(f"\nBar 1 — all variance across the {N} pools:")
print(f"  common signature (protein main effect) : {vp_shares['pct_signature']:5.1f} %")
print(f"  remainder                              : {vp_shares['pct_remainder']:5.1f} %")
print(f"\nBar 2 — that remainder:")
print(f"  between-patient, corrected             : {vp_shares['pct_between_of_remainder']:5.1f} %")
print(f"  sampling error removed from between    : {vp_shares['pct_sampling_of_remainder']:5.1f} %")
print(f"    (raw between before correction       : {100 * SS_B / SS_rem:5.1f} %)")
print(f"  within-patient (pool to pool, same donor): {vp_shares['pct_within_of_remainder']:5.1f} %")
print(f"\nBetween-patient as a share of TOTAL variance: "
      f"{vp_shares['pct_between_of_total']:5.1f} %")
print(f"\nCHECK — proteins with a negative between-patient estimate: "
      f"{vp_shares['n_proteins_negative_between']} / {P} "
      f"({100 * vp_shares['n_proteins_negative_between'] / P:.1f} %). These are summed "
      f"as they are; the expected fraction is large when the true component is near zero.")
print(f"CHECK — degrees of freedom: between-patient {k - 1}, "
      f"within-donor {N - k}, host-well {_hdf}")
if k - 1 <= 2:
    print(f"  The between-patient share rests on {k - 1} df. Treat it as an order of "
          f"magnitude, not a measurement. A bootstrap would not fix that; it would "
          f"report an interval that is itself estimated on {k - 1} df.")
if vp_shares["pct_between_of_remainder"] < 0:
    print("  NOTE: the corrected between-patient sum of squares is NEGATIVE. The "
          "sampling error in the donor means already accounts for the observed "
          "spread, so there is no evidence of patient variation at this depth.")

# Read by the Figure S6l analysis (section 12.3), which uses the same pools, donors
# and protein set.
vp_Y, vp_donors, vp_donor_of_pool = Y, donors, donor_of_pool
vp_bank, vp_region = BANK, REGION
vp_depth = _ad.obs.loc[pool_idx, "num_proteins"].astype(float)
assert list(vp_depth.index) == list(vp_Y.index), "depth covariate is not in value-matrix order"

_stub = f"{today}_panelA_variance_{BANK}_{REGION}".replace(" ", "")
pd.Series(vp_shares).to_csv(f"{output_dir}/{_stub}.csv", header=False)
print(f"\nSaved {_stub}.csv to {output_dir}/")

In [ ]:
# ============================================================================
# Figure S6k, plot: two nested stacked bars, from vp_shares of the analysis cell
# above.
#
# Colour: the dark violet marks the between-patient variance and nothing else.
# The remainder as a whole is the light violet in bar 1, the colour of the wedge,
# so bar 1 shows the part that bar 2 divides.
#
# The percentages stand inside their segments; the names stand outside, joined by
# a leader line. A name and a percentage outside take three lines, and three-line
# labels on the two lower segments overlap each other and run below the axis.
# ============================================================================
X1, X2     = 0.0, 1.5      # bar centres; the wedge fills the 3.1 mm between the bars
BAR_W      = 1.12          # 9.2 mm of the 33.7 mm axes: "Common" (8.7 mm) fits inside
WEDGE_A    = 0.45          # base violet at 45 % on white = _LIGHT, the colour of the remainder
SEG_EDGE   = "black"       # segment edges at the style's MARKER_LW, leaders THIN_LW
LEAD_X     = X2 + BAR_W / 2 + 0.22     # where the names of bar 2 start
Y_PAD      = 7.8           # keeps a two-line name clear of the axis ends

_v      = vp_shares
_df_b   = int(_v.get("df_between", int(_v["n_donors"]) - 1))
_VIOLET = CT_BASE["excised"]
_LIGHT  = shade(_VIOLET, 0.55)     # the remainder as a whole, and the wedge

# The remainder segment carries no name: the wedge shows that bar 2 divides it.
bar1 = [("Common", _v["pct_signature"], BG_COLOR, "black", None),
        ("",       _v["pct_remainder"], _LIGHT,   "black", None)]
# (name outside, pct, colour, hatch, colour of the pct drawn inside)
bar2 = [("Between\npatient",
         max(_v["pct_between_of_remainder"], 0.0), _VIOLET, None, "white"),
        ("Donor\nmeans",
         _v["pct_sampling_of_remainder"], "#efefef", "//", "black"),
        ("Within\npatient",
         _v["pct_within_of_remainder"], shade(_VIOLET, 0.35), None, "black")]

if _v["pct_between_of_remainder"] < 0:
    print(f"NOTE: the corrected between-patient share is "
          f"{_v['pct_between_of_remainder']:.1f} % and is drawn as 0, so the three "
          f"segments no longer sum to 100 % of the second bar.")

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

# --- bar 1: name and percentage inside; the remainder segment has its percentage only
bottom = 0.0
for name, pct, color, txt_color, hatch in bar1:
    ax.bar(X1, pct, width=BAR_W, bottom=bottom, color=color, hatch=hatch,
           edgecolor=SEG_EDGE, linewidth=MARKER_LW, zorder=3)
    ax.text(X1, bottom + pct / 2, f"{name}\n{pct:.0f}%" if name else f"{pct:.0f}%",
            ha="center", va="center",
            fontsize=ANNOT_FS, color=txt_color, zorder=4, linespacing=1.25)
    bottom += pct

# --- bar 2: percentage inside, name outside on an elbow leader
bottom = 0.0
_mid = X2 + BAR_W / 2 + 0.09
_names = []
for name, pct, color, hatch, pct_color in bar2:
    ax.bar(X2, pct, width=BAR_W, bottom=bottom, color=color, hatch=hatch,
           edgecolor=SEG_EDGE, linewidth=MARKER_LW, zorder=3)
    y = bottom + pct / 2
    ax.text(X2, y, f"{pct:.0f}%", ha="center", va="center", fontsize=ANNOT_FS,
            color=pct_color, zorder=5,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none",
                      boxstyle="round,pad=0.2") if hatch else None)
    # A name is moved away from the axis ends when its segment is too low or too
    # high for a two-line label; the elbow joins it to its segment.
    y_lab = min(max(y, Y_PAD), 100 - Y_PAD)
    ax.plot([X2 + BAR_W / 2 + 0.02, _mid, _mid, LEAD_X - 0.05],
            [y, y, y_lab, y_lab], color=REF_COLOR, lw=THIN_LW, zorder=4)
    _names.append(ax.text(LEAD_X, y_lab, name, ha="left", va="center", fontsize=ANNOT_FS,
                          color="black", zorder=4, linespacing=1.25))
    bottom += pct

# The wedge joins the remainder segment of bar 1 to the whole of bar 2, in the
# light violet, so the figure shows the nesting.
ax.add_patch(Polygon([[X1 + BAR_W / 2, _v["pct_signature"]],
                      [X1 + BAR_W / 2, 100.0],
                      [X2 - BAR_W / 2, 100.0],
                      [X2 - BAR_W / 2, 0.0]],
                     closed=True, facecolor=_VIOLET, alpha=WEDGE_A,
                     edgecolor="none", zorder=1))

ax.set_xlim(-0.7, 3.4)
ax.set_ylim(0, 100)
ax.set_xticks([X1, X2])
ax.set_xticklabels(["All variance", "Remainder"])
ax.grid(False)
label_axes(ax, ylabel="% of variance")

# ----------------------------------------------------------------- CHECKS
print(f"CHECK — bar 2 segments sum to {sum(p for _, p, _, _, _ in bar2):.1f} % "
      f"(should be 100.0)")
print(f"CHECK — between-patient rests on {_df_b} df "
      f"({int(_v['n_donors'])} donors); raw share before the correction was "
      f"{_v['pct_between_of_remainder'] + _v['pct_sampling_of_remainder']:.0f} % of "
      f"the remainder, of which "
      f"{100 * _v['pct_sampling_of_remainder'] / max(_v['pct_between_of_remainder'] + _v['pct_sampling_of_remainder'], 1e-9):.0f} % "
      f"is attributable to sampling in the donor means")
_dark = sum(1 for _, _, c, _, _ in bar1 if c == _VIOLET) + \
        sum(1 for _, _, c, _, _ in bar2 if c == _VIOLET)
print(f"CHECK — segments drawn in the base violet: {_dark} "
      f"(must be 1, so the hue carries one meaning)")
# Measured on the drawn text boxes, in printed mm: whether the texts inside the
# bars fit their width, whether the outside names overlap, and whether they stay
# inside the axes.
fig.canvas.draw()
_mm = lambda px: px / fig.dpi * MM_PER_IN * SCALE
_ab = ax.get_window_extent()
_bar_mm = _mm(ax.patches[0].get_window_extent().width)
_widest = max(_mm(t.get_window_extent().width) for t in ax.texts if t not in _names)
print(f"CHECK — widest text inside a bar {_widest:.1f} mm, bars {_bar_mm:.1f} mm wide -> "
      f"{'fits' if _widest < _bar_mm else 'TOO WIDE'}")
_nb = sorted((t.get_window_extent() for t in _names), key=lambda b: b.y0)
_gaps = [_mm(b.y0 - a.y1) for a, b in zip(_nb, _nb[1:])]
_edge = min(min(_mm(_ab.x1 - b.x1), _mm(b.y0 - _ab.y0), _mm(_ab.y1 - b.y1)) for b in _nb)
print(f"CHECK — outside names: smallest gap between two {min(_gaps):.1f} mm, closest to "
      f"a spine {_edge:.1f} mm -> {'clear' if min(_gaps) > 0 and _edge > 0 else 'OVERLAP'}")

save_panel(fig, "k", f"panelA_variance_{vp_bank}_{vp_region}".replace(" ", ""), SI6_DIR)

## 12.3 Per-protein differences between patients (Figure S6l)

In [ ]:
# ============================================================================
# Figure S6l, analysis: do individual proteins differ between patients?
#
# The pools, donors and protein set of Figure S6k (vp_Y), used as they are.
#
# Statistic per protein: the spread between the highest and the lowest donor mean
# of the pool-level ratio.
# Null: every distinct assignment of the pools to the donors, enumerated exactly.
# Whole samples move and gene labels stay, so the gene-gene correlation structure
# of the proteome is preserved. The statistic does not change when the donors are
# relabelled, so with equal group sizes each unordered partition is generated once.
# The host subtraction happens before permuting, so a pool keeps its own donor's
# host reference and only the donor label moves.
#
# The same enumeration serves both conditions (as measured, adjusted for the
# identifications per pool), so the two p-value columns share one null.
# ============================================================================
MAX_ARRANGEMENTS = 500_000   # exact enumeration only: exceeding this raises
FDR_CUT_B        = 0.05

for _needed in ("vp_Y", "vp_donors", "vp_donor_of_pool", "vp_depth"):
    if _needed not in globals():
        raise NameError(f"{_needed!r} is not defined. Run the Figure S6k analysis cell "
                        f"(section 12.2) first.")

Yv     = vp_Y.values
labels = vp_donor_of_pool.values
donors = list(vp_donors)
N, P   = Yv.shape
k      = len(donors)
sizes  = [int((labels == d).sum()) for d in donors]

# The covariate must be in the row order of the value matrix. A misaligned
# covariate fits noise, changes the answer and raises no error.
depth = vp_depth.reindex(vp_Y.index).values.astype(float)
assert not np.isnan(depth).any(), "identification count missing for at least one pool"
assert list(vp_depth.reindex(vp_Y.index).index) == list(vp_Y.index), "covariate misaligned"
print(f"Pools {N}, donors {k}, group sizes {sizes}, proteins {P}")
print(f"Identification count per pool: min {depth.min():.0f}, max {depth.max():.0f}, "
      f"spread {depth.max() - depth.min():.0f}")

# ------------------------------------------- depth-adjusted values
_dc  = depth - depth.mean()
_mu  = Yv.mean(axis=0)
_Yc  = Yv - _mu
_b   = (_dc @ _Yc) / (_dc @ _dc)
Yadj = _Yc - np.outer(_dc, _b) + _mu          # mean added back, so both x axes match
print(f"Depth slope across proteins: median {np.median(_b):+.2e} log2 per identification, "
      f"|slope| 95th percentile {np.percentile(np.abs(_b), 95):.2e}")

# ------------------------------------------------------- enumeration
_equal     = len(set(sizes)) == 1
n_ordered  = factorial(N) // int(np.prod([factorial(s) for s in sizes]))
n_distinct = n_ordered // factorial(k) if _equal else n_ordered
print(f"\nArrangements: ordered {n_ordered:,}"
      + (f", distinct with donor labels unordered {n_distinct:,}" if _equal else ""))
if n_distinct > MAX_ARRANGEMENTS:
    raise ValueError(f"{n_distinct:,} arrangements exceeds MAX_ARRANGEMENTS. The design "
                     f"calls for exact enumeration, so raise the cap deliberately "
                     f"rather than switching to sampling.")

def _gen_equal(idx, size, nb):
    """Partitions into unordered equal-size blocks, each emitted once. The range
    across donor means does not change when donor labels are swapped."""
    if nb == 1:
        yield (tuple(idx),); return
    first, rest = idx[0], idx[1:]
    for combo in itertools.combinations(rest, size - 1):
        used = set(combo)
        for tail in _gen_equal([i for i in rest if i not in used], size, nb - 1):
            yield ((first,) + combo,) + tail

def _gen_ordered(idx, szs):
    if not szs:
        yield (); return
    for combo in itertools.combinations(idx, szs[0]):
        used = set(combo)
        for tail in _gen_ordered([i for i in idx if i not in used], szs[1:]):
            yield (combo,) + tail

def _range_stat(blocks, M):
    means = np.stack([M[list(b)].mean(axis=0) for b in blocks])
    return means.max(axis=0) - means.min(axis=0)

blocks_obs = [np.flatnonzero(labels == d) for d in donors]
obs_meas   = _range_stat(blocks_obs, Yv)
obs_adj    = _range_stat(blocks_obs, Yadj)

gen = (_gen_equal(list(range(N)), sizes[0], k) if _equal
       else _gen_ordered(list(range(N)), sizes))
ge_meas, ge_adj = np.zeros(P, dtype=np.int64), np.zeros(P, dtype=np.int64)
for blocks in gen:
    ge_meas += _range_stat(blocks, Yv)   >= obs_meas - 1e-12
    ge_adj  += _range_stat(blocks, Yadj) >= obs_adj  - 1e-12

p_meas = np.maximum(ge_meas / n_distinct, 1.0 / n_distinct)
p_adj  = np.maximum(ge_adj  / n_distinct, 1.0 / n_distinct)
fdr_meas = multipletests(p_meas, method="fdr_bh")[1]
fdr_adj  = multipletests(p_adj,  method="fdr_bh")[1]

_gene_map = adata_log.var["single_gene_name"].to_dict()
vp_b_res = pd.DataFrame({
    "protein": vp_Y.columns,
    "single_gene_name": [_gene_map.get(v, v) for v in vp_Y.columns],
    "range_measured": obs_meas, "p_measured": p_meas, "fdr_measured": fdr_meas,
    "range_adjusted": obs_adj,  "p_adjusted": p_adj,  "fdr_adjusted": fdr_adj,
    "depth_slope": _b,
})
vp_b_meta = {"n_arrangements": int(n_distinct), "p_floor": 1.0 / n_distinct,
             "ordered_arrangements": int(n_ordered), "equal_sizes": bool(_equal),
             "fdr_cut": FDR_CUT_B, "correction": "Benjamini-Hochberg across proteins"}

# ----------------------------------------------------------------- CHECKS
print(f"\nPermutation p floor: {vp_b_meta['p_floor']:.3g} "
      f"({n_distinct:,} arrangements)")
for cond, fdr, rng in (("as measured", fdr_meas, obs_meas),
                       ("depth adjusted", fdr_adj, obs_adj)):
    _nl = (-np.log10(np.clip(fdr, 1e-300, None)))     # brackets: negate, then take the max
    print(f"\n{cond}:")
    print(f"  corrected p  min {fdr.min():.3g}   max {fdr.max():.3g}")
    print(f"  below {FDR_CUT_B}: {int((fdr < FDR_CUT_B).sum())} / {P}")
    print(f"  largest -log10(corrected p): {_nl.max():.3f}")
    print(f"  spread  min {rng.min():.3f}   max {rng.max():.3f} log2")
if fdr_meas.min() >= FDR_CUT_B:
    print(f"\nNOTE: nothing reaches FDR < {FDR_CUT_B}. With a p floor of "
          f"{vp_b_meta['p_floor']:.3g} and {P} tests, the smallest attainable BH value "
          f"is {vp_b_meta['p_floor'] * P:.3g}, so the enumeration size, not the effect, "
          f"may be the binding constraint.")

_stub = f"{today}_panelB_perprotein_{vp_bank}_{vp_region}".replace(" ", "")
vp_b_res.to_csv(f"{output_dir}/{_stub}.csv", index=False)
print(f"\nSaved {_stub}.csv to {output_dir}/")

In [ ]:
# ============================================================================
# Figure S6l, plot: spread between donors against corrected p, as measured
# (violet) and adjusted for the identifications per pool (grey), on one axes.
# Proteins with FDR below the cut lie above the dashed line and are drawn larger,
# with a white edge.
# ============================================================================
DOT_NS, DOT_SIG = native_pt(1.0), native_pt(2.0)   # dot diameters in print
LEG_DOT       = native_pt(3.5)   # legend markers, larger than the data dots
ALPHA_NS      = 0.30
HEADROOM      = 1.18       # multiple of the largest -log10 value
LEG_Y         = 0.0        # legend bottom, in axes fraction: the lower right corner
# Grey for the depth-adjusted values: darker than BG_COLOR, so the dots in the
# band at the bottom stay visible on top of the violet ones.
COND_COLOR = {"as measured": CT_BASE["excised"], "depth adjusted": "#8c8c8c"}

_thr_y = -np.log10(vp_b_meta["fdr_cut"])
_conds = [("as measured",    vp_b_res["range_measured"].values, vp_b_res["fdr_measured"].values),
          ("depth adjusted", vp_b_res["range_adjusted"].values, vp_b_res["fdr_adjusted"].values)]

# Brackets matter: -np.log10(v).max() would be the negated log of the largest
# corrected p, a small negative number, and every point would lie above the axis.
_all_y = np.concatenate([(-np.log10(np.clip(f, 1e-300, None))) for _, _, f in _conds])
_all_x = np.concatenate([r for _, r, _ in _conds])
xlim = (0.0, float(_all_x.max()) * 1.06)
ylim = (0.0, max(float(_all_y.max()) * HEADROOM, _thr_y * 1.35))

fig, axm = create_figure(1, 1, figsize=figsize(1, 1))
ax = axm.next()

ax.axhline(_thr_y, ls="--", color=REF_COLOR, lw=AXES_LW, zorder=2)
ax.text(xlim[1] * 0.99, _thr_y + ylim[1] * 0.012,
        f"FDR {vp_b_meta['fdr_cut']}", ha="right", va="bottom",
        fontsize=ANNOT_FS, color=REF_COLOR, zorder=3)

# The depth-adjusted values are drawn on top: they form a band at the bottom,
# where many of the as-measured values lie as well.
legend_handles = []
for k, (cond, rng, fdr) in enumerate(_conds):
    y   = -np.log10(np.clip(fdr, 1e-300, None))
    sig = fdr < vp_b_meta["fdr_cut"]
    ax.scatter(rng[~sig], y[~sig], s=DOT_NS ** 2, color=COND_COLOR[cond], alpha=ALPHA_NS,
               linewidths=0, zorder=3 + k)
    ax.scatter(rng[sig], y[sig], s=DOT_SIG ** 2, color=COND_COLOR[cond],
               edgecolors="white", linewidths=MARKER_LW, zorder=5 + k)
    # Counted from the plotted arrays, so the legend matches what is drawn.
    legend_handles.append(Line2D(
        [0], [0], marker="o", linestyle="none", markersize=LEG_DOT,
        markerfacecolor=COND_COLOR[cond], markeredgecolor="white",
        markeredgewidth=MARKER_LW,
        label=f"{cond.capitalize()}\n{int(sig.sum())} below FDR"))

# Legend in the lower right corner, where the band of depth-adjusted values
# thins out. Its entries name the count only: with "of N below FDR 0.05" the
# legend is wider and covers dots in the upper left. handlelength 0.6 holds the
# 3.5 pt legend marker.
ax.legend(handles=legend_handles, loc="lower right", bbox_to_anchor=(1, LEG_Y),
          frameon=False, handlelength=0.6, handletextpad=0.4, borderaxespad=0.3,
          labelspacing=0.5)

ax.set_xlim(*xlim); ax.set_ylim(*ylim)
ax.grid(False)
label_axes(ax, xlabel="Spread between donors (log2)",
           ylabel="−log10 corrected p")

print(f"CHECK — exact permutation over donor labels of whole pools: "
      f"{vp_b_meta['n_arrangements']:,} arrangements, attainable p floor "
      f"{vp_b_meta['p_floor']:.3g}. {vp_b_meta['correction']}.")
# ----------------------------------------------------------------- CHECKS
print(f"CHECK — x limits {xlim[0]:.3f} to {xlim[1]:.3f}; largest plotted spread "
      f"{_all_x.max():.3f}  -> inside: {_all_x.max() <= xlim[1]}")
print(f"CHECK — y limits {ylim[0]:.3f} to {ylim[1]:.3f}; largest plotted "
      f"-log10(corrected p) {_all_y.max():.3f}  -> inside: {_all_y.max() <= ylim[1]}")
print(f"CHECK — headroom above the highest point: "
      f"{ylim[1] - _all_y.max():.3f} on the y scale")
for cond, rng, fdr in _conds:
    print(f"CHECK — {cond}: below threshold {int((fdr < vp_b_meta['fdr_cut']).sum())}, "
          f"corrected p range {fdr.min():.3g} to {fdr.max():.3g}")
fig.canvas.draw()
_lb = ax.get_legend().get_window_extent().transformed(ax.transData.inverted())
_under = 0
for _, rng, fdr in _conds:
    _y = -np.log10(np.clip(fdr, 1e-300, None))
    _under += int(((rng >= _lb.x0) & (rng <= _lb.x1) & (_y >= _lb.y0) & (_y <= _lb.y1)).sum())
print(f"CHECK — plotted points under the legend: {_under} of {sum(len(f) for _, _, f in _conds)}")

save_panel(fig, "l", f"panelB_perprotein_{vp_bank}_{vp_region}".replace(" ", ""), SI6_DIR)

# 13. Replication cohort

## 13.1 Joint search of both cohorts

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Joint search: ADVP04_011_17_CombinedwithBannerLibrary holds all 385 discovery
# samples and the 149 samples of the replication cohort (ADVP017), quantified
# together. Both cohorts therefore share one MaxLFQ/QuantUMS solution, one
# cross-run normalisation and one protein-group inference, which a comparison
# across cohorts needs. Sections 13.2 to 13.4 read the h5ad this cell writes.
#
# The primary figures come from ADVP04_011_Combined_withBannerLibrary, a separate
# search of the 385 discovery samples. The panels of this section show values of
# the joint search only.
#
# The processing parameters and the contaminant list are the objects of the
# preprocessing cell (section 2.3).
# ═══════════════════════════════════════════════════════════════════════
PILOT_BANK = "BH2"          # BrainBank value that identifies the replication cohort
N_DISCOVERY, N_PILOT = 385, 149
AGG_CLASS, HOST_CLASS = "Round Aggregate Aggregate", "Neuron DAB Positive"

for _n in ("BRAIN_EXTRA_CONTAMINANTS", "MIN_PROTEINS", "MAX_MISSING_COND",
           "forfiltering", "var_metadata", "complete_in_any_group"):
    if _n not in globals():
        raise NameError(f"{_n!r} is not defined. Run the imports, data-load and "
                        f"preprocessing cells first; this cell reuses their contaminant "
                        f"list, thresholds and completeness rule.")
print(f"Reusing preprocessing parameters: MIN_PROTEINS={MIN_PROTEINS}, "
      f"MAX_MISSING_COND={MAX_MISSING_COND}, forfiltering={forfiltering!r}")

# ------------------------------------------------------------------ load
jt = at.io.read_pg_table(f"{JOINT_DIR}/report.pg_matrix.tsv", search_engine="diann")
_jt_meta = pd.read_csv(f"{JOINT_DIR}/metadata.csv", sep=",", index_col=0)
jt = at.pp.add_metadata(adata=jt, incoming_metadata=_jt_meta, axis=0,
                        keep_data_shape=False)
jt.var = jt.var.merge(var_metadata, left_index=True, right_index=True, how="left")
print(f"\nJoint search loaded: {jt.n_obs} samples x {jt.n_vars} protein groups")

# --------------------------------------------------------------- cohorts
jt.obs["Cohort"] = np.where(jt.obs["BrainBank"].astype(str) == PILOT_BANK,
                            "Pilot", "Discovery")
_cc = jt.obs["Cohort"].value_counts().to_dict()
print(f"Cohorts: {_cc}")
if _cc.get("Discovery") != N_DISCOVERY or _cc.get("Pilot") != N_PILOT:
    raise ValueError(f"Expected {N_DISCOVERY} discovery and {N_PILOT} pilot samples, "
                     f"got {_cc}. Check that BrainBank=={PILOT_BANK!r} still "
                     f"identifies the pilot.")

# CHECK: both cohorts come from this one matrix.
print(f"CHECK — single search for both cohorts: {JOINT_DIR.split('/')[-1]}")
print(f"        primary figures come from a DIFFERENT search "
      f"({input_dir.rstrip('/').split('/')[-1]}) and must not be mixed in.")

# ----------------------------------------- the inclusion classes must be labelled
_unlabelled = jt.obs["Cell Type"].isna() | jt.obs["CellType_Aggregate"].astype(str).isin(
    ["Without Aggregate", "nan"])
if _unlabelled.any():
    _bad = jt.obs.loc[_unlabelled, ["Cohort", "Brain", "Brain Region",
                                    "Cell Type", "Aggregate", "CellType_Aggregate"]]
    raise ValueError(
        f"{int(_unlabelled.sum())} samples have no dissected class: 'Cell Type' is "
        f"missing and 'CellType_Aggregate' collapsed to 'Without Aggregate'. These "
        f"are the RoundAggsSmall / RoundAggsLarge wells. Set 'Cell Type' = "
        f"'Round Aggregate' and 'Aggregate' = 'Aggregate' for them in "
        f"{JOINT_DIR}/metadata.csv, so CellType_Aggregate becomes "
        f"{AGG_CLASS!r}, then re-run.\n{_bad.to_string()}")
for _c in (AGG_CLASS, HOST_CLASS):
    if _c not in set(jt.obs["CellType_Aggregate"].dropna().unique()):
        raise KeyError(f"{_c!r} absent from CellType_Aggregate.")
print(f"\nCellType_Aggregate levels: "
      f"{sorted(jt.obs['CellType_Aggregate'].dropna().unique())}")

# --------------------------------------------------------------- filtering
# The sample filters and the completeness filter of section 2.3.
jt.obs["num_proteins"] = np.sum(~np.isnan(jt.X), axis=1)
_n0 = jt.n_obs
jt = jt[~jt.obs["Disease_CellType_Aggregate"].isin(["Blank", "EvoTip Blank"]), :].copy()
_n1 = jt.n_obs
jt = jt[jt.obs["num_proteins"] >= MIN_PROTEINS, :].copy()
_n2 = jt.n_obs
_nan = jt.obs["Disease_CellType_Aggregate"].isna() | jt.obs[forfiltering].isna()
if _nan.any():
    print(f"WARNING: dropping {int(_nan.sum())} samples with a missing group label")
    jt = jt[~_nan, :].copy()
print(f"\nSamples: {_n0} -> {_n1} (blanks) -> {_n2} (<{MIN_PROTEINS} proteins) "
      f"-> {jt.n_obs} (labels)")
print(f"  per cohort now: {jt.obs['Cohort'].value_counts().to_dict()}")

# Contaminants: the rules and the list of section 2.3.
jt.var["single_gene_name"] = jt.var["genes"].str.split(";").str[0].copy()
jt.var = flag_contaminants(jt.var, extra=BRAIN_EXTRA_CONTAMINANTS)
_ncont = int(jt.var["is_contaminant"].sum())
jt = jt[:, ~jt.var["is_contaminant"]].copy()
print(f"Contaminants dropped: {_ncont} -> {jt.n_vars} protein groups")

jt.var["median_intensity"] = np.nanmedian(jt.X, axis=0)   # one protein group per symbol, section 13.2

jt = complete_in_any_group(jt, forfiltering, MAX_MISSING_COND)
print(f"After {int(100 * (1 - MAX_MISSING_COND))}%-per-condition completeness: "
      f"{jt.n_vars} protein groups")

jt_log = jt.copy()
sc.pp.log1p(jt_log, base=2)
jt_log.layers["linear"] = jt.X.copy()     # linear intensities for the sums of section 13.4
print(f"\njt_log: {jt_log.n_obs} x {jt_log.n_vars}  (X = log2, layers['linear'] = raw)")

# ------------------------------- strata of sections 13.2 to 13.4
_o = jt_log.obs
_dis = _o[_o["disease_binary"].astype(str) == "Disease"]
_ct = (_dis.groupby(["Cohort", "Brain", "Brain Region", "CellType_Aggregate"],
                    observed=True).size().unstack(fill_value=0))
for _c in (AGG_CLASS, HOST_CLASS):
    if _c not in _ct:
        _ct[_c] = 0
_ct = _ct[[AGG_CLASS, HOST_CLASS]].rename(columns={AGG_CLASS: "round_agg",
                                                   HOST_CLASS: "host_DABpos"})
print("\nCHECK — every donor x region stratum, Disease donors, both classes:")
print(_ct.to_string())
jt_strata = _ct[(_ct["round_agg"] > 0) & (_ct["host_DABpos"] > 0)]
print(f"\nStrata WITH both classes (these are used): {len(jt_strata)}")
print(jt_strata.to_string())
_dropped_hosts = _ct[(_ct["round_agg"] == 0) & (_ct["host_DABpos"] > 0)]
print(f"\nHost wells dropped, no round aggregates in that donor x region: "
      f"{int(_dropped_hosts['host_DABpos'].sum())} wells in {len(_dropped_hosts)} strata")
_single = jt_strata[jt_strata["host_DABpos"] == 1]
if len(_single):
    print(f"\nFLAG — single-well host reference in {len(_single)} stratum/strata. The "
          f"host mean there is one measurement, so that donor's ratio carries the full "
          f"error of a single well and no within-stratum host variance is estimable:")
    print(_single.to_string())
print(f"\nDonors contributing per cohort: "
      f"{ {c: sorted({i[1] for i in jt_strata.index if i[0] == c}) for c in ('Discovery', 'Pilot')} }")

# ------------------------------------------------------------------- save
# h5py cannot write an object column that mixes strings with missing values, and
# the pg_matrix annotation columns (Protein.Group, Protein.Ids, Genes, ...) do.
for _df in (jt_log.var, jt_log.obs):
    for _c in _df.columns:
        if _df[_c].dtype == object:
            _df[_c] = _df[_c].fillna("").astype(str)
_jt_path = f"{output_dir}/{today}_jointsearch_log.h5ad"
jt_log.write_h5ad(_jt_path)
print(f"\nSaved {_jt_path}")

## 13.2 Set-level replication (Figure 4g)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 4g, analysis: set-level replication, discovery against replication
# cohort.
#
# Both cohorts from the joint search, so one quantification underlies both axes.
# The donor is the statistical unit in both. Significance comes from the
# discovery cohort only; the replication cohort contributes direction and
# magnitude, and no p value is computed for it.
#
# Reads the h5ad written by section 13.1 and the GO CC gene sets of section 10.
# ═══════════════════════════════════════════════════════════════════════

AGG_CLASS, HOST_CLASS = "Round Aggregate Aggregate", "Neuron DAB Positive"
MAIN_COHORT, REPL_COHORT = "Discovery", "Pilot"          # obs["Cohort"] values
MAIN_LABEL, REPL_LABEL = "Discovery cohort", "Replication cohort"   # display only
SET_MIN_DETECTED = 15        # below this the set mean is unstable
SET_MAX_DETECTED = 500       # above this the competitive score is not specific
MAX_SET_OVERLAP  = 0.50      # collapse sets sharing more than this of their members
ALPHA            = 0.05

_h5 = max(glob.glob(f"{output_dir}/*_jointsearch_log.h5ad"),
          key=lambda f: Path(f).stat().st_mtime, default=None)
if _h5 is None:
    raise FileNotFoundError(f"No *_jointsearch_log.h5ad in {output_dir}. Run section 13.1.")
jt = sc.read_h5ad(_h5)
print(f"Joint search: {Path(_h5).name}  {jt.shape}")
print(f"CHECK — one search for both cohorts, "
      f"cohorts present: {sorted(jt.obs['Cohort'].unique())}")

# GO CC gene sets of section 10, without the generic container terms.
cc = gocc[~gocc["source"].isin(GENERIC_COMPARTMENTS)]
print(f"GO CC: {cc['source'].nunique()} compartments "
      f"({len(GENERIC_COMPARTMENTS)} generic containers excluded)")

# ------------------------------------------------- strata, disease donors only
obs = jt.obs
elig = obs[(obs["disease_binary"].astype(str) == "Disease")
           & (obs["CellType_Aggregate"].isin([AGG_CLASS, HOST_CLASS]))]
_ct = (elig.groupby(["Cohort", "Brain", "Brain Region", "CellType_Aggregate"],
                    observed=True).size().unstack(fill_value=0))
for _c in (AGG_CLASS, HOST_CLASS):
    if _c not in _ct:
        _ct[_c] = 0
strata = _ct[(_ct[AGG_CLASS] > 0) & (_ct[HOST_CLASS] > 0)][[AGG_CLASS, HOST_CLASS]]
strata.columns = ["n_agg", "n_host"]
print(f"\nCHECK — strata used ({len(strata)}), sample count per class:")
print(strata.to_string())
_single = strata[strata["n_host"] == 1]
if len(_single):
    print(f"FLAG — single-well host reference: {[i[1] for i in _single.index]}")
_dropped = _ct[(_ct[AGG_CLASS] == 0) & (_ct[HOST_CLASS] > 0)]
print(f"Host wells dropped (no round aggregates in that donor x region): "
      f"{int(_dropped[HOST_CLASS].sum())}")

# ------------------------------------------- complete cases over the wells used
_used = elig.index[[(c, b, r) in strata.index for c, b, r in
                    zip(elig["Cohort"], elig["Brain"], elig["Brain Region"])]]
X = jt[_used, :].to_df()
complete = X.columns[X.notna().all(axis=0)]
X = X[complete]
print(f"\nCHECK — samples used: {len(_used)}  "
      f"({elig.loc[_used, 'Cohort'].value_counts().to_dict()})")
print(f"CHECK — complete-case proteins across ALL wells used, both cohorts: "
      f"{len(complete)} of {jt.n_vars}")

# ------------------------------------- per-donor signature, region by region
_meta = elig.loc[_used]
_rows = {}
for (coh, don, reg), _ in strata.iterrows():
    _m = _meta[(_meta["Cohort"] == coh) & (_meta["Brain"] == don)
               & (_meta["Brain Region"] == reg)]
    a = X.loc[_m.index[_m["CellType_Aggregate"] == AGG_CLASS]].mean(axis=0)
    h = X.loc[_m.index[_m["CellType_Aggregate"] == HOST_CLASS]].mean(axis=0)
    _rows[(coh, don, reg)] = a - h
region_delta = pd.DataFrame(_rows).T
region_delta.index = pd.MultiIndex.from_tuples(region_delta.index,
                                               names=["Cohort", "Brain", "Region"])
# Unweighted region average, so each donor contributes exactly one value.
sig = region_delta.groupby(level=["Cohort", "Brain"], observed=True).mean()
_nreg = region_delta.groupby(level=["Cohort", "Brain"], observed=True).size()
print(f"\nRegions averaged per donor:\n{_nreg.to_string()}")
a1_signature = sig                      # donors x proteins, read by section 13.3
donors_main = [i[1] for i in sig.index if i[0] == MAIN_COHORT]
donors_repl = [i[1] for i in sig.index if i[0] == REPL_COHORT]
print(f"Donors: {MAIN_COHORT} {donors_main}\n        {REPL_COHORT} {donors_repl}")

# ---------------------------- gene symbols, one representative per symbol
# The most abundant protein group (median intensity in the joint search).
_sym = jt.var["single_gene_name"].reindex(sig.columns).astype(str)
_ab  = jt.var["median_intensity"].reindex(sig.columns)
_rep = (pd.DataFrame({"pid": sig.columns, "gene": _sym.values, "abund": _ab.values})
          .query("gene not in ['nan', '', 'None']")
          .sort_values("abund", ascending=False).drop_duplicates("gene"))
G = sig[_rep["pid"].tolist()]
G.columns = _rep["gene"].tolist()
universe = set(G.columns)
print(f"Gene universe: {len(universe)} symbols from {sig.shape[1]} protein groups")

# ----------------------------------------------- set selection and collapse
_mem = cc.groupby("source")["target"].apply(lambda s: set(s.dropna()))
_det = {t: _mem[t] & universe for t in _mem.index}
_ok  = [t for t, d in _det.items() if SET_MIN_DETECTED <= len(d) <= SET_MAX_DETECTED]
print(f"\nSets with detected membership in [{SET_MIN_DETECTED}, {SET_MAX_DETECTED}]: "
      f"{len(_ok)} of {len(_det)}")

def _set_effect(gene_frame, members):
    """Set mean minus the mean over all non-members, per donor."""
    others = [g for g in gene_frame.columns if g not in members]
    return gene_frame[sorted(members)].mean(axis=1) - gene_frame[others].mean(axis=1)

_eff = {t: _set_effect(G, _det[t]) for t in _ok}
_main_mean = pd.Series({t: v.loc[MAIN_COHORT].mean() for t, v in _eff.items()})

# Collapse near-duplicate memberships, strongest main-cohort effect kept.
_order = _main_mean.abs().sort_values(ascending=False).index
kept, absorbed = [], {}
for t in _order:
    hit = next((s for s in kept
                if len(_det[t] & _det[s]) / min(len(_det[t]), len(_det[s]))
                > MAX_SET_OVERLAP), None)
    if hit is None:
        kept.append(t)
    else:
        absorbed.setdefault(hit, []).append(t)
print(f"After collapsing membership overlap > {MAX_SET_OVERLAP}: {len(kept)} sets "
      f"({sum(len(v) for v in absorbed.values())} absorbed)")

# --------------------------------- per-cohort means and the discovery test
rows = []
for t in kept:
    v = _eff[t]
    vm = v.loc[MAIN_COHORT].values
    vr = v.loc[REPL_COHORT].values
    tt = stats.ttest_1samp(vm, 0.0)
    rows.append({"set": t, "label": t.replace("GOCC_", "").replace("_", " ").capitalize(),
                 "n_detected": len(_det[t]),
                 "main_mean": float(vm.mean()), "repl_mean": float(vr.mean()),
                 "main_sd": float(np.std(vm, ddof=1)),
                 "repl_sd": float(np.std(vr, ddof=1)) if len(vr) > 1 else np.nan,
                 "p_main": float(tt.pvalue)})
a1 = pd.DataFrame(rows)
a1["fdr_main"] = multipletests(a1["p_main"], method="fdr_bh")[1]
a1["sig_main"] = a1["fdr_main"] < ALPHA
a1["same_sign"] = np.sign(a1["main_mean"]) == np.sign(a1["repl_mean"])
for d in donors_main:
    a1[f"main_{d}"] = [float(_eff[t].loc[(MAIN_COHORT, d)]) for t in a1["set"]]
for d in donors_repl:
    a1[f"repl_{d}"] = [float(_eff[t].loc[(REPL_COHORT, d)]) for t in a1["set"]]
print(f"\nSets tested: {len(a1)};  significant in {MAIN_COHORT} at BH<{ALPHA}: "
      f"{int(a1['sig_main'].sum())}")
print(f"NOTE: no p value is computed for {REPL_COHORT}. It contributes direction "
      f"and magnitude only.")

# ----------------------------------------------------------------- CHECKS
def _agree(d):
    r_p = stats.pearsonr(d["main_mean"], d["repl_mean"])
    r_s = stats.spearmanr(d["main_mean"], d["repl_mean"])
    return r_p[0], r_s[0], float(d["same_sign"].mean()), len(d)

_all = _agree(a1)
_sub = _agree(a1[a1["sig_main"]]) if a1["sig_main"].any() else (np.nan,)*4
print(f"\nCHECK — all {len(a1)} sets tested:")
print(f"  Pearson r {_all[0]:+.3f}   Spearman rho {_all[1]:+.3f}   "
      f"sign agreement {100*_all[2]:.0f} % ({int(_all[2]*_all[3])}/{_all[3]})")
print(f"CHECK — the {int(a1['sig_main'].sum())} sets significant in {MAIN_COHORT}:")
print(f"  Pearson r {_sub[0]:+.3f}   Spearman rho {_sub[1]:+.3f}   "
      f"sign agreement {100*_sub[2]:.0f} % ({int(_sub[2]*_sub[3])}/{_sub[3]})")
print("  Both are reported because they differ: the significant subset is selected "
      "on the main cohort, which inflates its apparent agreement.")

# Difference against mean (Bland-Altman): does one cohort compress the whole
# signature? diff = s * mean implies repl = main * (1 + s/2) / (1 - s/2), so the
# slope converts into a multiplicative scale factor.
_mean = (a1["main_mean"] + a1["repl_mean"]) / 2
_diff = a1["repl_mean"] - a1["main_mean"]
_fit = stats.linregress(_mean, _diff)
_ci = 1.96 * _fit.stderr
_lo, _hi = _fit.slope - _ci, _fit.slope + _ci
_scale = (1 + _fit.slope / 2) / (1 - _fit.slope / 2)
print(f"\nCHECK — difference against mean: slope {_fit.slope:+.3f} "
      f"(95% CI {_lo:+.3f} to {_hi:+.3f}), "
      f"intercept {_fit.intercept:+.3f}, r2 {_fit.rvalue**2:.3f}")
if _lo <= 0 <= _hi:
    print("  The interval includes zero: the cohorts agree in magnitude as well as in "
          "direction.")
else:
    print(f"  The interval excludes zero: the {REPL_LABEL.lower()} is "
          f"{'compressed' if _fit.slope < 0 else 'expanded'} relative to the "
          f"{MAIN_LABEL.lower()} by a factor of about {_scale:.2f} across the whole "
          f"signature.")

# Leave-one-donor-out on the replication cohort, which has three donors.
print(f"\nCHECK — leave-one-donor-out on {REPL_COHORT} (all sets, Pearson r):")
for d in donors_repl:
    _keep = [x for x in donors_repl if x != d]
    _y = np.array([[float(_eff[t].loc[(REPL_COHORT, x)]) for x in _keep]
                   for t in a1["set"]]).mean(axis=1)
    print(f"  without {d:<12} r = {stats.pearsonr(a1['main_mean'], _y)[0]:+.3f}")

_stub = f"{today}_A1_setreplication".replace(" ", "")
a1.to_csv(f"{output_dir}/{_stub}.csv", index=False)
a1_signature.to_csv(f"{output_dir}/{today}_A1_donor_signatures.csv")
print(f"\nSaved {_stub}.csv and {today}_A1_donor_signatures.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure 4g. Set effect in the discovery cohort (x) against the replication cohort
# (y), from the analysis cell above, drawn by replication_scatter
# (src/scatter.py). The text block gives r, n and sign agreement for the sets
# significant in the discovery cohort, then for all sets in brackets: the subset
# is selected on the discovery cohort, so its agreement is higher by construction.
# Not on the panel, and stated in the figure legend text: the marker encoding
# (filled = significant in the discovery cohort) and the slope of the difference
# against the mean (printed by the analysis cell).
# ═══════════════════════════════════════════════════════════════════════
print(f"CHECK — both cohorts from one search: {Path(_h5).name}")
print(f"CHECK — sets plotted {len(a1)}; significant in {MAIN_LABEL.lower()} "
      f"{int(a1['sig_main'].sum())} (filled), remainder {int((~a1['sig_main']).sum())} (open).")
replication_scatter(a1["main_mean"], a1["repl_mean"], a1["sig_main"],
                    "g", "A1_setreplication", FIG4_DIR,
                    xlabel=f"{MAIN_LABEL}\nset effect (log2)",
                    ylabel=f"{REPL_LABEL}\nset effect (log2)",
                    hit_color=CT_BASE["excised"], alpha=ALPHA,
                    ns_open=True, ns_pt=1.6, hit_pt=2.2, all_label="sets tested")
print("NOTE: no p value is plotted or quoted for the replication cohort.")

## 13.3 Protein-level replication (Figure S6m)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure S6m, analysis: protein-level replication. The per-donor signatures of
# section 13.2 (same strata, same complete-case protein set, same region
# averaging). Significance comes from the discovery cohort only.
# ═══════════════════════════════════════════════════════════════════════
MIN_ABS_EFFECT = 0.5      # log2, for the third sign-agreement figure
NULL_AGREEMENT = 0.50     # agreement expected with no replication at all

if "a1_signature" in globals():
    sig = a1_signature
    print("Using a1_signature from the kernel.")
else:                     # the CSV of section 13.2, so this cell also runs in a new kernel
    _p = max(glob.glob(f"{output_dir}/*_A1_donor_signatures.csv"),
             key=lambda f: Path(f).stat().st_mtime, default=None)
    if _p is None:
        raise FileNotFoundError("No A1 donor signatures found. Run section 13.2 first.")
    sig = pd.read_csv(_p, index_col=[0, 1])
    print(f"Using {Path(_p).name}")
_main = sig.loc[MAIN_COHORT]
_repl = sig.loc[REPL_COHORT]
print(f"{MAIN_LABEL}: {len(_main)} donors {list(_main.index)}")
print(f"{REPL_LABEL}: {len(_repl)} donors {list(_repl.index)}")
print(f"Proteins: {sig.shape[1]} (the A1 complete-case set, not rebuilt)")

# ------------------------------------- per-cohort means and the discovery test
_tt = stats.ttest_1samp(_main.values, 0.0, axis=0)
b1 = pd.DataFrame({
    "protein": sig.columns,
    "single_gene_name": jt.var["single_gene_name"].reindex(sig.columns).astype(str).values,
    "main_mean": _main.mean(axis=0).values,
    "repl_mean": _repl.mean(axis=0).values,
    "main_sd":   _main.std(axis=0, ddof=1).values,
    "repl_sd":   _repl.std(axis=0, ddof=1).values if len(_repl) > 1 else np.nan,
    "p_main":    np.asarray(_tt.pvalue, dtype=float),
})
b1["fdr_main"] = multipletests(b1["p_main"], method="fdr_bh")[1]
b1["sig_main"] = b1["fdr_main"] < ALPHA
b1["same_sign"] = np.sign(b1["main_mean"]) == np.sign(b1["repl_mean"])
print(f"\nSignificant in {MAIN_LABEL.lower()} at BH<{ALPHA}: "
      f"{int(b1['sig_main'].sum())} of {len(b1)}")
print(f"NOTE: no p value is computed for the {REPL_LABEL.lower()}.")

# ------------------------------------------------- the three agreement figures
def _stat(d, name):
    if len(d) < 3:
        print(f"  {name:<34} n={len(d)} — too few for a correlation")
        return dict(subset=name, n=len(d), r=np.nan, rho=np.nan, agree=np.nan)
    r  = stats.pearsonr(d["main_mean"], d["repl_mean"])[0]
    rho = stats.spearmanr(d["main_mean"], d["repl_mean"])[0]
    a  = float(d["same_sign"].mean())
    print(f"  {name:<34} n={len(d):>5}  r={r:+.3f}  rho={rho:+.3f}  "
          f"sign agreement {100*a:>5.1f}%  ({int(d['same_sign'].sum())}/{len(d)})")
    return dict(subset=name, n=len(d), r=r, rho=rho, agree=a)

print(f"\nCHECK — agreement, with {100*NULL_AGREEMENT:.0f}% expected under no "
      f"replication:")
_rows = [_stat(b1, "all shared proteins"),
         _stat(b1[b1["sig_main"]], f"{MAIN_LABEL.lower()} hits"),
         _stat(b1[b1["main_mean"].abs() > MIN_ABS_EFFECT],
               f"|effect| > {MIN_ABS_EFFECT} log2")]
b1_agreement = pd.DataFrame(_rows)
print("  The first two differ substantially and both belong in the panel: the hit "
      "subset is selected on the discovery cohort, which raises its agreement.")

_stub = f"{today}_B1_proteinreplication".replace(" ", "")
b1.to_csv(f"{output_dir}/{_stub}.csv", index=False)
b1_agreement.to_csv(f"{output_dir}/{_stub}_agreement.csv", index=False)
print(f"\nSaved {_stub}.csv and {_stub}_agreement.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure S6m. Per-protein aggregate-over-host effect in the discovery cohort (x)
# against the replication cohort (y), drawn by replication_scatter
# (src/scatter.py); text block as in Figure 4g. No gene labels: Figure 4d names
# the proteins. Not on the panel, and stated in the figure legend text: the
# marker encoding (prominent = significant in the discovery cohort), the 50 % sign
# agreement expected with no replication, and the agreement among proteins with
# |effect| > MIN_ABS_EFFECT; all three are printed below.
# ═══════════════════════════════════════════════════════════════════════
# The three subsets, looked up by the names the analysis cell writes, so a renamed
# subset stops here with a KeyError.
_a        = b1_agreement.set_index("subset")
_all_key  = "all shared proteins"
_hit_key  = f"{MAIN_LABEL.lower()} hits"
_eff_key  = f"|effect| > {MIN_ABS_EFFECT} log2"
_expected = [_all_key, _hit_key, _eff_key]
_absent   = [k for k in _expected if k not in _a.index]
if _absent:
    raise KeyError(f"b1_agreement has no row(s) {_absent}; it holds "
                   f"{list(_a.index)}. The subset names are written by the "
                   f"analysis cell above and read here.")

print(f"CHECK — proteins plotted {len(b1)}: {int(b1['sig_main'].sum())} hits (prominent), "
      f"{int((~b1['sig_main']).sum())} others (recessive).")
for _k in (_all_key, _hit_key, _eff_key):
    _r = _a.loc[_k]
    print(f"CHECK — {_k:<34} n={int(_r['n']):>5}  r={_r['r']:+.3f}  "
          f"sign agreement {100 * _r['agree']:.1f}%")
print(f"CHECK — sign agreement expected with no replication: "
      f"{100 * NULL_AGREEMENT:.0f}%")

replication_scatter(b1["main_mean"], b1["repl_mean"], b1["sig_main"],
                    "m", "B1_proteinreplication", SI6_DIR,
                    xlabel=f"{MAIN_LABEL}\nlog2 aggregate / host",
                    ylabel=f"{REPL_LABEL}\nlog2 aggregate / host",
                    hit_color=CT_BASE["excised"], alpha=ALPHA,
                    ns_open=False, ns_pt=1.0, hit_pt=2.0, all_label="shared proteins")
print("NOTE: no p value is plotted or quoted for the replication cohort.")

## 13.4 Membrane-to-nuclear ratio in the replication cohort (Figure S6n)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure S6n, analysis: membrane-to-nuclear ratio per sample in both cohorts of
# the joint search: log2 of the summed linear intensities of the two protein
# groups, with no normalisation, correction or donor-level model.
#
# The two term lists are those of Figure 4e (section 10.1), so the groups are the
# same. Their detected members are printed in full below.
# ═══════════════════════════════════════════════════════════════════════
for _n in ("MEMBRANE_TERMS", "NUCLEAR_TRANSLATION_TERMS", "genes_for"):
    if _n not in globals():
        raise NameError(f"{_n!r} is not defined. Run the GO CC cell of section 10 and "
                        f"the cell of section 10.1 that begins "
                        f"'# Membrane and vesicle proteins over soluble nuclear'; "
                        f"this panel uses the two groups of Figure 4e.")

CLASS_ORDER = ["Neuron Without Aggregate", "Neuron DAB Positive",
               "Glia Without Aggregate",   "Glia DAB Positive",
               "Neuropil Without Aggregate",
               "Round Aggregate Aggregate", "Elongated Aggregate Aggregate"]
COHORT_ORDER = [MAIN_COHORT, REPL_COHORT]
EPS = 1e-12

_h5 = max(glob.glob(f"{output_dir}/*_jointsearch_log.h5ad"),
          key=lambda f: Path(f).stat().st_mtime, default=None)
if _h5 is None:
    raise FileNotFoundError(f"No *_jointsearch_log.h5ad in {output_dir}. Run section 13.1.")
jt = sc.read_h5ad(_h5)
print(f"Joint search: {Path(_h5).name}  {jt.shape}")

# --------------------------------------------- GO terms to genes (section 10)
mem_genes, mem_missing = genes_for(MEMBRANE_TERMS)
nuc_genes, nuc_missing = genes_for(NUCLEAR_TRANSLATION_TERMS)
_overlap = mem_genes & nuc_genes
mem_only, nuc_only = mem_genes - _overlap, nuc_genes - _overlap
print(f"\nMEMBRANE/VESICULAR:  {len(mem_genes)} genes "
      f"({len(mem_only)} after dropping the overlap)")
print(f"NUCLEAR/TRANSLATION: {len(nuc_genes)} genes ({len(nuc_only)} after)")
print(f"Ambiguous, in both:  {len(_overlap)} genes — dropped from BOTH groups")
if mem_missing: print(f"  MEMBRANE terms not in the cache: {mem_missing}")
if nuc_missing: print(f"  NUCLEAR terms not in the cache: {nuc_missing}")

# One protein group per symbol: the first in var order, the rule of Figure 4e.
_g2p = {}
for pid, gene in jt.var["single_gene_name"].to_dict().items():
    if pd.notna(gene):
        _g2p.setdefault(str(gene), pid)
mem_proteins = [_g2p[g] for g in sorted(mem_only) if g in _g2p]
nuc_proteins = [_g2p[g] for g in sorted(nuc_only) if g in _g2p]
print(f"\nDetected here: MEMBRANE {len(mem_proteins)} proteins, "
      f"NUCLEAR {len(nuc_proteins)} proteins")

print("\nFULL MEMBERSHIP — membrane/vesicular "
      f"({len(mem_proteins)} detected proteins):")
print("  " + ", ".join(sorted(jt.var['single_gene_name'].reindex(mem_proteins)
                              .astype(str))))
print(f"\nFULL MEMBERSHIP — nuclear/translational "
      f"({len(nuc_proteins)} detected proteins):")
print("  " + ", ".join(sorted(jt.var['single_gene_name'].reindex(nuc_proteins)
                              .astype(str))))

# ------------------------------------------- per-sample sums of the linear intensities
# layers["linear"] holds the values before log2(x + 1), so the sums need no
# back-transformation.
_lin = pd.DataFrame(jt.layers["linear"], index=jt.obs_names, columns=jt.var_names)
mem_sum = _lin[mem_proteins].sum(axis=1, min_count=1)
nuc_sum = _lin[nuc_proteins].sum(axis=1, min_count=1)

d1 = jt.obs[["Cohort", "CellType_Aggregate", "disease_binary", "Brain",
             "BrainBank", "Brain Region"]].copy()
d1["mem_sum"], d1["nuc_sum"] = mem_sum.values, nuc_sum.values
_bad = (~np.isfinite(d1["mem_sum"])) | (~np.isfinite(d1["nuc_sum"])) \
       | (d1["mem_sum"] <= 0) | (d1["nuc_sum"] <= 0)
print(f"\nCHECK — samples dropped because one group summed to nothing: "
      f"{int(_bad.sum())}")
if _bad.any():
    print(d1.loc[_bad, ["Cohort", "Brain", "CellType_Aggregate",
                        "mem_sum", "nuc_sum"]].to_string())
d1 = d1[~_bad].copy()
d1["mem_nuc_log2r"] = np.log2((d1["mem_sum"] + EPS) / (d1["nuc_sum"] + EPS))
d1 = d1[d1["CellType_Aggregate"].isin(CLASS_ORDER)].copy()
d1["CellType_Aggregate"] = pd.Categorical(d1["CellType_Aggregate"],
                                          categories=CLASS_ORDER, ordered=True)

# ----------------------------------------------------------------- CHECKS
print(f"\nCHECK — samples per class per cohort (all classes, both disease states):")
print(pd.crosstab(d1["CellType_Aggregate"], d1["Cohort"]).to_string())
_missing_cls = {c: [k for k in CLASS_ORDER
                    if not ((d1["Cohort"] == c) & (d1["CellType_Aggregate"] == k)).any()]
                for c in COHORT_ORDER}
for c, ks in _missing_cls.items():
    if ks:
        print(f"  NOTE: {c} has no samples in {ks}")
for _st_ in ("Disease", "Control"):
    print(f"\n--- {_st_} samples, log2(membrane / nuclear-translational) ---")
    _s = d1[d1["disease_binary"].astype(str) == _st_]
    if len(_s):
        print(_s.groupby(["Cohort", "CellType_Aggregate"], observed=True)
                ["mem_nuc_log2r"].agg(["count", "mean", "median", "std"])
                .round(2).to_string())
    else:
        print("  none")
print(f"\nNOTE: the two cohorts differ in overall level for reasons this panel "
      f"cannot separate. It supports comparison of fractions WITHIN a cohort, "
      f"not of absolute values between cohorts.")

# Per disease donor of both cohorts: mean ratio of the round inclusions minus that
# of the DAB+ host neurons, for donors with both. The Results state that it is
# positive in every donor.
_pd_ = (d1[d1["disease_binary"].astype(str) == "Disease"]
        .groupby(["Cohort", "Brain", "CellType_Aggregate"], observed=True)["mem_nuc_log2r"]
        .mean().unstack())
_pd_ = _pd_[["Round Aggregate Aggregate", "Neuron DAB Positive"]].dropna()
_pd_["round_minus_host"] = _pd_["Round Aggregate Aggregate"] - _pd_["Neuron DAB Positive"]
print(f"\nCHECK — round inclusion minus DAB+ host neuron per donor (log2 ratio):")
print(_pd_["round_minus_host"].round(2).to_string())
print(f"  positive in {int((_pd_['round_minus_host'] > 0).sum())} of {len(_pd_)} donors")

_stub = f"{today}_D1_membrane_nuclear_bothcohorts".replace(" ", "")
d1.to_csv(f"{output_dir}/{_stub}_persample.csv")
print(f"\nSaved {_stub}_persample.csv  ({len(d1)} samples)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Figure S6n. Membrane-to-nuclear ratio per sample in the replication cohort, in
# the design of Figure 4e: CELLTYPE_AGG colours, boxes for the disease samples,
# crosses for the control samples, two-level x labels. Drawn by fraction_boxplot
# (src/boxplot.py). The unit is the sample: this panel is the summary without a
# donor-level model. The cohort name is the panel title.
# ═══════════════════════════════════════════════════════════════════════
class_colors = {c: CELLTYPE_AGG_COLORS[c] for c in CLASS_ORDER}
# Two-level x labels, as in Figure 4e: the DAB state or EA shape under each box,
# the cell type in a row below, centred under its boxes. Neuropil, a single box,
# is named in the row of rotated labels.
SUB_LABEL = {
    "Neuron Without Aggregate":   "DAB−", "Neuron DAB Positive": "DAB+",
    "Glia Without Aggregate":     "DAB−", "Glia DAB Positive":   "DAB+",
    "Neuropil Without Aggregate": "Neuropil",
    "Round Aggregate Aggregate":  "round", "Elongated Aggregate Aggregate": "elong.",
}
GROUP_OF = [
    ("Neuron", ["Neuron Without Aggregate", "Neuron DAB Positive"]),
    ("Glia",   ["Glia Without Aggregate", "Glia DAB Positive"]),
    ("EA",     ["Round Aggregate Aggregate", "Elongated Aggregate Aggregate"]),
]

# Dots 1.6 pt; the legend markers (3.5 pt) are larger, so the key does not read as data.
fraction_boxplot(d1[d1["Cohort"] == REPL_COHORT], "CellType_Aggregate", "mem_nuc_log2r",
                 CLASS_ORDER, class_colors, SUB_LABEL, GROUP_OF,
                 "n", "D1_membrane_nuclear_replication", SI6_DIR,
                 unit="sample", dot_pt=1.6, legend_dot_pt=3.5, legend_handlelength=0.6,
                 title=REPL_LABEL)

# 14. Figure assembly

## 14.1 Figure 4

In [ ]:
# Figure 4 from the newest panel file per letter in FIG4_DIR. The offsets move
# panels c and e 2 mm down.
figure_name = f"{today}_Figure4_Brain"
layout = "a b c d\n f f c e\n f f g ."

compose_figure(
    folder=FIG4_DIR,
    layout=layout,
    output=f"{FIG4_DIR}/{figure_name}.svg",
    show_box=True,
    panel_offset_mm={"c": (0, 2),
                     "e": (0, 2)},
)

## 14.2 Figure S6

In [ ]:
# Figure S6 from the newest panel file per letter in SI6_DIR; the letters are those
# save_panel wrote in the panel cells.
figure_name = f"{today}_SI6_HumanBrain"
layout = "a a b c\n d e f h\n g g i j\n k l m n"

compose_figure(
    folder=SI6_DIR,
    layout=layout,
    output=f"{SI6_DIR}/{figure_name}.svg",
    show_box=True,
)